# 04b Extraction, batch b: twelve included studies coded from their PDFs

Design log.
- Batch b of notebook 04 (plan v2 B4), coded under the frozen schema (schema/record_schema.yaml, v1) and the coding rules, as batch a was. The twelve reports: Liu et al.'s benchmark (2310.12815), Signed-Prompt (2401.07612), AgentDojo (2406.13352, NeurIPS 2024 Datasets and Benchmarks), Agent Security Bench (2410.02644, ICLR 2025), defences built from attack techniques (2411.00459), adaptive attacks on agent defences (2503.00061), position-enhanced fine-tuning (2505.00626), Google DeepMind's Gemini report (2505.14534), Checkpoint-GCG (2505.15738, HAIPS at COLM 2026), AIR (2505.18907), Polymorphic Prompt Assembling (2506.05739) and LLMail-Inject (2506.09956).
- Each record's extraction_scope names the tables and rows coded. In the benchmark and re-measurement papers only defences that act on the protected model's own generation are coded as defences (amendment 4): paraphrasing, retokenization and dynamic prompt rewriting change the data before an unchanged model, and detectors and tool filters sit outside it, so their rows are left out and the scope says so.
- New check in Stage 3: pypdf glues some adjacent table cells together in both of its extraction modes (AIR's Table 1 prints 48.1 and 4.1 side by side; pypdf reads 48.14.1). A value not found in the plain or the column-preserving text is now looked for in a third extraction (pypdfium2) before it counts as a problem, and every value found that way is listed in reports/extraction_04b_check.csv. This keeps coding rule 34 (values found in an extraction that does not glue table cells) and adds pypdfium2 to the bootstrap.
- Coding decisions worth knowing when reading these records (reading aids for notebook 05, not findings):
  - Three reports give results for defended models only (Signed-Prompt, Polymorphic Prompt Assembling, LLMail-Inject's Spotlight and Meta SecAlign rows). Their results are kept for coverage; they carry no comparisons, because there is no undefended arm under the same protocol.
  - The adaptive-attack report (2503.00061) prints its per-defence attack success rates as labels inside a raster chart. Values that cannot be found in the page text are not coded; the three numbers its text states are coded, Table 6 (AgentDojo) is coded, and a residual note lists the chart labels for a human check.
  - The Gemini report compares the adversarially trained Gemini 2.5 with Gemini 2.0, a different model, so those comparisons grade B; the Warning defence on Gemini 2.5 against Gemini 2.5 alone is a component comparison.
  - Checkpoint-GCG is coded with the defence-against-undefended comparisons and, within each defended model, a comparison that varies only the attack.
  - Liu et al.'s version thanks reviewers and a shepherd but prints no venue, so peer review is coded unknown.
- Full-text exclusions found while coding this batch: none.
- Outputs: reports/records/ (twelve files), reports/extraction_04b_check.csv, reports/extraction_04b_summary.csv, reports/extraction_batches.csv (rows for batch b added, batch a kept), reports/decision_04b.txt.
- CPU; no network calls beyond pip; model calls: 0.

## Stage 0. Session bootstrap

In [ ]:
import os, sys, subprocess, importlib
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/SourceMap_Research"
    REPO_DIR = PROJECT_ROOT + "/sourcemap-research"
except ImportError:
    PROJECT_ROOT = os.environ["SOURCEMAP_PROJECT_ROOT"]
    REPO_DIR = os.environ["SOURCEMAP_REPO_ROOT"]
os.environ["SOURCEMAP_PROJECT_ROOT"] = PROJECT_ROOT
os.environ["SOURCEMAP_REPO_ROOT"] = REPO_DIR
os.chdir(REPO_DIR)
if "src" not in sys.path:
    sys.path.insert(0, "src")
for module, package in (("yaml", "pyyaml"), ("pypdf", "pypdf"), ("pypdfium2", "pypdfium2")):
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=False)
import pypdf
if int(pypdf.__version__.split(".")[0]) < 4:                 # the column-preserving extraction needs pypdf 4 or later
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pypdf>=4"], check=False)
    print("pypdf upgraded; restart the runtime if the next stage reports an old version")
print(subprocess.run(["git", "pull", "--ff-only"], capture_output=True, text=True).stdout.strip() or "pull: nothing to report")
print("cwd:", os.getcwd(), "| pypdf", pypdf.__version__)

## Stage 1. The 12 records

In [ ]:
Path("reports/records").mkdir(parents=True, exist_ok=True)       # %%writefile does not create folders

`liu_benchmark_arxiv2023` (2310.12815)

In [ ]:
%%writefile reports/records/liu_benchmark_arxiv2023.yaml
report:
  id: liu_benchmark_arxiv2023
  slug: '2310.12815'
  title: Formalizing and Benchmarking Prompt Injection Attacks and Defenses
  first_author: Yupei Liu
  venue: arXiv preprint
  peer_reviewed: unknown
  arxiv: '2310.12815'
  version_read: arXiv v5 (acknowledges anonymous reviewers and a shepherd; venue not printed)
  first_date: '2023-10-19'
  read: F
  role: re_measures
  code_release: https://github.com/liu00222/Open-Prompt-Injection
  extraction_scope: 'Sections 6.1 and 6.3: Table 7 (GPT-4), the No defense, Delimiters, Sandwich prevention and Instructional prevention columns: ASV in part (a) and PNA-T in part (b) for all seven target tasks; the abstract
    claims (paraphrasing and retokenization, which change the data before an unchanged model, the detection-based defences of Table 8, MR, other LLMs and the appendix tables not coded)'
defences:
- id: delim_gpt4
  name: Delimiters
  origin: other_report
  base_checkpoint: GPT-4 (Azure OpenAI)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  components:
  - data enclosed in delimiters so the model treats it as data
  layout: instruction prompt in the system role, data in the user role
  layout_compat:
    user_turn: untested
    separate_role: supported
    system_context: untested
    agent_tool_result: untested
  anchor: Use delimiters to enclose the data to force
  pages:
    layout: 7
- id: sandwich_gpt4
  name: Sandwich prevention
  origin: other_report
  base_checkpoint: GPT-4 (Azure OpenAI)
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  components:
  - the instruction prompt repeated after the data
  layout: instruction prompt in the system role, data in the user role
  layout_compat:
    user_turn: untested
    separate_role: supported
    system_context: untested
    agent_tool_result: untested
  anchor: Append another instruction prompt at the end of the data
  pages:
    layout: 7
- id: instr_gpt4
  name: Instructional prevention
  origin: other_report
  base_checkpoint: GPT-4 (Azure OpenAI)
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  components:
  - instruction prompt rewritten to tell the model to ignore instructions in the data
  layout: instruction prompt in the system role, data in the user role
  layout_compat:
    user_turn: untested
    separate_role: supported
    system_context: untested
    agent_tool_result: untested
  anchor: Re-design the instruction prompt to make the LLM ignore
  pages:
    layout: 7
evaluations:
- id: ev_none_dup_atk
  defence_id: none:GPT-4 (Azure OpenAI)
  benchmark: Dup. sentence detection as the target task, 100 target examples, averaged over the 7 injected tasks
  attacker_type: static
  attacker_access: black_box
  judge: 'task metric on the injected task''s label: accuracy for classification, GLEU for grammar correction, Rouge-1 for summarisation'
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  attacker_budget: Combined Attack
  sample_size: 100 target x 100 injected examples
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
  baseline_kind: off_the_shelf
- id: ev_none_dup_clean
  defence_id: none:GPT-4 (Azure OpenAI)
  benchmark: Dup. sentence detection as the target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric on the target task's label (PNA-T)
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  sample_size: '100'
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
  baseline_kind: off_the_shelf
- id: ev_none_gram_atk
  defence_id: none:GPT-4 (Azure OpenAI)
  benchmark: Grammar correction as the target task, 100 target examples, averaged over the 7 injected tasks
  attacker_type: static
  attacker_access: black_box
  judge: 'task metric on the injected task''s label: accuracy for classification, GLEU for grammar correction, Rouge-1 for summarisation'
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  attacker_budget: Combined Attack
  sample_size: 100 target x 100 injected examples
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
  baseline_kind: off_the_shelf
- id: ev_none_gram_clean
  defence_id: none:GPT-4 (Azure OpenAI)
  benchmark: Grammar correction as the target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric on the target task's label (PNA-T)
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  sample_size: '100'
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
  baseline_kind: off_the_shelf
- id: ev_none_hate_atk
  defence_id: none:GPT-4 (Azure OpenAI)
  benchmark: Hate detection as the target task, 100 target examples, averaged over the 7 injected tasks
  attacker_type: static
  attacker_access: black_box
  judge: 'task metric on the injected task''s label: accuracy for classification, GLEU for grammar correction, Rouge-1 for summarisation'
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  attacker_budget: Combined Attack
  sample_size: 100 target x 100 injected examples
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
  baseline_kind: off_the_shelf
- id: ev_none_hate_clean
  defence_id: none:GPT-4 (Azure OpenAI)
  benchmark: Hate detection as the target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric on the target task's label (PNA-T)
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  sample_size: '100'
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
  baseline_kind: off_the_shelf
- id: ev_none_nli_atk
  defence_id: none:GPT-4 (Azure OpenAI)
  benchmark: Nat. lang. inference as the target task, 100 target examples, averaged over the 7 injected tasks
  attacker_type: static
  attacker_access: black_box
  judge: 'task metric on the injected task''s label: accuracy for classification, GLEU for grammar correction, Rouge-1 for summarisation'
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  attacker_budget: Combined Attack
  sample_size: 100 target x 100 injected examples
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
  baseline_kind: off_the_shelf
- id: ev_none_nli_clean
  defence_id: none:GPT-4 (Azure OpenAI)
  benchmark: Nat. lang. inference as the target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric on the target task's label (PNA-T)
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  sample_size: '100'
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
  baseline_kind: off_the_shelf
- id: ev_none_sent_atk
  defence_id: none:GPT-4 (Azure OpenAI)
  benchmark: Sentiment analysis as the target task, 100 target examples, averaged over the 7 injected tasks
  attacker_type: static
  attacker_access: black_box
  judge: 'task metric on the injected task''s label: accuracy for classification, GLEU for grammar correction, Rouge-1 for summarisation'
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  attacker_budget: Combined Attack
  sample_size: 100 target x 100 injected examples
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
  baseline_kind: off_the_shelf
- id: ev_none_sent_clean
  defence_id: none:GPT-4 (Azure OpenAI)
  benchmark: Sentiment analysis as the target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric on the target task's label (PNA-T)
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  sample_size: '100'
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
  baseline_kind: off_the_shelf
- id: ev_none_spam_atk
  defence_id: none:GPT-4 (Azure OpenAI)
  benchmark: Spam detection as the target task, 100 target examples, averaged over the 7 injected tasks
  attacker_type: static
  attacker_access: black_box
  judge: 'task metric on the injected task''s label: accuracy for classification, GLEU for grammar correction, Rouge-1 for summarisation'
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  attacker_budget: Combined Attack
  sample_size: 100 target x 100 injected examples
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
  baseline_kind: off_the_shelf
- id: ev_none_spam_clean
  defence_id: none:GPT-4 (Azure OpenAI)
  benchmark: Spam detection as the target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric on the target task's label (PNA-T)
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  sample_size: '100'
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
  baseline_kind: off_the_shelf
- id: ev_none_summ_atk
  defence_id: none:GPT-4 (Azure OpenAI)
  benchmark: Summarization as the target task, 100 target examples, averaged over the 7 injected tasks
  attacker_type: static
  attacker_access: black_box
  judge: 'task metric on the injected task''s label: accuracy for classification, GLEU for grammar correction, Rouge-1 for summarisation'
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  attacker_budget: Combined Attack
  sample_size: 100 target x 100 injected examples
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
  baseline_kind: off_the_shelf
- id: ev_none_summ_clean
  defence_id: none:GPT-4 (Azure OpenAI)
  benchmark: Summarization as the target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric on the target task's label (PNA-T)
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  sample_size: '100'
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
  baseline_kind: off_the_shelf
- id: ev_delim_dup_atk
  defence_id: delim_gpt4
  benchmark: Dup. sentence detection as the target task, 100 target examples, averaged over the 7 injected tasks
  attacker_type: static
  attacker_access: black_box
  judge: 'task metric on the injected task''s label: accuracy for classification, GLEU for grammar correction, Rouge-1 for summarisation'
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  attacker_budget: Combined Attack
  sample_size: 100 target x 100 injected examples
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_delim_dup_clean
  defence_id: delim_gpt4
  benchmark: Dup. sentence detection as the target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric on the target task's label (PNA-T)
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  sample_size: '100'
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_delim_gram_atk
  defence_id: delim_gpt4
  benchmark: Grammar correction as the target task, 100 target examples, averaged over the 7 injected tasks
  attacker_type: static
  attacker_access: black_box
  judge: 'task metric on the injected task''s label: accuracy for classification, GLEU for grammar correction, Rouge-1 for summarisation'
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  attacker_budget: Combined Attack
  sample_size: 100 target x 100 injected examples
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_delim_gram_clean
  defence_id: delim_gpt4
  benchmark: Grammar correction as the target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric on the target task's label (PNA-T)
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  sample_size: '100'
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_delim_hate_atk
  defence_id: delim_gpt4
  benchmark: Hate detection as the target task, 100 target examples, averaged over the 7 injected tasks
  attacker_type: static
  attacker_access: black_box
  judge: 'task metric on the injected task''s label: accuracy for classification, GLEU for grammar correction, Rouge-1 for summarisation'
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  attacker_budget: Combined Attack
  sample_size: 100 target x 100 injected examples
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_delim_hate_clean
  defence_id: delim_gpt4
  benchmark: Hate detection as the target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric on the target task's label (PNA-T)
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  sample_size: '100'
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_delim_nli_atk
  defence_id: delim_gpt4
  benchmark: Nat. lang. inference as the target task, 100 target examples, averaged over the 7 injected tasks
  attacker_type: static
  attacker_access: black_box
  judge: 'task metric on the injected task''s label: accuracy for classification, GLEU for grammar correction, Rouge-1 for summarisation'
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  attacker_budget: Combined Attack
  sample_size: 100 target x 100 injected examples
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_delim_nli_clean
  defence_id: delim_gpt4
  benchmark: Nat. lang. inference as the target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric on the target task's label (PNA-T)
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  sample_size: '100'
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_delim_sent_atk
  defence_id: delim_gpt4
  benchmark: Sentiment analysis as the target task, 100 target examples, averaged over the 7 injected tasks
  attacker_type: static
  attacker_access: black_box
  judge: 'task metric on the injected task''s label: accuracy for classification, GLEU for grammar correction, Rouge-1 for summarisation'
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  attacker_budget: Combined Attack
  sample_size: 100 target x 100 injected examples
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_delim_sent_clean
  defence_id: delim_gpt4
  benchmark: Sentiment analysis as the target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric on the target task's label (PNA-T)
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  sample_size: '100'
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_delim_spam_atk
  defence_id: delim_gpt4
  benchmark: Spam detection as the target task, 100 target examples, averaged over the 7 injected tasks
  attacker_type: static
  attacker_access: black_box
  judge: 'task metric on the injected task''s label: accuracy for classification, GLEU for grammar correction, Rouge-1 for summarisation'
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  attacker_budget: Combined Attack
  sample_size: 100 target x 100 injected examples
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_delim_spam_clean
  defence_id: delim_gpt4
  benchmark: Spam detection as the target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric on the target task's label (PNA-T)
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  sample_size: '100'
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_delim_summ_atk
  defence_id: delim_gpt4
  benchmark: Summarization as the target task, 100 target examples, averaged over the 7 injected tasks
  attacker_type: static
  attacker_access: black_box
  judge: 'task metric on the injected task''s label: accuracy for classification, GLEU for grammar correction, Rouge-1 for summarisation'
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  attacker_budget: Combined Attack
  sample_size: 100 target x 100 injected examples
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_delim_summ_clean
  defence_id: delim_gpt4
  benchmark: Summarization as the target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric on the target task's label (PNA-T)
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  sample_size: '100'
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_sandwich_dup_atk
  defence_id: sandwich_gpt4
  benchmark: Dup. sentence detection as the target task, 100 target examples, averaged over the 7 injected tasks
  attacker_type: static
  attacker_access: black_box
  judge: 'task metric on the injected task''s label: accuracy for classification, GLEU for grammar correction, Rouge-1 for summarisation'
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  attacker_budget: Combined Attack
  sample_size: 100 target x 100 injected examples
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_sandwich_dup_clean
  defence_id: sandwich_gpt4
  benchmark: Dup. sentence detection as the target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric on the target task's label (PNA-T)
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  sample_size: '100'
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_sandwich_gram_atk
  defence_id: sandwich_gpt4
  benchmark: Grammar correction as the target task, 100 target examples, averaged over the 7 injected tasks
  attacker_type: static
  attacker_access: black_box
  judge: 'task metric on the injected task''s label: accuracy for classification, GLEU for grammar correction, Rouge-1 for summarisation'
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  attacker_budget: Combined Attack
  sample_size: 100 target x 100 injected examples
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_sandwich_gram_clean
  defence_id: sandwich_gpt4
  benchmark: Grammar correction as the target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric on the target task's label (PNA-T)
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  sample_size: '100'
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_sandwich_hate_atk
  defence_id: sandwich_gpt4
  benchmark: Hate detection as the target task, 100 target examples, averaged over the 7 injected tasks
  attacker_type: static
  attacker_access: black_box
  judge: 'task metric on the injected task''s label: accuracy for classification, GLEU for grammar correction, Rouge-1 for summarisation'
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  attacker_budget: Combined Attack
  sample_size: 100 target x 100 injected examples
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_sandwich_hate_clean
  defence_id: sandwich_gpt4
  benchmark: Hate detection as the target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric on the target task's label (PNA-T)
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  sample_size: '100'
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_sandwich_nli_atk
  defence_id: sandwich_gpt4
  benchmark: Nat. lang. inference as the target task, 100 target examples, averaged over the 7 injected tasks
  attacker_type: static
  attacker_access: black_box
  judge: 'task metric on the injected task''s label: accuracy for classification, GLEU for grammar correction, Rouge-1 for summarisation'
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  attacker_budget: Combined Attack
  sample_size: 100 target x 100 injected examples
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_sandwich_nli_clean
  defence_id: sandwich_gpt4
  benchmark: Nat. lang. inference as the target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric on the target task's label (PNA-T)
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  sample_size: '100'
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_sandwich_sent_atk
  defence_id: sandwich_gpt4
  benchmark: Sentiment analysis as the target task, 100 target examples, averaged over the 7 injected tasks
  attacker_type: static
  attacker_access: black_box
  judge: 'task metric on the injected task''s label: accuracy for classification, GLEU for grammar correction, Rouge-1 for summarisation'
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  attacker_budget: Combined Attack
  sample_size: 100 target x 100 injected examples
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_sandwich_sent_clean
  defence_id: sandwich_gpt4
  benchmark: Sentiment analysis as the target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric on the target task's label (PNA-T)
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  sample_size: '100'
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_sandwich_spam_atk
  defence_id: sandwich_gpt4
  benchmark: Spam detection as the target task, 100 target examples, averaged over the 7 injected tasks
  attacker_type: static
  attacker_access: black_box
  judge: 'task metric on the injected task''s label: accuracy for classification, GLEU for grammar correction, Rouge-1 for summarisation'
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  attacker_budget: Combined Attack
  sample_size: 100 target x 100 injected examples
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_sandwich_spam_clean
  defence_id: sandwich_gpt4
  benchmark: Spam detection as the target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric on the target task's label (PNA-T)
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  sample_size: '100'
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_sandwich_summ_atk
  defence_id: sandwich_gpt4
  benchmark: Summarization as the target task, 100 target examples, averaged over the 7 injected tasks
  attacker_type: static
  attacker_access: black_box
  judge: 'task metric on the injected task''s label: accuracy for classification, GLEU for grammar correction, Rouge-1 for summarisation'
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  attacker_budget: Combined Attack
  sample_size: 100 target x 100 injected examples
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_sandwich_summ_clean
  defence_id: sandwich_gpt4
  benchmark: Summarization as the target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric on the target task's label (PNA-T)
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  sample_size: '100'
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_instr_dup_atk
  defence_id: instr_gpt4
  benchmark: Dup. sentence detection as the target task, 100 target examples, averaged over the 7 injected tasks
  attacker_type: static
  attacker_access: black_box
  judge: 'task metric on the injected task''s label: accuracy for classification, GLEU for grammar correction, Rouge-1 for summarisation'
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  attacker_budget: Combined Attack
  sample_size: 100 target x 100 injected examples
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_instr_dup_clean
  defence_id: instr_gpt4
  benchmark: Dup. sentence detection as the target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric on the target task's label (PNA-T)
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  sample_size: '100'
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_instr_gram_atk
  defence_id: instr_gpt4
  benchmark: Grammar correction as the target task, 100 target examples, averaged over the 7 injected tasks
  attacker_type: static
  attacker_access: black_box
  judge: 'task metric on the injected task''s label: accuracy for classification, GLEU for grammar correction, Rouge-1 for summarisation'
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  attacker_budget: Combined Attack
  sample_size: 100 target x 100 injected examples
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_instr_gram_clean
  defence_id: instr_gpt4
  benchmark: Grammar correction as the target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric on the target task's label (PNA-T)
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  sample_size: '100'
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_instr_hate_atk
  defence_id: instr_gpt4
  benchmark: Hate detection as the target task, 100 target examples, averaged over the 7 injected tasks
  attacker_type: static
  attacker_access: black_box
  judge: 'task metric on the injected task''s label: accuracy for classification, GLEU for grammar correction, Rouge-1 for summarisation'
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  attacker_budget: Combined Attack
  sample_size: 100 target x 100 injected examples
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_instr_hate_clean
  defence_id: instr_gpt4
  benchmark: Hate detection as the target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric on the target task's label (PNA-T)
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  sample_size: '100'
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_instr_nli_atk
  defence_id: instr_gpt4
  benchmark: Nat. lang. inference as the target task, 100 target examples, averaged over the 7 injected tasks
  attacker_type: static
  attacker_access: black_box
  judge: 'task metric on the injected task''s label: accuracy for classification, GLEU for grammar correction, Rouge-1 for summarisation'
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  attacker_budget: Combined Attack
  sample_size: 100 target x 100 injected examples
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_instr_nli_clean
  defence_id: instr_gpt4
  benchmark: Nat. lang. inference as the target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric on the target task's label (PNA-T)
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  sample_size: '100'
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_instr_sent_atk
  defence_id: instr_gpt4
  benchmark: Sentiment analysis as the target task, 100 target examples, averaged over the 7 injected tasks
  attacker_type: static
  attacker_access: black_box
  judge: 'task metric on the injected task''s label: accuracy for classification, GLEU for grammar correction, Rouge-1 for summarisation'
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  attacker_budget: Combined Attack
  sample_size: 100 target x 100 injected examples
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_instr_sent_clean
  defence_id: instr_gpt4
  benchmark: Sentiment analysis as the target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric on the target task's label (PNA-T)
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  sample_size: '100'
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_instr_spam_atk
  defence_id: instr_gpt4
  benchmark: Spam detection as the target task, 100 target examples, averaged over the 7 injected tasks
  attacker_type: static
  attacker_access: black_box
  judge: 'task metric on the injected task''s label: accuracy for classification, GLEU for grammar correction, Rouge-1 for summarisation'
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  attacker_budget: Combined Attack
  sample_size: 100 target x 100 injected examples
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_instr_spam_clean
  defence_id: instr_gpt4
  benchmark: Spam detection as the target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric on the target task's label (PNA-T)
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  sample_size: '100'
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_instr_summ_atk
  defence_id: instr_gpt4
  benchmark: Summarization as the target task, 100 target examples, averaged over the 7 injected tasks
  attacker_type: static
  attacker_access: black_box
  judge: 'task metric on the injected task''s label: accuracy for classification, GLEU for grammar correction, Rouge-1 for summarisation'
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  attacker_budget: Combined Attack
  sample_size: 100 target x 100 injected examples
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
- id: ev_instr_summ_clean
  defence_id: instr_gpt4
  benchmark: Summarization as the target task, no attack
  attacker_type: none
  attacker_access: not_applicable
  judge: task metric on the target task's label (PNA-T)
  layout_same_as_defence: 'yes'
  measured_by: liu_benchmark_arxiv2023
  page: 11
  sample_size: '100'
  decoding: temperature 0.1
  pages:
    benchmark: 7
    judge: 8
    decoding: 7
results:
- id: r_none_dup_asv
  evaluation_id: ev_none_dup_atk
  quantity: R1
  coverage: direct
  metric_as_printed: ASV
  value: '0.76'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(a)
- id: r_none_dup_pna
  evaluation_id: ev_none_dup_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA-T
  value: '0.73'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(b)
- id: r_none_gram_asv
  evaluation_id: ev_none_gram_atk
  quantity: R1
  coverage: direct
  metric_as_printed: ASV
  value: '0.73'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(a)
- id: r_none_gram_pna
  evaluation_id: ev_none_gram_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA-T
  value: '0.48'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(b)
- id: r_none_hate_asv
  evaluation_id: ev_none_hate_atk
  quantity: R1
  coverage: direct
  metric_as_printed: ASV
  value: '0.74'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(a)
- id: r_none_hate_pna
  evaluation_id: ev_none_hate_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA-T
  value: '0.79'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(b)
- id: r_none_nli_asv
  evaluation_id: ev_none_nli_atk
  quantity: R1
  coverage: direct
  metric_as_printed: ASV
  value: '0.75'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(a)
- id: r_none_nli_pna
  evaluation_id: ev_none_nli_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA-T
  value: '0.86'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(b)
- id: r_none_sent_asv
  evaluation_id: ev_none_sent_atk
  quantity: R1
  coverage: direct
  metric_as_printed: ASV
  value: '0.76'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(a)
- id: r_none_sent_pna
  evaluation_id: ev_none_sent_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA-T
  value: '0.96'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(b)
- id: r_none_spam_asv
  evaluation_id: ev_none_spam_atk
  quantity: R1
  coverage: direct
  metric_as_printed: ASV
  value: '0.76'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(a)
- id: r_none_spam_pna
  evaluation_id: ev_none_spam_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA-T
  value: '0.92'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(b)
- id: r_none_summ_asv
  evaluation_id: ev_none_summ_atk
  quantity: R1
  coverage: direct
  metric_as_printed: ASV
  value: '0.75'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(a)
- id: r_none_summ_pna
  evaluation_id: ev_none_summ_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA-T
  value: '0.38'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(b)
- id: r_delim_dup_asv
  evaluation_id: ev_delim_dup_atk
  quantity: R1
  coverage: direct
  metric_as_printed: ASV
  value: '0.36'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(a)
- id: r_delim_dup_pna
  evaluation_id: ev_delim_dup_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA-T
  value: '0.75'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(b)
- id: r_delim_gram_asv
  evaluation_id: ev_delim_gram_atk
  quantity: R1
  coverage: direct
  metric_as_printed: ASV
  value: '0.29'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(a)
- id: r_delim_gram_pna
  evaluation_id: ev_delim_gram_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA-T
  value: '0.00'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(b)
- id: r_delim_hate_asv
  evaluation_id: ev_delim_hate_atk
  quantity: R1
  coverage: direct
  metric_as_printed: ASV
  value: '0.39'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(a)
- id: r_delim_hate_pna
  evaluation_id: ev_delim_hate_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA-T
  value: '0.88'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(b)
- id: r_delim_nli_asv
  evaluation_id: ev_delim_nli_atk
  quantity: R1
  coverage: direct
  metric_as_printed: ASV
  value: '0.42'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(a)
- id: r_delim_nli_pna
  evaluation_id: ev_delim_nli_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA-T
  value: '0.85'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(b)
- id: r_delim_sent_asv
  evaluation_id: ev_delim_sent_atk
  quantity: R1
  coverage: direct
  metric_as_printed: ASV
  value: '0.51'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(a)
- id: r_delim_sent_pna
  evaluation_id: ev_delim_sent_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA-T
  value: '0.92'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(b)
- id: r_delim_spam_asv
  evaluation_id: ev_delim_spam_atk
  quantity: R1
  coverage: direct
  metric_as_printed: ASV
  value: '0.65'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(a)
- id: r_delim_spam_pna
  evaluation_id: ev_delim_spam_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA-T
  value: '0.92'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(b)
- id: r_delim_summ_asv
  evaluation_id: ev_delim_summ_atk
  quantity: R1
  coverage: direct
  metric_as_printed: ASV
  value: '0.72'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(a)
- id: r_delim_summ_pna
  evaluation_id: ev_delim_summ_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA-T
  value: '0.22'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(b)
- id: r_sandwich_dup_asv
  evaluation_id: ev_sandwich_dup_atk
  quantity: R1
  coverage: direct
  metric_as_printed: ASV
  value: '0.39'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(a)
- id: r_sandwich_dup_pna
  evaluation_id: ev_sandwich_dup_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA-T
  value: '0.77'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(b)
- id: r_sandwich_gram_asv
  evaluation_id: ev_sandwich_gram_atk
  quantity: R1
  coverage: direct
  metric_as_printed: ASV
  value: '0.26'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(a)
- id: r_sandwich_gram_pna
  evaluation_id: ev_sandwich_gram_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA-T
  value: '0.53'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(b)
- id: r_sandwich_hate_asv
  evaluation_id: ev_sandwich_hate_atk
  quantity: R1
  coverage: direct
  metric_as_printed: ASV
  value: '0.36'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(a)
- id: r_sandwich_hate_pna
  evaluation_id: ev_sandwich_hate_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA-T
  value: '0.88'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(b)
- id: r_sandwich_nli_asv
  evaluation_id: ev_sandwich_nli_atk
  quantity: R1
  coverage: direct
  metric_as_printed: ASV
  value: '0.65'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(a)
- id: r_sandwich_nli_pna
  evaluation_id: ev_sandwich_nli_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA-T
  value: '0.86'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(b)
- id: r_sandwich_sent_asv
  evaluation_id: ev_sandwich_sent_atk
  quantity: R1
  coverage: direct
  metric_as_printed: ASV
  value: '0.26'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(a)
- id: r_sandwich_sent_pna
  evaluation_id: ev_sandwich_sent_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA-T
  value: '0.92'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(b)
- id: r_sandwich_spam_asv
  evaluation_id: ev_sandwich_spam_atk
  quantity: R1
  coverage: direct
  metric_as_printed: ASV
  value: '0.57'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(a)
- id: r_sandwich_spam_pna
  evaluation_id: ev_sandwich_spam_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA-T
  value: '0.86'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(b)
- id: r_sandwich_summ_asv
  evaluation_id: ev_sandwich_summ_atk
  quantity: R1
  coverage: direct
  metric_as_printed: ASV
  value: '0.70'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(a)
- id: r_sandwich_summ_pna
  evaluation_id: ev_sandwich_summ_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA-T
  value: '0.24'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(b)
- id: r_instr_dup_asv
  evaluation_id: ev_instr_dup_atk
  quantity: R1
  coverage: direct
  metric_as_printed: ASV
  value: '0.17'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(a)
- id: r_instr_dup_pna
  evaluation_id: ev_instr_dup_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA-T
  value: '0.76'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(b)
- id: r_instr_gram_asv
  evaluation_id: ev_instr_gram_atk
  quantity: R1
  coverage: direct
  metric_as_printed: ASV
  value: '0.45'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(a)
- id: r_instr_gram_pna
  evaluation_id: ev_instr_gram_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA-T
  value: '0.52'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(b)
- id: r_instr_hate_asv
  evaluation_id: ev_instr_hate_atk
  quantity: R1
  coverage: direct
  metric_as_printed: ASV
  value: '0.13'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(a)
- id: r_instr_hate_pna
  evaluation_id: ev_instr_hate_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA-T
  value: '0.88'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(b)
- id: r_instr_nli_asv
  evaluation_id: ev_instr_nli_atk
  quantity: R1
  coverage: direct
  metric_as_printed: ASV
  value: '0.45'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(a)
- id: r_instr_nli_pna
  evaluation_id: ev_instr_nli_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA-T
  value: '0.84'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(b)
- id: r_instr_sent_asv
  evaluation_id: ev_instr_sent_atk
  quantity: R1
  coverage: direct
  metric_as_printed: ASV
  value: '0.48'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(a)
- id: r_instr_sent_pna
  evaluation_id: ev_instr_sent_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA-T
  value: '0.95'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(b)
- id: r_instr_spam_asv
  evaluation_id: ev_instr_spam_atk
  quantity: R1
  coverage: direct
  metric_as_printed: ASV
  value: '0.28'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(a)
- id: r_instr_spam_pna
  evaluation_id: ev_instr_spam_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA-T
  value: '0.92'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(b)
- id: r_instr_summ_asv
  evaluation_id: ev_instr_summ_atk
  quantity: R1
  coverage: direct
  metric_as_printed: ASV
  value: '0.73'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(a)
- id: r_instr_summ_pna
  evaluation_id: ev_instr_summ_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA-T
  value: '0.23'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 11
  location: Table 7(b)
comparisons:
- id: c_delim_dup_atk
  left: ev_delim_dup_atk
  right: ev_none_dup_atk
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_delim_dup_clean
  left: ev_delim_dup_clean
  right: ev_none_dup_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_delim_gram_atk
  left: ev_delim_gram_atk
  right: ev_none_gram_atk
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_delim_gram_clean
  left: ev_delim_gram_clean
  right: ev_none_gram_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_delim_hate_atk
  left: ev_delim_hate_atk
  right: ev_none_hate_atk
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_delim_hate_clean
  left: ev_delim_hate_clean
  right: ev_none_hate_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_delim_nli_atk
  left: ev_delim_nli_atk
  right: ev_none_nli_atk
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_delim_nli_clean
  left: ev_delim_nli_clean
  right: ev_none_nli_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_delim_sent_atk
  left: ev_delim_sent_atk
  right: ev_none_sent_atk
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_delim_sent_clean
  left: ev_delim_sent_clean
  right: ev_none_sent_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_delim_spam_atk
  left: ev_delim_spam_atk
  right: ev_none_spam_atk
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_delim_spam_clean
  left: ev_delim_spam_clean
  right: ev_none_spam_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_delim_summ_atk
  left: ev_delim_summ_atk
  right: ev_none_summ_atk
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_delim_summ_clean
  left: ev_delim_summ_clean
  right: ev_none_summ_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_sandwich_dup_atk
  left: ev_sandwich_dup_atk
  right: ev_none_dup_atk
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_sandwich_dup_clean
  left: ev_sandwich_dup_clean
  right: ev_none_dup_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_sandwich_gram_atk
  left: ev_sandwich_gram_atk
  right: ev_none_gram_atk
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_sandwich_gram_clean
  left: ev_sandwich_gram_clean
  right: ev_none_gram_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_sandwich_hate_atk
  left: ev_sandwich_hate_atk
  right: ev_none_hate_atk
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_sandwich_hate_clean
  left: ev_sandwich_hate_clean
  right: ev_none_hate_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_sandwich_nli_atk
  left: ev_sandwich_nli_atk
  right: ev_none_nli_atk
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_sandwich_nli_clean
  left: ev_sandwich_nli_clean
  right: ev_none_nli_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_sandwich_sent_atk
  left: ev_sandwich_sent_atk
  right: ev_none_sent_atk
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_sandwich_sent_clean
  left: ev_sandwich_sent_clean
  right: ev_none_sent_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_sandwich_spam_atk
  left: ev_sandwich_spam_atk
  right: ev_none_spam_atk
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_sandwich_spam_clean
  left: ev_sandwich_spam_clean
  right: ev_none_spam_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_sandwich_summ_atk
  left: ev_sandwich_summ_atk
  right: ev_none_summ_atk
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_sandwich_summ_clean
  left: ev_sandwich_summ_clean
  right: ev_none_summ_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_instr_dup_atk
  left: ev_instr_dup_atk
  right: ev_none_dup_atk
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_instr_dup_clean
  left: ev_instr_dup_clean
  right: ev_none_dup_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_instr_gram_atk
  left: ev_instr_gram_atk
  right: ev_none_gram_atk
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_instr_gram_clean
  left: ev_instr_gram_clean
  right: ev_none_gram_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_instr_hate_atk
  left: ev_instr_hate_atk
  right: ev_none_hate_atk
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_instr_hate_clean
  left: ev_instr_hate_clean
  right: ev_none_hate_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_instr_nli_atk
  left: ev_instr_nli_atk
  right: ev_none_nli_atk
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_instr_nli_clean
  left: ev_instr_nli_clean
  right: ev_none_nli_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_instr_sent_atk
  left: ev_instr_sent_atk
  right: ev_none_sent_atk
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_instr_sent_clean
  left: ev_instr_sent_clean
  right: ev_none_sent_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_instr_spam_atk
  left: ev_instr_spam_atk
  right: ev_none_spam_atk
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_instr_spam_clean
  left: ev_instr_spam_clean
  right: ev_none_spam_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_instr_summ_atk
  left: ev_instr_summ_atk
  right: ev_none_summ_atk
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
- id: c_instr_summ_clean
  left: ev_instr_summ_clean
  right: ev_none_summ_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 11
claims:
- id: k1
  text: systematic evaluation of 5 attacks and 10 defences with 10 LLMs and 7 tasks, a common benchmark
  location: abstract
  anchor: we conduct a systematic evaluation on 5 prompt injection attacks and 10 defenses
  jobs:
  - R1
  - R2
  supported_by:
  - c_delim_dup_atk
  - c_delim_gram_atk
  - c_delim_hate_atk
  - c_delim_nli_atk
  - c_delim_sent_atk
  - c_delim_spam_atk
  - c_delim_summ_atk
  - c_sandwich_dup_atk
  - c_sandwich_gram_atk
  - c_sandwich_hate_atk
  - c_sandwich_nli_atk
  - c_sandwich_sent_atk
  - c_sandwich_spam_atk
  - c_sandwich_summ_atk
  - c_instr_dup_atk
  - c_instr_gram_atk
  - c_instr_hate_atk
  - c_instr_nli_atk
  - c_instr_sent_atk
  - c_instr_spam_atk
  - c_instr_summ_atk
  page: 1
coverage:
  R1: direct
  R2clean: direct
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 7(a) (ASV, averaged over injected tasks)
    R2clean: Table 7(b) (PNA-T)

`signedprompt_arxiv2024` (2401.07612)

In [ ]:
%%writefile reports/records/signedprompt_arxiv2024.yaml
report:
  id: signedprompt_arxiv2024
  slug: '2401.07612'
  title: 'Signed-Prompt: A New Approach to Prevent Prompt Injection Attacks Against LLM-Integrated Applications'
  first_author: Xuchen Suo
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2401.07612'
  version_read: arXiv v1
  first_date: '2024-01-15'
  read: F
  role: proposes
  code_release: none found in the paper
  later_version: DOI 10.1063/5.0222987 (from the search record; not read)
  extraction_scope: 'whole report (7 pages): Tables 1 to 3 read; Table 3 coded (encoder accuracy in Table 1 is not a defence outcome); the abstract claim'
defences:
- id: sp_pe_gpt4
  name: Signed-Prompt (prompt-engineered adjusted LLM, LLM-PE)
  origin: this_report
  base_checkpoint: ChatGPT-4
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: no_ablation
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  anchor: this paper employs prompt engineering to construct a Large Language Model (LLM-PE)
  released_weights: not applicable
  components: &id001
  - an encoder (ChatGPT-4 prompted) replaces the authorised user's sensitive instruction word with a rare signature string (delete -> toeowx)
  - the adjusted model emits the real command only for the signed form and an invalid command for the raw word
  layout: 'one prompt: signed user instruction and unsigned external content'
  layout_compat: &id002
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
- id: sp_ft_chatglm6b
  name: Signed-Prompt (fine-tuned adjusted LLM, LLM-FT)
  origin: this_report
  base_checkpoint: ChatGLM-6B
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: no_ablation
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  anchor: this paper develops another LLM (LLM-FT) based on the ChatGLM-6B model
  released_weights: not stated
  components: *id001
  layout: 'one prompt: signed user instruction and unsigned external content'
  layout_compat: *id002
evaluations:
- id: ev_sp_pe_gpt4_direct_user
  defence_id: sp_pe_gpt4
  benchmark: Delete Command Dataset, Direct group, signed user commands
  attacker_type: none
  attacker_access: not_applicable
  judge: actual output equals the expected output (the true deletion command)
  layout_same_as_defence: 'yes'
  measured_by: signedprompt_arxiv2024
  page: 6
  decoding: not reported
- id: ev_sp_pe_gpt4_direct_atk
  defence_id: sp_pe_gpt4
  benchmark: Delete Command Dataset, Direct group, unsigned attacker commands
  attacker_type: static
  attacker_access: black_box
  judge: attacker succeeds if the model outputs the deletion command
  layout_same_as_defence: 'yes'
  measured_by: signedprompt_arxiv2024
  page: 6
  decoding: not reported
- id: ev_sp_pe_gpt4_multi_user
  defence_id: sp_pe_gpt4
  benchmark: Delete Command Dataset, Multilingual group, signed user commands
  attacker_type: none
  attacker_access: not_applicable
  judge: actual output equals the expected output (the true deletion command)
  layout_same_as_defence: 'yes'
  measured_by: signedprompt_arxiv2024
  page: 6
  decoding: not reported
- id: ev_sp_pe_gpt4_multi_atk
  defence_id: sp_pe_gpt4
  benchmark: Delete Command Dataset, Multilingual group, unsigned attacker commands
  attacker_type: static
  attacker_access: black_box
  judge: attacker succeeds if the model outputs the deletion command
  layout_same_as_defence: 'yes'
  measured_by: signedprompt_arxiv2024
  page: 6
  decoding: not reported
- id: ev_sp_pe_gpt4_varied_user
  defence_id: sp_pe_gpt4
  benchmark: Delete Command Dataset, Varied Exp. group, signed user commands
  attacker_type: none
  attacker_access: not_applicable
  judge: actual output equals the expected output (the true deletion command)
  layout_same_as_defence: 'yes'
  measured_by: signedprompt_arxiv2024
  page: 6
  decoding: not reported
- id: ev_sp_pe_gpt4_varied_atk
  defence_id: sp_pe_gpt4
  benchmark: Delete Command Dataset, Varied Exp. group, unsigned attacker commands
  attacker_type: static
  attacker_access: black_box
  judge: attacker succeeds if the model outputs the deletion command
  layout_same_as_defence: 'yes'
  measured_by: signedprompt_arxiv2024
  page: 6
  decoding: not reported
- id: ev_sp_pe_gpt4_impl_user
  defence_id: sp_pe_gpt4
  benchmark: Delete Command Dataset, Implication group, signed user commands
  attacker_type: none
  attacker_access: not_applicable
  judge: actual output equals the expected output (the true deletion command)
  layout_same_as_defence: 'yes'
  measured_by: signedprompt_arxiv2024
  page: 6
  decoding: not reported
- id: ev_sp_pe_gpt4_impl_atk
  defence_id: sp_pe_gpt4
  benchmark: Delete Command Dataset, Implication group, unsigned attacker commands
  attacker_type: static
  attacker_access: black_box
  judge: attacker succeeds if the model outputs the deletion command
  layout_same_as_defence: 'yes'
  measured_by: signedprompt_arxiv2024
  page: 6
  decoding: not reported
- id: ev_sp_ft_chatglm6b_direct_user
  defence_id: sp_ft_chatglm6b
  benchmark: Delete Command Dataset, Direct group, signed user commands
  attacker_type: none
  attacker_access: not_applicable
  judge: actual output equals the expected output (the true deletion command)
  layout_same_as_defence: 'yes'
  measured_by: signedprompt_arxiv2024
  page: 6
  decoding: not reported
- id: ev_sp_ft_chatglm6b_direct_atk
  defence_id: sp_ft_chatglm6b
  benchmark: Delete Command Dataset, Direct group, unsigned attacker commands
  attacker_type: static
  attacker_access: black_box
  judge: attacker succeeds if the model outputs the deletion command
  layout_same_as_defence: 'yes'
  measured_by: signedprompt_arxiv2024
  page: 6
  decoding: not reported
- id: ev_sp_ft_chatglm6b_multi_user
  defence_id: sp_ft_chatglm6b
  benchmark: Delete Command Dataset, Multilingual group, signed user commands
  attacker_type: none
  attacker_access: not_applicable
  judge: actual output equals the expected output (the true deletion command)
  layout_same_as_defence: 'yes'
  measured_by: signedprompt_arxiv2024
  page: 6
  decoding: not reported
- id: ev_sp_ft_chatglm6b_multi_atk
  defence_id: sp_ft_chatglm6b
  benchmark: Delete Command Dataset, Multilingual group, unsigned attacker commands
  attacker_type: static
  attacker_access: black_box
  judge: attacker succeeds if the model outputs the deletion command
  layout_same_as_defence: 'yes'
  measured_by: signedprompt_arxiv2024
  page: 6
  decoding: not reported
- id: ev_sp_ft_chatglm6b_varied_user
  defence_id: sp_ft_chatglm6b
  benchmark: Delete Command Dataset, Varied Exp. group, signed user commands
  attacker_type: none
  attacker_access: not_applicable
  judge: actual output equals the expected output (the true deletion command)
  layout_same_as_defence: 'yes'
  measured_by: signedprompt_arxiv2024
  page: 6
  decoding: not reported
- id: ev_sp_ft_chatglm6b_varied_atk
  defence_id: sp_ft_chatglm6b
  benchmark: Delete Command Dataset, Varied Exp. group, unsigned attacker commands
  attacker_type: static
  attacker_access: black_box
  judge: attacker succeeds if the model outputs the deletion command
  layout_same_as_defence: 'yes'
  measured_by: signedprompt_arxiv2024
  page: 6
  decoding: not reported
- id: ev_sp_ft_chatglm6b_impl_user
  defence_id: sp_ft_chatglm6b
  benchmark: Delete Command Dataset, Implication group, signed user commands
  attacker_type: none
  attacker_access: not_applicable
  judge: actual output equals the expected output (the true deletion command)
  layout_same_as_defence: 'yes'
  measured_by: signedprompt_arxiv2024
  page: 6
  decoding: not reported
- id: ev_sp_ft_chatglm6b_impl_atk
  defence_id: sp_ft_chatglm6b
  benchmark: Delete Command Dataset, Implication group, unsigned attacker commands
  attacker_type: static
  attacker_access: black_box
  judge: attacker succeeds if the model outputs the deletion command
  layout_same_as_defence: 'yes'
  measured_by: signedprompt_arxiv2024
  page: 6
  decoding: not reported
results:
- id: r_sp_pe_gpt4_direct_user
  evaluation_id: ev_sp_pe_gpt4_direct_user
  quantity: R2clean
  coverage: direct
  metric_as_printed: Corr. Rate
  value: '100'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 3
- id: r_sp_pe_gpt4_direct_atk
  evaluation_id: ev_sp_pe_gpt4_direct_atk
  quantity: R1
  coverage: direct
  metric_as_printed: Succ. Rate (attacker)
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 3
- id: r_sp_pe_gpt4_multi_user
  evaluation_id: ev_sp_pe_gpt4_multi_user
  quantity: R2clean
  coverage: direct
  metric_as_printed: Corr. Rate
  value: '100'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 3
- id: r_sp_pe_gpt4_multi_atk
  evaluation_id: ev_sp_pe_gpt4_multi_atk
  quantity: R1
  coverage: direct
  metric_as_printed: Succ. Rate (attacker)
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 3
- id: r_sp_pe_gpt4_varied_user
  evaluation_id: ev_sp_pe_gpt4_varied_user
  quantity: R2clean
  coverage: direct
  metric_as_printed: Corr. Rate
  value: '100'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 3
- id: r_sp_pe_gpt4_varied_atk
  evaluation_id: ev_sp_pe_gpt4_varied_atk
  quantity: R1
  coverage: direct
  metric_as_printed: Succ. Rate (attacker)
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 3
- id: r_sp_pe_gpt4_impl_user
  evaluation_id: ev_sp_pe_gpt4_impl_user
  quantity: R2clean
  coverage: direct
  metric_as_printed: Corr. Rate
  value: '100'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 3
- id: r_sp_pe_gpt4_impl_atk
  evaluation_id: ev_sp_pe_gpt4_impl_atk
  quantity: R1
  coverage: direct
  metric_as_printed: Succ. Rate (attacker)
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 3
- id: r_sp_ft_chatglm6b_direct_user
  evaluation_id: ev_sp_ft_chatglm6b_direct_user
  quantity: R2clean
  coverage: direct
  metric_as_printed: Corr. Rate
  value: '86.67'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 3
- id: r_sp_ft_chatglm6b_direct_atk
  evaluation_id: ev_sp_ft_chatglm6b_direct_atk
  quantity: R1
  coverage: direct
  metric_as_printed: Succ. Rate (attacker)
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 3
- id: r_sp_ft_chatglm6b_multi_user
  evaluation_id: ev_sp_ft_chatglm6b_multi_user
  quantity: R2clean
  coverage: direct
  metric_as_printed: Corr. Rate
  value: '73.34'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 3
- id: r_sp_ft_chatglm6b_multi_atk
  evaluation_id: ev_sp_ft_chatglm6b_multi_atk
  quantity: R1
  coverage: direct
  metric_as_printed: Succ. Rate (attacker)
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 3
- id: r_sp_ft_chatglm6b_varied_user
  evaluation_id: ev_sp_ft_chatglm6b_varied_user
  quantity: R2clean
  coverage: direct
  metric_as_printed: Corr. Rate
  value: '100'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 3
- id: r_sp_ft_chatglm6b_varied_atk
  evaluation_id: ev_sp_ft_chatglm6b_varied_atk
  quantity: R1
  coverage: direct
  metric_as_printed: Succ. Rate (attacker)
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 3
- id: r_sp_ft_chatglm6b_impl_user
  evaluation_id: ev_sp_ft_chatglm6b_impl_user
  quantity: R2clean
  coverage: direct
  metric_as_printed: Corr. Rate
  value: '100'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 3
- id: r_sp_ft_chatglm6b_impl_atk
  evaluation_id: ev_sp_ft_chatglm6b_impl_atk
  quantity: R1
  coverage: direct
  metric_as_printed: Succ. Rate (attacker)
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 3
claims:
- id: k1
  text: substantial resistance to various types of prompt injection attacks
  location: abstract
  anchor: showing substantial resistance to various types of prompt injection attacks
  jobs:
  - R1
  supported_by: []
  page: 1
coverage:
  R1: direct
  R2clean: direct
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 3 (attacker success, defended models only; no undefended run reported)
    R2clean: Table 3 (correct handling of signed user commands)

`agentdojo_neurips2024` (2406.13352)

In [ ]:
%%writefile reports/records/agentdojo_neurips2024.yaml
report:
  id: agentdojo_neurips2024
  slug: '2406.13352'
  title: 'AgentDojo: A Dynamic Environment to Evaluate Prompt Injection Attacks and Defenses for LLM Agents'
  first_author: Edoardo Debenedetti
  venue: NeurIPS 2024 Datasets and Benchmarks Track
  peer_reviewed: 'yes'
  arxiv: '2406.13352'
  version_read: arXiv v3 (carries the NeurIPS 2024 Datasets and Benchmarks footer)
  first_date: '2024-06-19'
  read: F
  role: benchmark
  code_release: https://agentdojo.spylab.ai
  extraction_scope: 'Sections 3, 4.1 and 4.3 and Appendix C: the No defense, Delimiting and Repeat prompt columns of Table 5 (GPT-4o); the abstract claim (the PI detector, a detector, and the tool filter, a system-level isolation,
    are not coded; Tables 3 and 4 and the attack ablations are attack measurements, not coded)'
defences:
- id: delim_gpt4o
  name: Data delimiters (following spotlighting)
  origin: other_report
  base_checkpoint: GPT-4o
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 8
  components:
  - every tool output wrapped in special delimiters
  - system prompt tells the model to ignore instructions inside them
  layout: user task in the user turn, tool outputs as tool messages in the agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: we format all tool outputs with special delimiters
- id: repeat_gpt4o
  name: Prompt sandwiching (repeat the user prompt)
  origin: other_report
  base_checkpoint: GPT-4o
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 8
  components:
  - the user instruction repeated after each function call
  layout: user task in the user turn, tool outputs as tool messages in the agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: which repeats the user instructions after each function call
evaluations:
- id: ev_none_clean
  defence_id: none:GPT-4o
  benchmark: AgentDojo, 97 user tasks without injection
  attacker_type: none
  attacker_access: not_applicable
  judge: deterministic utility function over the model output and environment state
  layout_same_as_defence: 'yes'
  measured_by: agentdojo_neurips2024
  page: 20
  sample_size: '97'
  decoding: not reported
  pages:
    judge: 4
    sample_size: 7
  baseline_kind: off_the_shelf
- id: ev_none_attack
  defence_id: none:GPT-4o
  benchmark: AgentDojo, 629 security test cases
  attacker_type: static
  attacker_access: black_box
  judge: utility function (user task) and security function (attacker goal met)
  layout_same_as_defence: 'yes'
  measured_by: agentdojo_neurips2024
  page: 20
  attacker_budget: Important message prompt injection addressing the model and the user by name
  sample_size: '629'
  decoding: not reported
  pages:
    judge: 5
    sample_size: 7
    attacker_budget: 8
  baseline_kind: off_the_shelf
- id: ev_delim_clean
  defence_id: delim_gpt4o
  benchmark: AgentDojo, 97 user tasks without injection
  attacker_type: none
  attacker_access: not_applicable
  judge: deterministic utility function over the model output and environment state
  layout_same_as_defence: 'yes'
  measured_by: agentdojo_neurips2024
  page: 20
  sample_size: '97'
  decoding: not reported
  pages:
    judge: 4
    sample_size: 7
- id: ev_delim_attack
  defence_id: delim_gpt4o
  benchmark: AgentDojo, 629 security test cases
  attacker_type: static
  attacker_access: black_box
  judge: utility function (user task) and security function (attacker goal met)
  layout_same_as_defence: 'yes'
  measured_by: agentdojo_neurips2024
  page: 20
  attacker_budget: Important message prompt injection addressing the model and the user by name
  sample_size: '629'
  decoding: not reported
  pages:
    judge: 5
    sample_size: 7
    attacker_budget: 8
- id: ev_repeat_clean
  defence_id: repeat_gpt4o
  benchmark: AgentDojo, 97 user tasks without injection
  attacker_type: none
  attacker_access: not_applicable
  judge: deterministic utility function over the model output and environment state
  layout_same_as_defence: 'yes'
  measured_by: agentdojo_neurips2024
  page: 20
  sample_size: '97'
  decoding: not reported
  pages:
    judge: 4
    sample_size: 7
- id: ev_repeat_attack
  defence_id: repeat_gpt4o
  benchmark: AgentDojo, 629 security test cases
  attacker_type: static
  attacker_access: black_box
  judge: utility function (user task) and security function (attacker goal met)
  layout_same_as_defence: 'yes'
  measured_by: agentdojo_neurips2024
  page: 20
  attacker_budget: Important message prompt injection addressing the model and the user by name
  sample_size: '629'
  decoding: not reported
  pages:
    judge: 5
    sample_size: 7
    attacker_budget: 8
results:
- id: r_none_util
  evaluation_id: ev_none_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Benign utility
  value: '69.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 5
  spread: 95% CI ±3.6
- id: r_none_util_atk
  evaluation_id: ev_none_attack
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility w. attack
  value: '50.01'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 5
  spread: 95% CI ±3.9
- id: r_none_asr
  evaluation_id: ev_none_attack
  quantity: R1
  coverage: direct
  metric_as_printed: Targeted ASR
  value: '57.69'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 5
  spread: 95% CI ±3.9
- id: r_delim_util
  evaluation_id: ev_delim_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Benign utility
  value: '72.66'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 5
  spread: 95% CI ±3.5
- id: r_delim_util_atk
  evaluation_id: ev_delim_attack
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility w. attack
  value: '55.64'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 5
  spread: 95% CI ±3.9
- id: r_delim_asr
  evaluation_id: ev_delim_attack
  quantity: R1
  coverage: direct
  metric_as_printed: Targeted ASR
  value: '41.65'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 5
  spread: 95% CI ±3.9
- id: r_repeat_util
  evaluation_id: ev_repeat_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: Benign utility
  value: '85.53'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 5
  spread: 95% CI ±2.8
- id: r_repeat_util_atk
  evaluation_id: ev_repeat_attack
  quantity: R2attack
  coverage: direct
  metric_as_printed: Utility w. attack
  value: '67.25'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 5
  spread: 95% CI ±3.7
- id: r_repeat_asr
  evaluation_id: ev_repeat_attack
  quantity: R1
  coverage: direct
  metric_as_printed: Targeted ASR
  value: '27.82'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 5
  spread: 95% CI ±3.5
comparisons:
- id: c_delim_clean
  left: ev_delim_clean
  right: ev_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 20
- id: c_delim_attack
  left: ev_delim_attack
  right: ev_none_attack
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 20
- id: c_repeat_clean
  left: ev_repeat_clean
  right: ev_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 20
- id: c_repeat_attack
  left: ev_repeat_attack
  right: ev_none_attack
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 20
claims:
- id: k1
  text: existing attacks break some security properties but not all; state-of-the-art LLMs fail many tasks even without attacks
  location: abstract
  anchor: existing prompt injection attacks break some security properties but not all
  jobs:
  - R1
  - R2
  supported_by:
  - c_delim_attack
  - c_repeat_attack
  page: 1
coverage:
  R1: direct
  R2clean: direct
  R2attack: direct
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 5 (targeted ASR)
    R2clean: Table 5 (benign utility)
    R2attack: Table 5 (utility with attack)

`asb_iclr2025` (2410.02644)

In [ ]:
%%writefile reports/records/asb_iclr2025.yaml
report:
  id: asb_iclr2025
  slug: '2410.02644'
  title: 'Agent Security Bench (ASB): Formalizing and Benchmarking Attacks and Defenses in LLM-based Agents'
  first_author: Hanrong Zhang
  venue: ICLR 2025
  peer_reviewed: 'yes'
  arxiv: '2410.02644'
  version_read: arXiv v4 (marked as published at ICLR 2025)
  first_date: '2024-10-03'
  read: F
  role: benchmark
  code_release: https://github.com/agiresearch/ASB
  extraction_scope: 'Sections 3 to 5.4 and Appendix D.2: Table 8 (defences against indirect prompt injection, all 13 LLMs: no defence, Delimiter, Instruction, Sandwich) and the matching columns of Table 19 (no-attack PNA);
    the abstract claim (direct injection in Table 7, paraphrasing and dynamic prompt rewriting, which change the input before an unchanged model, memory and backdoor attacks not coded)'
defences:
- id: delim_m00
  name: Delimiters
  origin: other_report
  base_checkpoint: Gemma2-9B
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - delimiters around the user query so the agent executes only the query inside them
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Use delimiters to encapsulate the user query
- id: instr_m00
  name: Instructional prevention
  origin: other_report
  base_checkpoint: Gemma2-9B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - instruction prompt rebuilt to tell the agent to disregard all commands except the user's
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Reconstruct the instruction prompt to ensure the agent disregards
- id: sand_m00
  name: Sandwich prevention
  origin: other_report
  base_checkpoint: Gemma2-9B
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - an extra instruction prompt attached after each tool response
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Attach an additional instruction prompt at the end of the tool’s response
- id: delim_m01
  name: Delimiters
  origin: other_report
  base_checkpoint: Gemma2-27B
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - delimiters around the user query so the agent executes only the query inside them
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Use delimiters to encapsulate the user query
- id: instr_m01
  name: Instructional prevention
  origin: other_report
  base_checkpoint: Gemma2-27B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - instruction prompt rebuilt to tell the agent to disregard all commands except the user's
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Reconstruct the instruction prompt to ensure the agent disregards
- id: sand_m01
  name: Sandwich prevention
  origin: other_report
  base_checkpoint: Gemma2-27B
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - an extra instruction prompt attached after each tool response
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Attach an additional instruction prompt at the end of the tool’s response
- id: delim_m02
  name: Delimiters
  origin: other_report
  base_checkpoint: LLaMA3-8B
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - delimiters around the user query so the agent executes only the query inside them
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Use delimiters to encapsulate the user query
- id: instr_m02
  name: Instructional prevention
  origin: other_report
  base_checkpoint: LLaMA3-8B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - instruction prompt rebuilt to tell the agent to disregard all commands except the user's
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Reconstruct the instruction prompt to ensure the agent disregards
- id: sand_m02
  name: Sandwich prevention
  origin: other_report
  base_checkpoint: LLaMA3-8B
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - an extra instruction prompt attached after each tool response
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Attach an additional instruction prompt at the end of the tool’s response
- id: delim_m03
  name: Delimiters
  origin: other_report
  base_checkpoint: LLaMA3-70B
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - delimiters around the user query so the agent executes only the query inside them
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Use delimiters to encapsulate the user query
- id: instr_m03
  name: Instructional prevention
  origin: other_report
  base_checkpoint: LLaMA3-70B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - instruction prompt rebuilt to tell the agent to disregard all commands except the user's
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Reconstruct the instruction prompt to ensure the agent disregards
- id: sand_m03
  name: Sandwich prevention
  origin: other_report
  base_checkpoint: LLaMA3-70B
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - an extra instruction prompt attached after each tool response
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Attach an additional instruction prompt at the end of the tool’s response
- id: delim_m04
  name: Delimiters
  origin: other_report
  base_checkpoint: LLaMA3.1-8B
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - delimiters around the user query so the agent executes only the query inside them
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Use delimiters to encapsulate the user query
- id: instr_m04
  name: Instructional prevention
  origin: other_report
  base_checkpoint: LLaMA3.1-8B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - instruction prompt rebuilt to tell the agent to disregard all commands except the user's
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Reconstruct the instruction prompt to ensure the agent disregards
- id: sand_m04
  name: Sandwich prevention
  origin: other_report
  base_checkpoint: LLaMA3.1-8B
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - an extra instruction prompt attached after each tool response
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Attach an additional instruction prompt at the end of the tool’s response
- id: delim_m05
  name: Delimiters
  origin: other_report
  base_checkpoint: LLaMA3.1-70B
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - delimiters around the user query so the agent executes only the query inside them
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Use delimiters to encapsulate the user query
- id: instr_m05
  name: Instructional prevention
  origin: other_report
  base_checkpoint: LLaMA3.1-70B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - instruction prompt rebuilt to tell the agent to disregard all commands except the user's
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Reconstruct the instruction prompt to ensure the agent disregards
- id: sand_m05
  name: Sandwich prevention
  origin: other_report
  base_checkpoint: LLaMA3.1-70B
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - an extra instruction prompt attached after each tool response
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Attach an additional instruction prompt at the end of the tool’s response
- id: delim_m06
  name: Delimiters
  origin: other_report
  base_checkpoint: Mixtral-8x7B
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - delimiters around the user query so the agent executes only the query inside them
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Use delimiters to encapsulate the user query
- id: instr_m06
  name: Instructional prevention
  origin: other_report
  base_checkpoint: Mixtral-8x7B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - instruction prompt rebuilt to tell the agent to disregard all commands except the user's
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Reconstruct the instruction prompt to ensure the agent disregards
- id: sand_m06
  name: Sandwich prevention
  origin: other_report
  base_checkpoint: Mixtral-8x7B
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - an extra instruction prompt attached after each tool response
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Attach an additional instruction prompt at the end of the tool’s response
- id: delim_m07
  name: Delimiters
  origin: other_report
  base_checkpoint: Qwen2-7B
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - delimiters around the user query so the agent executes only the query inside them
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Use delimiters to encapsulate the user query
- id: instr_m07
  name: Instructional prevention
  origin: other_report
  base_checkpoint: Qwen2-7B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - instruction prompt rebuilt to tell the agent to disregard all commands except the user's
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Reconstruct the instruction prompt to ensure the agent disregards
- id: sand_m07
  name: Sandwich prevention
  origin: other_report
  base_checkpoint: Qwen2-7B
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - an extra instruction prompt attached after each tool response
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Attach an additional instruction prompt at the end of the tool’s response
- id: delim_m08
  name: Delimiters
  origin: other_report
  base_checkpoint: Qwen2-72B
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - delimiters around the user query so the agent executes only the query inside them
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Use delimiters to encapsulate the user query
- id: instr_m08
  name: Instructional prevention
  origin: other_report
  base_checkpoint: Qwen2-72B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - instruction prompt rebuilt to tell the agent to disregard all commands except the user's
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Reconstruct the instruction prompt to ensure the agent disregards
- id: sand_m08
  name: Sandwich prevention
  origin: other_report
  base_checkpoint: Qwen2-72B
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - an extra instruction prompt attached after each tool response
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Attach an additional instruction prompt at the end of the tool’s response
- id: delim_m09
  name: Delimiters
  origin: other_report
  base_checkpoint: Claude-3.5 Sonnet
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - delimiters around the user query so the agent executes only the query inside them
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Use delimiters to encapsulate the user query
- id: instr_m09
  name: Instructional prevention
  origin: other_report
  base_checkpoint: Claude-3.5 Sonnet
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - instruction prompt rebuilt to tell the agent to disregard all commands except the user's
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Reconstruct the instruction prompt to ensure the agent disregards
- id: sand_m09
  name: Sandwich prevention
  origin: other_report
  base_checkpoint: Claude-3.5 Sonnet
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - an extra instruction prompt attached after each tool response
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Attach an additional instruction prompt at the end of the tool’s response
- id: delim_m10
  name: Delimiters
  origin: other_report
  base_checkpoint: GPT-3.5 Turbo
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - delimiters around the user query so the agent executes only the query inside them
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Use delimiters to encapsulate the user query
- id: instr_m10
  name: Instructional prevention
  origin: other_report
  base_checkpoint: GPT-3.5 Turbo
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - instruction prompt rebuilt to tell the agent to disregard all commands except the user's
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Reconstruct the instruction prompt to ensure the agent disregards
- id: sand_m10
  name: Sandwich prevention
  origin: other_report
  base_checkpoint: GPT-3.5 Turbo
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - an extra instruction prompt attached after each tool response
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Attach an additional instruction prompt at the end of the tool’s response
- id: delim_m11
  name: Delimiters
  origin: other_report
  base_checkpoint: GPT-4o
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - delimiters around the user query so the agent executes only the query inside them
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Use delimiters to encapsulate the user query
- id: instr_m11
  name: Instructional prevention
  origin: other_report
  base_checkpoint: GPT-4o
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - instruction prompt rebuilt to tell the agent to disregard all commands except the user's
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Reconstruct the instruction prompt to ensure the agent disregards
- id: sand_m11
  name: Sandwich prevention
  origin: other_report
  base_checkpoint: GPT-4o
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - an extra instruction prompt attached after each tool response
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Attach an additional instruction prompt at the end of the tool’s response
- id: delim_m12
  name: Delimiters
  origin: other_report
  base_checkpoint: GPT-4o-mini
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - delimiters around the user query so the agent executes only the query inside them
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Use delimiters to encapsulate the user query
- id: instr_m12
  name: Instructional prevention
  origin: other_report
  base_checkpoint: GPT-4o-mini
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - instruction prompt rebuilt to tell the agent to disregard all commands except the user's
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Reconstruct the instruction prompt to ensure the agent disregards
- id: sand_m12
  name: Sandwich prevention
  origin: other_report
  base_checkpoint: GPT-4o-mini
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 7
  components:
  - an extra instruction prompt attached after each tool response
  layout: system prompt, user query and tool responses in an agent loop
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Attach an additional instruction prompt at the end of the tool’s response
evaluations:
- id: ev_m00_none_ipi
  defence_id: none:Gemma2-9B
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
  baseline_kind: off_the_shelf
- id: ev_m00_none_pna
  defence_id: none:Gemma2-9B
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
  baseline_kind: off_the_shelf
- id: ev_m00_delim_ipi
  defence_id: delim_m00
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m00_delim_pna
  defence_id: delim_m00
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
- id: ev_m00_instr_ipi
  defence_id: instr_m00
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m00_instr_pna
  defence_id: instr_m00
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
- id: ev_m00_sand_ipi
  defence_id: sand_m00
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m00_sand_pna
  defence_id: sand_m00
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
- id: ev_m01_none_ipi
  defence_id: none:Gemma2-27B
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
  baseline_kind: off_the_shelf
- id: ev_m01_none_pna
  defence_id: none:Gemma2-27B
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
  baseline_kind: off_the_shelf
- id: ev_m01_delim_ipi
  defence_id: delim_m01
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m01_delim_pna
  defence_id: delim_m01
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
- id: ev_m01_instr_ipi
  defence_id: instr_m01
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m01_instr_pna
  defence_id: instr_m01
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
- id: ev_m01_sand_ipi
  defence_id: sand_m01
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m01_sand_pna
  defence_id: sand_m01
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
- id: ev_m02_none_ipi
  defence_id: none:LLaMA3-8B
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
  baseline_kind: off_the_shelf
- id: ev_m02_none_pna
  defence_id: none:LLaMA3-8B
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
  baseline_kind: off_the_shelf
- id: ev_m02_delim_ipi
  defence_id: delim_m02
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m02_delim_pna
  defence_id: delim_m02
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
- id: ev_m02_instr_ipi
  defence_id: instr_m02
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m02_instr_pna
  defence_id: instr_m02
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
- id: ev_m02_sand_ipi
  defence_id: sand_m02
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m02_sand_pna
  defence_id: sand_m02
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
- id: ev_m03_none_ipi
  defence_id: none:LLaMA3-70B
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
  baseline_kind: off_the_shelf
- id: ev_m03_none_pna
  defence_id: none:LLaMA3-70B
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
  baseline_kind: off_the_shelf
- id: ev_m03_delim_ipi
  defence_id: delim_m03
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m03_delim_pna
  defence_id: delim_m03
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
- id: ev_m03_instr_ipi
  defence_id: instr_m03
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m03_instr_pna
  defence_id: instr_m03
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
- id: ev_m03_sand_ipi
  defence_id: sand_m03
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m03_sand_pna
  defence_id: sand_m03
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
- id: ev_m04_none_ipi
  defence_id: none:LLaMA3.1-8B
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
  baseline_kind: off_the_shelf
- id: ev_m04_none_pna
  defence_id: none:LLaMA3.1-8B
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
  baseline_kind: off_the_shelf
- id: ev_m04_delim_ipi
  defence_id: delim_m04
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m04_delim_pna
  defence_id: delim_m04
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
- id: ev_m04_instr_ipi
  defence_id: instr_m04
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m04_instr_pna
  defence_id: instr_m04
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
- id: ev_m04_sand_ipi
  defence_id: sand_m04
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m04_sand_pna
  defence_id: sand_m04
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
- id: ev_m05_none_ipi
  defence_id: none:LLaMA3.1-70B
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
  baseline_kind: off_the_shelf
- id: ev_m05_none_pna
  defence_id: none:LLaMA3.1-70B
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
  baseline_kind: off_the_shelf
- id: ev_m05_delim_ipi
  defence_id: delim_m05
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m05_delim_pna
  defence_id: delim_m05
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
- id: ev_m05_instr_ipi
  defence_id: instr_m05
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m05_instr_pna
  defence_id: instr_m05
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
- id: ev_m05_sand_ipi
  defence_id: sand_m05
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m05_sand_pna
  defence_id: sand_m05
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
- id: ev_m06_none_ipi
  defence_id: none:Mixtral-8x7B
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
  baseline_kind: off_the_shelf
- id: ev_m06_none_pna
  defence_id: none:Mixtral-8x7B
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
  baseline_kind: off_the_shelf
- id: ev_m06_delim_ipi
  defence_id: delim_m06
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m06_delim_pna
  defence_id: delim_m06
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
- id: ev_m06_instr_ipi
  defence_id: instr_m06
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m06_instr_pna
  defence_id: instr_m06
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
- id: ev_m06_sand_ipi
  defence_id: sand_m06
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m06_sand_pna
  defence_id: sand_m06
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
- id: ev_m07_none_ipi
  defence_id: none:Qwen2-7B
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
  baseline_kind: off_the_shelf
- id: ev_m07_none_pna
  defence_id: none:Qwen2-7B
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
  baseline_kind: off_the_shelf
- id: ev_m07_delim_ipi
  defence_id: delim_m07
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m07_delim_pna
  defence_id: delim_m07
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
- id: ev_m07_instr_ipi
  defence_id: instr_m07
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m07_instr_pna
  defence_id: instr_m07
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
- id: ev_m07_sand_ipi
  defence_id: sand_m07
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m07_sand_pna
  defence_id: sand_m07
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
- id: ev_m08_none_ipi
  defence_id: none:Qwen2-72B
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
  baseline_kind: off_the_shelf
- id: ev_m08_none_pna
  defence_id: none:Qwen2-72B
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
  baseline_kind: off_the_shelf
- id: ev_m08_delim_ipi
  defence_id: delim_m08
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m08_delim_pna
  defence_id: delim_m08
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
- id: ev_m08_instr_ipi
  defence_id: instr_m08
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m08_instr_pna
  defence_id: instr_m08
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
- id: ev_m08_sand_ipi
  defence_id: sand_m08
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m08_sand_pna
  defence_id: sand_m08
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
- id: ev_m09_none_ipi
  defence_id: none:Claude-3.5 Sonnet
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
  baseline_kind: off_the_shelf
- id: ev_m09_none_pna
  defence_id: none:Claude-3.5 Sonnet
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
  baseline_kind: off_the_shelf
- id: ev_m09_delim_ipi
  defence_id: delim_m09
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m09_delim_pna
  defence_id: delim_m09
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
- id: ev_m09_instr_ipi
  defence_id: instr_m09
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m09_instr_pna
  defence_id: instr_m09
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
- id: ev_m09_sand_ipi
  defence_id: sand_m09
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m09_sand_pna
  defence_id: sand_m09
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
- id: ev_m10_none_ipi
  defence_id: none:GPT-3.5 Turbo
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
  baseline_kind: off_the_shelf
- id: ev_m10_none_pna
  defence_id: none:GPT-3.5 Turbo
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
  baseline_kind: off_the_shelf
- id: ev_m10_delim_ipi
  defence_id: delim_m10
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m10_delim_pna
  defence_id: delim_m10
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
- id: ev_m10_instr_ipi
  defence_id: instr_m10
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m10_instr_pna
  defence_id: instr_m10
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
- id: ev_m10_sand_ipi
  defence_id: sand_m10
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m10_sand_pna
  defence_id: sand_m10
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
- id: ev_m11_none_ipi
  defence_id: none:GPT-4o
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
  baseline_kind: off_the_shelf
- id: ev_m11_none_pna
  defence_id: none:GPT-4o
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
  baseline_kind: off_the_shelf
- id: ev_m11_delim_ipi
  defence_id: delim_m11
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m11_delim_pna
  defence_id: delim_m11
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
- id: ev_m11_instr_ipi
  defence_id: instr_m11
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m11_instr_pna
  defence_id: instr_m11
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
- id: ev_m11_sand_ipi
  defence_id: sand_m11
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m11_sand_pna
  defence_id: sand_m11
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
- id: ev_m12_none_ipi
  defence_id: none:GPT-4o-mini
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
  baseline_kind: off_the_shelf
- id: ev_m12_none_pna
  defence_id: none:GPT-4o-mini
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
  baseline_kind: off_the_shelf
- id: ev_m12_delim_ipi
  defence_id: delim_m12
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m12_delim_pna
  defence_id: delim_m12
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
- id: ev_m12_instr_ipi
  defence_id: instr_m12
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m12_instr_pna
  defence_id: instr_m12
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
- id: ev_m12_sand_ipi
  defence_id: sand_m12
  benchmark: ASB, indirect prompt injection in tool responses (10 scenarios, 400 attack tasks)
  attacker_type: static
  attacker_access: black_box
  judge: agent calls the attack-specific tool
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 10
  attacker_budget: average over the five IPI attack types
  decoding: not reported
  pages:
    benchmark: 8
    judge: 8
- id: ev_m12_sand_pna
  defence_id: sand_m12
  benchmark: ASB tasks without attack
  attacker_type: none
  attacker_access: not_applicable
  judge: percentage of tasks completed (PNA)
  layout_same_as_defence: 'yes'
  measured_by: asb_iclr2025
  page: 33
  decoding: not reported
  pages:
    judge: 8
results:
- id: r_m00_none_ipi
  evaluation_id: ev_m00_none_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '14.50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m00_none_pna
  evaluation_id: ev_m00_none_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '10.75'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m00_delim_ipi
  evaluation_id: ev_m00_delim_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '10.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m00_delim_pna
  evaluation_id: ev_m00_delim_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '8.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m00_instr_ipi
  evaluation_id: ev_m00_instr_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '13.50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m00_instr_pna
  evaluation_id: ev_m00_instr_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '8.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m00_sand_ipi
  evaluation_id: ev_m00_sand_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '10.25'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m00_sand_pna
  evaluation_id: ev_m00_sand_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '14.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m01_none_ipi
  evaluation_id: ev_m01_none_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '15.50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m01_none_pna
  evaluation_id: ev_m01_none_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '31.50'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m01_delim_ipi
  evaluation_id: ev_m01_delim_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '13.75'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m01_delim_pna
  evaluation_id: ev_m01_delim_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '14.75'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m01_instr_ipi
  evaluation_id: ev_m01_instr_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '16.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m01_instr_pna
  evaluation_id: ev_m01_instr_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '23.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m01_sand_ipi
  evaluation_id: ev_m01_sand_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '14.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m01_sand_pna
  evaluation_id: ev_m01_sand_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '27.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m02_none_ipi
  evaluation_id: ev_m02_none_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '11.50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m02_none_pna
  evaluation_id: ev_m02_none_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '1.50'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m02_delim_ipi
  evaluation_id: ev_m02_delim_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '9.25'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m02_delim_pna
  evaluation_id: ev_m02_delim_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '1.75'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m02_instr_ipi
  evaluation_id: ev_m02_instr_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '8.75'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m02_instr_pna
  evaluation_id: ev_m02_instr_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '5.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m02_sand_ipi
  evaluation_id: ev_m02_sand_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '13.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m02_sand_pna
  evaluation_id: ev_m02_sand_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '8.50'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m03_none_ipi
  evaluation_id: ev_m03_none_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '45.50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m03_none_pna
  evaluation_id: ev_m03_none_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '66.50'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m03_delim_ipi
  evaluation_id: ev_m03_delim_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '34.50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m03_delim_pna
  evaluation_id: ev_m03_delim_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '58.50'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m03_instr_ipi
  evaluation_id: ev_m03_instr_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '41.50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m03_instr_pna
  evaluation_id: ev_m03_instr_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '70.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m03_sand_ipi
  evaluation_id: ev_m03_sand_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '39.75'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m03_sand_pna
  evaluation_id: ev_m03_sand_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '68.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m04_none_ipi
  evaluation_id: ev_m04_none_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '5.50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m04_none_pna
  evaluation_id: ev_m04_none_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '0.75'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m04_delim_ipi
  evaluation_id: ev_m04_delim_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '9.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m04_delim_pna
  evaluation_id: ev_m04_delim_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '0.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m04_instr_ipi
  evaluation_id: ev_m04_instr_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '9.50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m04_instr_pna
  evaluation_id: ev_m04_instr_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '0.75'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m04_sand_ipi
  evaluation_id: ev_m04_sand_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '9.50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m04_sand_pna
  evaluation_id: ev_m04_sand_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '0.25'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m05_none_ipi
  evaluation_id: ev_m05_none_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '14.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m05_none_pna
  evaluation_id: ev_m05_none_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '21.25'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m05_delim_ipi
  evaluation_id: ev_m05_delim_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '11.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m05_delim_pna
  evaluation_id: ev_m05_delim_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '9.50'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m05_instr_ipi
  evaluation_id: ev_m05_instr_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '10.75'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m05_instr_pna
  evaluation_id: ev_m05_instr_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '14.50'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m05_sand_ipi
  evaluation_id: ev_m05_sand_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '12.75'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m05_sand_pna
  evaluation_id: ev_m05_sand_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '16.50'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m06_none_ipi
  evaluation_id: ev_m06_none_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '5.75'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m06_none_pna
  evaluation_id: ev_m06_none_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '0.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m06_delim_ipi
  evaluation_id: ev_m06_delim_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '8.50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m06_delim_pna
  evaluation_id: ev_m06_delim_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '0.25'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m06_instr_ipi
  evaluation_id: ev_m06_instr_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '7.75'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m06_instr_pna
  evaluation_id: ev_m06_instr_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '0.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m06_sand_ipi
  evaluation_id: ev_m06_sand_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '10.25'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m06_sand_pna
  evaluation_id: ev_m06_sand_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '0.25'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m07_none_ipi
  evaluation_id: ev_m07_none_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '9.25'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m07_none_pna
  evaluation_id: ev_m07_none_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '9.75'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m07_delim_ipi
  evaluation_id: ev_m07_delim_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '11.25'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m07_delim_pna
  evaluation_id: ev_m07_delim_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '1.75'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m07_instr_ipi
  evaluation_id: ev_m07_instr_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '9.50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m07_instr_pna
  evaluation_id: ev_m07_instr_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '5.75'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m07_sand_ipi
  evaluation_id: ev_m07_sand_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '11.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m07_sand_pna
  evaluation_id: ev_m07_sand_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '5.25'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m08_none_ipi
  evaluation_id: ev_m08_none_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '23.75'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m08_none_pna
  evaluation_id: ev_m08_none_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '4.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m08_delim_ipi
  evaluation_id: ev_m08_delim_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '17.50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m08_delim_pna
  evaluation_id: ev_m08_delim_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '0.25'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m08_instr_ipi
  evaluation_id: ev_m08_instr_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '26.50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m08_instr_pna
  evaluation_id: ev_m08_instr_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '0.50'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m08_sand_ipi
  evaluation_id: ev_m08_sand_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '21.75'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m08_sand_pna
  evaluation_id: ev_m08_sand_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '0.50'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m09_none_ipi
  evaluation_id: ev_m09_none_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '56.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m09_none_pna
  evaluation_id: ev_m09_none_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '100.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m09_delim_ipi
  evaluation_id: ev_m09_delim_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '59.75'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m09_delim_pna
  evaluation_id: ev_m09_delim_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '90.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m09_instr_ipi
  evaluation_id: ev_m09_instr_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '56.25'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m09_instr_pna
  evaluation_id: ev_m09_instr_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '90.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m09_sand_ipi
  evaluation_id: ev_m09_sand_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '56.50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m09_sand_pna
  evaluation_id: ev_m09_sand_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '100.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m10_none_ipi
  evaluation_id: ev_m10_none_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '59.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m10_none_pna
  evaluation_id: ev_m10_none_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '8.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m10_delim_ipi
  evaluation_id: ev_m10_delim_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '23.75'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m10_delim_pna
  evaluation_id: ev_m10_delim_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '0.75'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m10_instr_ipi
  evaluation_id: ev_m10_instr_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '44.25'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m10_instr_pna
  evaluation_id: ev_m10_instr_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '13.50'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m10_sand_ipi
  evaluation_id: ev_m10_sand_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '58.50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m10_sand_pna
  evaluation_id: ev_m10_sand_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '10.25'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m11_none_ipi
  evaluation_id: ev_m11_none_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '62.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m11_none_pna
  evaluation_id: ev_m11_none_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '79.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m11_delim_ipi
  evaluation_id: ev_m11_delim_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '66.75'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m11_delim_pna
  evaluation_id: ev_m11_delim_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '70.25'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m11_instr_ipi
  evaluation_id: ev_m11_instr_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '61.75'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m11_instr_pna
  evaluation_id: ev_m11_instr_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '72.75'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m11_sand_ipi
  evaluation_id: ev_m11_sand_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '64.75'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m11_sand_pna
  evaluation_id: ev_m11_sand_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '79.50'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m12_none_ipi
  evaluation_id: ev_m12_none_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '41.50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m12_none_pna
  evaluation_id: ev_m12_none_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '50.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m12_delim_ipi
  evaluation_id: ev_m12_delim_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '49.50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m12_delim_pna
  evaluation_id: ev_m12_delim_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '37.00'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m12_instr_ipi
  evaluation_id: ev_m12_instr_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '36.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m12_instr_pna
  evaluation_id: ev_m12_instr_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '42.50'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
- id: r_m12_sand_ipi
  evaluation_id: ev_m12_sand_ipi
  quantity: R1
  coverage: direct
  metric_as_printed: IPI ASR
  value: '42.50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 10
  location: Table 8
- id: r_m12_sand_pna
  evaluation_id: ev_m12_sand_pna
  quantity: R2clean
  coverage: direct
  metric_as_printed: PNA
  value: '37.75'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 33
  location: Table 19
comparisons:
- id: c_m00_delim_ipi
  left: ev_m00_delim_ipi
  right: ev_m00_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m00_delim_pna
  left: ev_m00_delim_pna
  right: ev_m00_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_m00_instr_ipi
  left: ev_m00_instr_ipi
  right: ev_m00_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m00_instr_pna
  left: ev_m00_instr_pna
  right: ev_m00_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_m00_sand_ipi
  left: ev_m00_sand_ipi
  right: ev_m00_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m00_sand_pna
  left: ev_m00_sand_pna
  right: ev_m00_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_m01_delim_ipi
  left: ev_m01_delim_ipi
  right: ev_m01_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m01_delim_pna
  left: ev_m01_delim_pna
  right: ev_m01_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_m01_instr_ipi
  left: ev_m01_instr_ipi
  right: ev_m01_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m01_instr_pna
  left: ev_m01_instr_pna
  right: ev_m01_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_m01_sand_ipi
  left: ev_m01_sand_ipi
  right: ev_m01_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m01_sand_pna
  left: ev_m01_sand_pna
  right: ev_m01_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_m02_delim_ipi
  left: ev_m02_delim_ipi
  right: ev_m02_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m02_delim_pna
  left: ev_m02_delim_pna
  right: ev_m02_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_m02_instr_ipi
  left: ev_m02_instr_ipi
  right: ev_m02_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m02_instr_pna
  left: ev_m02_instr_pna
  right: ev_m02_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_m02_sand_ipi
  left: ev_m02_sand_ipi
  right: ev_m02_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m02_sand_pna
  left: ev_m02_sand_pna
  right: ev_m02_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_m03_delim_ipi
  left: ev_m03_delim_ipi
  right: ev_m03_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m03_delim_pna
  left: ev_m03_delim_pna
  right: ev_m03_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_m03_instr_ipi
  left: ev_m03_instr_ipi
  right: ev_m03_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m03_instr_pna
  left: ev_m03_instr_pna
  right: ev_m03_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_m03_sand_ipi
  left: ev_m03_sand_ipi
  right: ev_m03_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m03_sand_pna
  left: ev_m03_sand_pna
  right: ev_m03_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_m04_delim_ipi
  left: ev_m04_delim_ipi
  right: ev_m04_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m04_delim_pna
  left: ev_m04_delim_pna
  right: ev_m04_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_m04_instr_ipi
  left: ev_m04_instr_ipi
  right: ev_m04_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m04_instr_pna
  left: ev_m04_instr_pna
  right: ev_m04_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_m04_sand_ipi
  left: ev_m04_sand_ipi
  right: ev_m04_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m04_sand_pna
  left: ev_m04_sand_pna
  right: ev_m04_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_m05_delim_ipi
  left: ev_m05_delim_ipi
  right: ev_m05_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m05_delim_pna
  left: ev_m05_delim_pna
  right: ev_m05_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_m05_instr_ipi
  left: ev_m05_instr_ipi
  right: ev_m05_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m05_instr_pna
  left: ev_m05_instr_pna
  right: ev_m05_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_m05_sand_ipi
  left: ev_m05_sand_ipi
  right: ev_m05_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m05_sand_pna
  left: ev_m05_sand_pna
  right: ev_m05_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_m06_delim_ipi
  left: ev_m06_delim_ipi
  right: ev_m06_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m06_delim_pna
  left: ev_m06_delim_pna
  right: ev_m06_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_m06_instr_ipi
  left: ev_m06_instr_ipi
  right: ev_m06_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m06_instr_pna
  left: ev_m06_instr_pna
  right: ev_m06_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_m06_sand_ipi
  left: ev_m06_sand_ipi
  right: ev_m06_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m06_sand_pna
  left: ev_m06_sand_pna
  right: ev_m06_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_m07_delim_ipi
  left: ev_m07_delim_ipi
  right: ev_m07_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m07_delim_pna
  left: ev_m07_delim_pna
  right: ev_m07_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_m07_instr_ipi
  left: ev_m07_instr_ipi
  right: ev_m07_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m07_instr_pna
  left: ev_m07_instr_pna
  right: ev_m07_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_m07_sand_ipi
  left: ev_m07_sand_ipi
  right: ev_m07_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m07_sand_pna
  left: ev_m07_sand_pna
  right: ev_m07_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_m08_delim_ipi
  left: ev_m08_delim_ipi
  right: ev_m08_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m08_delim_pna
  left: ev_m08_delim_pna
  right: ev_m08_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_m08_instr_ipi
  left: ev_m08_instr_ipi
  right: ev_m08_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m08_instr_pna
  left: ev_m08_instr_pna
  right: ev_m08_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_m08_sand_ipi
  left: ev_m08_sand_ipi
  right: ev_m08_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m08_sand_pna
  left: ev_m08_sand_pna
  right: ev_m08_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_m09_delim_ipi
  left: ev_m09_delim_ipi
  right: ev_m09_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m09_delim_pna
  left: ev_m09_delim_pna
  right: ev_m09_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_m09_instr_ipi
  left: ev_m09_instr_ipi
  right: ev_m09_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m09_instr_pna
  left: ev_m09_instr_pna
  right: ev_m09_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_m09_sand_ipi
  left: ev_m09_sand_ipi
  right: ev_m09_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m09_sand_pna
  left: ev_m09_sand_pna
  right: ev_m09_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_m10_delim_ipi
  left: ev_m10_delim_ipi
  right: ev_m10_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m10_delim_pna
  left: ev_m10_delim_pna
  right: ev_m10_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_m10_instr_ipi
  left: ev_m10_instr_ipi
  right: ev_m10_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m10_instr_pna
  left: ev_m10_instr_pna
  right: ev_m10_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_m10_sand_ipi
  left: ev_m10_sand_ipi
  right: ev_m10_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m10_sand_pna
  left: ev_m10_sand_pna
  right: ev_m10_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_m11_delim_ipi
  left: ev_m11_delim_ipi
  right: ev_m11_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m11_delim_pna
  left: ev_m11_delim_pna
  right: ev_m11_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_m11_instr_ipi
  left: ev_m11_instr_ipi
  right: ev_m11_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m11_instr_pna
  left: ev_m11_instr_pna
  right: ev_m11_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_m11_sand_ipi
  left: ev_m11_sand_ipi
  right: ev_m11_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m11_sand_pna
  left: ev_m11_sand_pna
  right: ev_m11_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_m12_delim_ipi
  left: ev_m12_delim_ipi
  right: ev_m12_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m12_delim_pna
  left: ev_m12_delim_pna
  right: ev_m12_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_m12_instr_ipi
  left: ev_m12_instr_ipi
  right: ev_m12_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m12_instr_pna
  left: ev_m12_instr_pna
  right: ev_m12_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
- id: c_m12_sand_ipi
  left: ev_m12_sand_ipi
  right: ev_m12_none_ipi
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 10
- id: c_m12_sand_pna
  left: ev_m12_sand_pna
  right: ev_m12_none_pna
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 33
claims:
- id: k1
  text: current defences show limited effectiveness
  location: abstract
  anchor: but limited effectiveness shown in current defenses
  jobs:
  - R1
  supported_by:
  - c_m00_delim_ipi
  - c_m00_instr_ipi
  - c_m00_sand_ipi
  - c_m01_delim_ipi
  - c_m01_instr_ipi
  - c_m01_sand_ipi
  - c_m02_delim_ipi
  - c_m02_instr_ipi
  - c_m02_sand_ipi
  - c_m03_delim_ipi
  - c_m03_instr_ipi
  - c_m03_sand_ipi
  - c_m04_delim_ipi
  - c_m04_instr_ipi
  - c_m04_sand_ipi
  - c_m05_delim_ipi
  - c_m05_instr_ipi
  - c_m05_sand_ipi
  - c_m06_delim_ipi
  - c_m06_instr_ipi
  - c_m06_sand_ipi
  - c_m07_delim_ipi
  - c_m07_instr_ipi
  - c_m07_sand_ipi
  - c_m08_delim_ipi
  - c_m08_instr_ipi
  - c_m08_sand_ipi
  - c_m09_delim_ipi
  - c_m09_instr_ipi
  - c_m09_sand_ipi
  - c_m10_delim_ipi
  - c_m10_instr_ipi
  - c_m10_sand_ipi
  - c_m11_delim_ipi
  - c_m11_instr_ipi
  - c_m11_sand_ipi
  - c_m12_delim_ipi
  - c_m12_instr_ipi
  - c_m12_sand_ipi
  page: 1
coverage:
  R1: direct
  R2clean: direct
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 8 (IPI ASR, averaged over five attack types)
    R2clean: Table 19 (PNA, no attack)

`attackdefence_arxiv2024` (2411.00459)

In [ ]:
%%writefile reports/records/attackdefence_arxiv2024.yaml
report:
  id: attackdefence_arxiv2024
  slug: '2411.00459'
  title: Defense Against Prompt Injection Attack by Leveraging Attack Techniques
  first_author: Yulin Chen
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2411.00459'
  version_read: arXiv v6
  first_date: '2024-11-01'
  read: F
  role: proposes
  code_release: stated in a footnote; not checked
  later_version: ACL 2025, DOI 10.18653/v1/2025.acl-long.897 (from the search record; not read)
  extraction_scope: 'Sections 4 and 5.1 to 5.3: Table 2 (indirect scenario) rows None, Sandwich, Ours-Ignore and Ours-Fakecom-t for the three models and five attacks, and the same rows of Table 3 (QA utility); the abstract
    claim (direct scenario of Table 1, other baselines and variants, Tables 4 to 7 not coded)'
defences:
- id: ign_l3
  name: Ignore defence (ignore-attack wording turned into a shield prompt)
  origin: this_report
  base_checkpoint: Llama3-8b-Instruct
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  components:
  - after the data, a shield prompt telling the model to disregard all previous instructions
  - the original instruction repeated after it
  layout: system prompt; user turn with <Instruction> and <Data> sections; the shield prompt and the repeated instruction after the data
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: The ignore defense is inspired by the ignore attack
- id: fct_l3
  name: Fake completion defence with template
  origin: this_report
  base_checkpoint: Llama3-8b-Instruct
  axis1:
  - chat_roles
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  components:
  - after the data, a fabricated assistant turn reporting an injection and distrusting earlier instructions
  - a fabricated user turn confirming the original instruction
  layout: system prompt; user turn with <Instruction> and <Data> sections; the shield prompt and the repeated instruction after the data
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: Fake Completion Defense with Template
- id: sand_l3
  name: Sandwich (baseline)
  origin: other_report
  base_checkpoint: Llama3-8b-Instruct
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 6
  anchor: we select Sandwich (san, 2023), Instructional
- id: ign_q2
  name: Ignore defence (ignore-attack wording turned into a shield prompt)
  origin: this_report
  base_checkpoint: Qwen2-7b-Instruct
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  components:
  - after the data, a shield prompt telling the model to disregard all previous instructions
  - the original instruction repeated after it
  layout: system prompt; user turn with <Instruction> and <Data> sections; the shield prompt and the repeated instruction after the data
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: The ignore defense is inspired by the ignore attack
- id: fct_q2
  name: Fake completion defence with template
  origin: this_report
  base_checkpoint: Qwen2-7b-Instruct
  axis1:
  - chat_roles
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  components:
  - after the data, a fabricated assistant turn reporting an injection and distrusting earlier instructions
  - a fabricated user turn confirming the original instruction
  layout: system prompt; user turn with <Instruction> and <Data> sections; the shield prompt and the repeated instruction after the data
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: Fake Completion Defense with Template
- id: sand_q2
  name: Sandwich (baseline)
  origin: other_report
  base_checkpoint: Qwen2-7b-Instruct
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 6
  anchor: we select Sandwich (san, 2023), Instructional
- id: ign_l31
  name: Ignore defence (ignore-attack wording turned into a shield prompt)
  origin: this_report
  base_checkpoint: Llama3.1-8b-Instruct
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  components:
  - after the data, a shield prompt telling the model to disregard all previous instructions
  - the original instruction repeated after it
  layout: system prompt; user turn with <Instruction> and <Data> sections; the shield prompt and the repeated instruction after the data
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: The ignore defense is inspired by the ignore attack
- id: fct_l31
  name: Fake completion defence with template
  origin: this_report
  base_checkpoint: Llama3.1-8b-Instruct
  axis1:
  - chat_roles
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  components:
  - after the data, a fabricated assistant turn reporting an injection and distrusting earlier instructions
  - a fabricated user turn confirming the original instruction
  layout: system prompt; user turn with <Instruction> and <Data> sections; the shield prompt and the repeated instruction after the data
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: Fake Completion Defense with Template
- id: sand_l31
  name: Sandwich (baseline)
  origin: other_report
  base_checkpoint: Llama3.1-8b-Instruct
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 6
  anchor: we select Sandwich (san, 2023), Instructional
evaluations:
- id: ev_l3_none_naive
  defence_id: none:Llama3-8b-Instruct
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Naive attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
  baseline_kind: off_the_shelf
- id: ev_l3_none_ignore
  defence_id: none:Llama3-8b-Instruct
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Ignore attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
  baseline_kind: off_the_shelf
- id: ev_l3_none_escape
  defence_id: none:Llama3-8b-Instruct
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Escape attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
  baseline_kind: off_the_shelf
- id: ev_l3_none_fakecom
  defence_id: none:Llama3-8b-Instruct
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Fakecom attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
  baseline_kind: off_the_shelf
- id: ev_l3_none_combined
  defence_id: none:Llama3-8b-Instruct
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Combined attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
  baseline_kind: off_the_shelf
- id: ev_q2_none_naive
  defence_id: none:Qwen2-7b-Instruct
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Naive attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
  baseline_kind: off_the_shelf
- id: ev_q2_none_ignore
  defence_id: none:Qwen2-7b-Instruct
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Ignore attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
  baseline_kind: off_the_shelf
- id: ev_q2_none_escape
  defence_id: none:Qwen2-7b-Instruct
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Escape attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
  baseline_kind: off_the_shelf
- id: ev_q2_none_fakecom
  defence_id: none:Qwen2-7b-Instruct
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Fakecom attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
  baseline_kind: off_the_shelf
- id: ev_q2_none_combined
  defence_id: none:Qwen2-7b-Instruct
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Combined attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
  baseline_kind: off_the_shelf
- id: ev_l31_none_naive
  defence_id: none:Llama3.1-8b-Instruct
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Naive attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
  baseline_kind: off_the_shelf
- id: ev_l31_none_ignore
  defence_id: none:Llama3.1-8b-Instruct
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Ignore attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
  baseline_kind: off_the_shelf
- id: ev_l31_none_escape
  defence_id: none:Llama3.1-8b-Instruct
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Escape attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
  baseline_kind: off_the_shelf
- id: ev_l31_none_fakecom
  defence_id: none:Llama3.1-8b-Instruct
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Fakecom attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
  baseline_kind: off_the_shelf
- id: ev_l31_none_combined
  defence_id: none:Llama3.1-8b-Instruct
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Combined attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
  baseline_kind: off_the_shelf
- id: ev_l3_sand_naive
  defence_id: sand_l3
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Naive attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_l3_sand_ignore
  defence_id: sand_l3
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Ignore attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_l3_sand_escape
  defence_id: sand_l3
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Escape attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_l3_sand_fakecom
  defence_id: sand_l3
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Fakecom attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_l3_sand_combined
  defence_id: sand_l3
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Combined attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_q2_sand_naive
  defence_id: sand_q2
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Naive attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_q2_sand_ignore
  defence_id: sand_q2
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Ignore attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_q2_sand_escape
  defence_id: sand_q2
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Escape attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_q2_sand_fakecom
  defence_id: sand_q2
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Fakecom attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_q2_sand_combined
  defence_id: sand_q2
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Combined attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_l31_sand_naive
  defence_id: sand_l31
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Naive attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_l31_sand_ignore
  defence_id: sand_l31
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Ignore attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_l31_sand_escape
  defence_id: sand_l31
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Escape attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_l31_sand_fakecom
  defence_id: sand_l31
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Fakecom attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_l31_sand_combined
  defence_id: sand_l31
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Combined attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_l3_ign_naive
  defence_id: ign_l3
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Naive attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_l3_ign_ignore
  defence_id: ign_l3
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Ignore attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_l3_ign_escape
  defence_id: ign_l3
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Escape attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_l3_ign_fakecom
  defence_id: ign_l3
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Fakecom attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_l3_ign_combined
  defence_id: ign_l3
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Combined attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_q2_ign_naive
  defence_id: ign_q2
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Naive attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_q2_ign_ignore
  defence_id: ign_q2
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Ignore attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_q2_ign_escape
  defence_id: ign_q2
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Escape attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_q2_ign_fakecom
  defence_id: ign_q2
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Fakecom attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_q2_ign_combined
  defence_id: ign_q2
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Combined attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_l31_ign_naive
  defence_id: ign_l31
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Naive attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_l31_ign_ignore
  defence_id: ign_l31
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Ignore attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_l31_ign_escape
  defence_id: ign_l31
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Escape attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_l31_ign_fakecom
  defence_id: ign_l31
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Fakecom attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_l31_ign_combined
  defence_id: ign_l31
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Combined attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_l3_fct_naive
  defence_id: fct_l3
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Naive attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_l3_fct_ignore
  defence_id: fct_l3
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Ignore attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_l3_fct_escape
  defence_id: fct_l3
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Escape attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_l3_fct_fakecom
  defence_id: fct_l3
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Fakecom attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_l3_fct_combined
  defence_id: fct_l3
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Combined attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_q2_fct_naive
  defence_id: fct_q2
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Naive attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_q2_fct_ignore
  defence_id: fct_q2
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Ignore attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_q2_fct_escape
  defence_id: fct_q2
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Escape attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_q2_fct_fakecom
  defence_id: fct_q2
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Fakecom attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_q2_fct_combined
  defence_id: fct_q2
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Combined attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_l31_fct_naive
  defence_id: fct_l31
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Naive attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_l31_fct_ignore
  defence_id: fct_l31
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Ignore attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_l31_fct_escape
  defence_id: fct_l31
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Escape attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_l31_fct_fakecom
  defence_id: fct_l31
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Fakecom attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_l31_fct_combined
  defence_id: fct_l31
  benchmark: QA dataset filtered by Li et al. (2023b), instructions injected into retrieved content (2000 samples)
  attacker_type: static
  attacker_access: black_box
  judge: answer to the injected instruction appears in the response
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 6
  attacker_budget: Combined attack
  sample_size: '2000'
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_l3_none_util
  defence_id: none:Llama3-8b-Instruct
  benchmark: filtered QA dataset without attacks
  attacker_type: none
  attacker_access: not_applicable
  judge: golden answer appears in the response (accuracy)
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 7
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
  baseline_kind: off_the_shelf
- id: ev_l31_none_util
  defence_id: none:Llama3.1-8b-Instruct
  benchmark: filtered QA dataset without attacks
  attacker_type: none
  attacker_access: not_applicable
  judge: golden answer appears in the response (accuracy)
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 7
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
  baseline_kind: off_the_shelf
- id: ev_q2_none_util
  defence_id: none:Qwen2-7b-Instruct
  benchmark: filtered QA dataset without attacks
  attacker_type: none
  attacker_access: not_applicable
  judge: golden answer appears in the response (accuracy)
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 7
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
  baseline_kind: off_the_shelf
- id: ev_l3_sand_util
  defence_id: sand_l3
  benchmark: filtered QA dataset without attacks
  attacker_type: none
  attacker_access: not_applicable
  judge: golden answer appears in the response (accuracy)
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 7
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_l31_sand_util
  defence_id: sand_l31
  benchmark: filtered QA dataset without attacks
  attacker_type: none
  attacker_access: not_applicable
  judge: golden answer appears in the response (accuracy)
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 7
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_q2_sand_util
  defence_id: sand_q2
  benchmark: filtered QA dataset without attacks
  attacker_type: none
  attacker_access: not_applicable
  judge: golden answer appears in the response (accuracy)
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 7
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_l3_ign_util
  defence_id: ign_l3
  benchmark: filtered QA dataset without attacks
  attacker_type: none
  attacker_access: not_applicable
  judge: golden answer appears in the response (accuracy)
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 7
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_l31_ign_util
  defence_id: ign_l31
  benchmark: filtered QA dataset without attacks
  attacker_type: none
  attacker_access: not_applicable
  judge: golden answer appears in the response (accuracy)
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 7
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_q2_ign_util
  defence_id: ign_q2
  benchmark: filtered QA dataset without attacks
  attacker_type: none
  attacker_access: not_applicable
  judge: golden answer appears in the response (accuracy)
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 7
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_l3_fct_util
  defence_id: fct_l3
  benchmark: filtered QA dataset without attacks
  attacker_type: none
  attacker_access: not_applicable
  judge: golden answer appears in the response (accuracy)
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 7
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_l31_fct_util
  defence_id: fct_l31
  benchmark: filtered QA dataset without attacks
  attacker_type: none
  attacker_access: not_applicable
  judge: golden answer appears in the response (accuracy)
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 7
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
- id: ev_q2_fct_util
  defence_id: fct_q2
  benchmark: filtered QA dataset without attacks
  attacker_type: none
  attacker_access: not_applicable
  judge: golden answer appears in the response (accuracy)
  layout_same_as_defence: 'yes'
  measured_by: attackdefence_arxiv2024
  page: 7
  decoding: do_sample false, max_new_tokens 256
  pages:
    benchmark: 5
    judge: 5
    decoding: 12
results:
- id: r_l3_none_naive
  evaluation_id: ev_l3_none_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Naive
  value: '10.55'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l3_none_ignore
  evaluation_id: ev_l3_none_ignore
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Ignore
  value: '53.35'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l3_none_escape
  evaluation_id: ev_l3_none_escape
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape
  value: '88.25'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l3_none_fakecom
  evaluation_id: ev_l3_none_fakecom
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Fakecom
  value: '75.30'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l3_none_combined
  evaluation_id: ev_l3_none_combined
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Combined
  value: '86.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q2_none_naive
  evaluation_id: ev_q2_none_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Naive
  value: '92.45'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q2_none_ignore
  evaluation_id: ev_q2_none_ignore
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Ignore
  value: '95.90'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q2_none_escape
  evaluation_id: ev_q2_none_escape
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape
  value: '100.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q2_none_fakecom
  evaluation_id: ev_q2_none_fakecom
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Fakecom
  value: '100.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q2_none_combined
  evaluation_id: ev_q2_none_combined
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Combined
  value: '100.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_none_naive
  evaluation_id: ev_l31_none_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Naive
  value: '85.90'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_none_ignore
  evaluation_id: ev_l31_none_ignore
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Ignore
  value: '91.10'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_none_escape
  evaluation_id: ev_l31_none_escape
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape
  value: '81.70'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_none_fakecom
  evaluation_id: ev_l31_none_fakecom
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Fakecom
  value: '95.25'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_none_combined
  evaluation_id: ev_l31_none_combined
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Combined
  value: '92.30'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l3_sand_naive
  evaluation_id: ev_l3_sand_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Naive
  value: '0.45'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l3_sand_ignore
  evaluation_id: ev_l3_sand_ignore
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Ignore
  value: '9.35'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l3_sand_escape
  evaluation_id: ev_l3_sand_escape
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape
  value: '49.55'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l3_sand_fakecom
  evaluation_id: ev_l3_sand_fakecom
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Fakecom
  value: '7.30'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l3_sand_combined
  evaluation_id: ev_l3_sand_combined
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Combined
  value: '21.25'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q2_sand_naive
  evaluation_id: ev_q2_sand_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Naive
  value: '4.80'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q2_sand_ignore
  evaluation_id: ev_q2_sand_ignore
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Ignore
  value: '6.15'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q2_sand_escape
  evaluation_id: ev_q2_sand_escape
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape
  value: '14.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q2_sand_fakecom
  evaluation_id: ev_q2_sand_fakecom
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Fakecom
  value: '34.20'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q2_sand_combined
  evaluation_id: ev_q2_sand_combined
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Combined
  value: '34.60'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_sand_naive
  evaluation_id: ev_l31_sand_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Naive
  value: '2.50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_sand_ignore
  evaluation_id: ev_l31_sand_ignore
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Ignore
  value: '3.05'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_sand_escape
  evaluation_id: ev_l31_sand_escape
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape
  value: '22.90'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_sand_fakecom
  evaluation_id: ev_l31_sand_fakecom
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Fakecom
  value: '3.35'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_sand_combined
  evaluation_id: ev_l31_sand_combined
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Combined
  value: '9.55'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l3_ign_naive
  evaluation_id: ev_l3_ign_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Naive
  value: '0.05'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l3_ign_ignore
  evaluation_id: ev_l3_ign_ignore
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Ignore
  value: '0.35'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l3_ign_escape
  evaluation_id: ev_l3_ign_escape
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape
  value: '0.30'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l3_ign_fakecom
  evaluation_id: ev_l3_ign_fakecom
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Fakecom
  value: '0.10'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l3_ign_combined
  evaluation_id: ev_l3_ign_combined
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Combined
  value: '1.35'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q2_ign_naive
  evaluation_id: ev_q2_ign_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Naive
  value: '0.85'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q2_ign_ignore
  evaluation_id: ev_q2_ign_ignore
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Ignore
  value: '0.70'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q2_ign_escape
  evaluation_id: ev_q2_ign_escape
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape
  value: '0.80'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q2_ign_fakecom
  evaluation_id: ev_q2_ign_fakecom
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Fakecom
  value: '0.95'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q2_ign_combined
  evaluation_id: ev_q2_ign_combined
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Combined
  value: '4.10'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_ign_naive
  evaluation_id: ev_l31_ign_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Naive
  value: '0.25'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_ign_ignore
  evaluation_id: ev_l31_ign_ignore
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Ignore
  value: '0.30'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_ign_escape
  evaluation_id: ev_l31_ign_escape
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape
  value: '0.35'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_ign_fakecom
  evaluation_id: ev_l31_ign_fakecom
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Fakecom
  value: '0.45'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_ign_combined
  evaluation_id: ev_l31_ign_combined
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Combined
  value: '1.10'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l3_fct_naive
  evaluation_id: ev_l3_fct_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Naive
  value: '0.05'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l3_fct_ignore
  evaluation_id: ev_l3_fct_ignore
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Ignore
  value: '0.05'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l3_fct_escape
  evaluation_id: ev_l3_fct_escape
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape
  value: '0.30'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l3_fct_fakecom
  evaluation_id: ev_l3_fct_fakecom
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Fakecom
  value: '0.05'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l3_fct_combined
  evaluation_id: ev_l3_fct_combined
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Combined
  value: '0.05'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q2_fct_naive
  evaluation_id: ev_q2_fct_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Naive
  value: '0.25'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q2_fct_ignore
  evaluation_id: ev_q2_fct_ignore
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Ignore
  value: '0.20'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q2_fct_escape
  evaluation_id: ev_q2_fct_escape
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape
  value: '0.15'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q2_fct_fakecom
  evaluation_id: ev_q2_fct_fakecom
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Fakecom
  value: '0.05'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_q2_fct_combined
  evaluation_id: ev_q2_fct_combined
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Combined
  value: '0.05'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_fct_naive
  evaluation_id: ev_l31_fct_naive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Naive
  value: '0.05'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_fct_ignore
  evaluation_id: ev_l31_fct_ignore
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Ignore
  value: '0.10'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_fct_escape
  evaluation_id: ev_l31_fct_escape
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape
  value: '0.10'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_fct_fakecom
  evaluation_id: ev_l31_fct_fakecom
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Fakecom
  value: '0.05'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l31_fct_combined
  evaluation_id: ev_l31_fct_combined
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Combined
  value: '0.10'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_l3_none_util
  evaluation_id: ev_l3_none_util
  quantity: R2clean
  coverage: direct
  metric_as_printed: QA accuracy
  value: '78.05'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l31_none_util
  evaluation_id: ev_l31_none_util
  quantity: R2clean
  coverage: direct
  metric_as_printed: QA accuracy
  value: '77.10'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q2_none_util
  evaluation_id: ev_q2_none_util
  quantity: R2clean
  coverage: direct
  metric_as_printed: QA accuracy
  value: '76.60'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l3_sand_util
  evaluation_id: ev_l3_sand_util
  quantity: R2clean
  coverage: direct
  metric_as_printed: QA accuracy
  value: '80.80'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l31_sand_util
  evaluation_id: ev_l31_sand_util
  quantity: R2clean
  coverage: direct
  metric_as_printed: QA accuracy
  value: '79.50'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q2_sand_util
  evaluation_id: ev_q2_sand_util
  quantity: R2clean
  coverage: direct
  metric_as_printed: QA accuracy
  value: '77.35'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l3_ign_util
  evaluation_id: ev_l3_ign_util
  quantity: R2clean
  coverage: direct
  metric_as_printed: QA accuracy
  value: '78.55'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l31_ign_util
  evaluation_id: ev_l31_ign_util
  quantity: R2clean
  coverage: direct
  metric_as_printed: QA accuracy
  value: '79.60'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q2_ign_util
  evaluation_id: ev_q2_ign_util
  quantity: R2clean
  coverage: direct
  metric_as_printed: QA accuracy
  value: '77.60'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l3_fct_util
  evaluation_id: ev_l3_fct_util
  quantity: R2clean
  coverage: direct
  metric_as_printed: QA accuracy
  value: '79.40'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_l31_fct_util
  evaluation_id: ev_l31_fct_util
  quantity: R2clean
  coverage: direct
  metric_as_printed: QA accuracy
  value: '80.45'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_q2_fct_util
  evaluation_id: ev_q2_fct_util
  quantity: R2clean
  coverage: direct
  metric_as_printed: QA accuracy
  value: '77.40'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
comparisons:
- id: c_l3_ign_naive_none
  left: ev_l3_ign_naive
  right: ev_l3_none_naive
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l3_ign_naive_sand
  left: ev_l3_ign_naive
  right: ev_l3_sand_naive
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: the attack-derived shield prompt against the sandwich prompt on the same base'
  page: 6
- id: c_l3_ign_ignore_none
  left: ev_l3_ign_ignore
  right: ev_l3_none_ignore
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l3_ign_ignore_sand
  left: ev_l3_ign_ignore
  right: ev_l3_sand_ignore
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: the attack-derived shield prompt against the sandwich prompt on the same base'
  page: 6
- id: c_l3_ign_escape_none
  left: ev_l3_ign_escape
  right: ev_l3_none_escape
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l3_ign_escape_sand
  left: ev_l3_ign_escape
  right: ev_l3_sand_escape
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: the attack-derived shield prompt against the sandwich prompt on the same base'
  page: 6
- id: c_l3_ign_fakecom_none
  left: ev_l3_ign_fakecom
  right: ev_l3_none_fakecom
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l3_ign_fakecom_sand
  left: ev_l3_ign_fakecom
  right: ev_l3_sand_fakecom
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: the attack-derived shield prompt against the sandwich prompt on the same base'
  page: 6
- id: c_l3_ign_combined_none
  left: ev_l3_ign_combined
  right: ev_l3_none_combined
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l3_ign_combined_sand
  left: ev_l3_ign_combined
  right: ev_l3_sand_combined
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: the attack-derived shield prompt against the sandwich prompt on the same base'
  page: 6
- id: c_l3_ign_util_none
  left: ev_l3_ign_util
  right: ev_l3_none_util
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l3_ign_util_sand
  left: ev_l3_ign_util
  right: ev_l3_sand_util
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: the attack-derived shield prompt against the sandwich prompt on the same base'
  page: 7
- id: c_l3_fct_naive_none
  left: ev_l3_fct_naive
  right: ev_l3_none_naive
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l3_fct_naive_sand
  left: ev_l3_fct_naive
  right: ev_l3_sand_naive
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: the attack-derived shield prompt against the sandwich prompt on the same base'
  page: 6
- id: c_l3_fct_ignore_none
  left: ev_l3_fct_ignore
  right: ev_l3_none_ignore
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l3_fct_ignore_sand
  left: ev_l3_fct_ignore
  right: ev_l3_sand_ignore
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: the attack-derived shield prompt against the sandwich prompt on the same base'
  page: 6
- id: c_l3_fct_escape_none
  left: ev_l3_fct_escape
  right: ev_l3_none_escape
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l3_fct_escape_sand
  left: ev_l3_fct_escape
  right: ev_l3_sand_escape
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: the attack-derived shield prompt against the sandwich prompt on the same base'
  page: 6
- id: c_l3_fct_fakecom_none
  left: ev_l3_fct_fakecom
  right: ev_l3_none_fakecom
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l3_fct_fakecom_sand
  left: ev_l3_fct_fakecom
  right: ev_l3_sand_fakecom
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: the attack-derived shield prompt against the sandwich prompt on the same base'
  page: 6
- id: c_l3_fct_combined_none
  left: ev_l3_fct_combined
  right: ev_l3_none_combined
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l3_fct_combined_sand
  left: ev_l3_fct_combined
  right: ev_l3_sand_combined
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: the attack-derived shield prompt against the sandwich prompt on the same base'
  page: 6
- id: c_l3_fct_util_none
  left: ev_l3_fct_util
  right: ev_l3_none_util
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l3_fct_util_sand
  left: ev_l3_fct_util
  right: ev_l3_sand_util
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: the attack-derived shield prompt against the sandwich prompt on the same base'
  page: 7
- id: c_q2_ign_naive_none
  left: ev_q2_ign_naive
  right: ev_q2_none_naive
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q2_ign_naive_sand
  left: ev_q2_ign_naive
  right: ev_q2_sand_naive
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: the attack-derived shield prompt against the sandwich prompt on the same base'
  page: 6
- id: c_q2_ign_ignore_none
  left: ev_q2_ign_ignore
  right: ev_q2_none_ignore
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q2_ign_ignore_sand
  left: ev_q2_ign_ignore
  right: ev_q2_sand_ignore
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: the attack-derived shield prompt against the sandwich prompt on the same base'
  page: 6
- id: c_q2_ign_escape_none
  left: ev_q2_ign_escape
  right: ev_q2_none_escape
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q2_ign_escape_sand
  left: ev_q2_ign_escape
  right: ev_q2_sand_escape
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: the attack-derived shield prompt against the sandwich prompt on the same base'
  page: 6
- id: c_q2_ign_fakecom_none
  left: ev_q2_ign_fakecom
  right: ev_q2_none_fakecom
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q2_ign_fakecom_sand
  left: ev_q2_ign_fakecom
  right: ev_q2_sand_fakecom
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: the attack-derived shield prompt against the sandwich prompt on the same base'
  page: 6
- id: c_q2_ign_combined_none
  left: ev_q2_ign_combined
  right: ev_q2_none_combined
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q2_ign_combined_sand
  left: ev_q2_ign_combined
  right: ev_q2_sand_combined
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: the attack-derived shield prompt against the sandwich prompt on the same base'
  page: 6
- id: c_q2_ign_util_none
  left: ev_q2_ign_util
  right: ev_q2_none_util
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q2_ign_util_sand
  left: ev_q2_ign_util
  right: ev_q2_sand_util
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: the attack-derived shield prompt against the sandwich prompt on the same base'
  page: 7
- id: c_q2_fct_naive_none
  left: ev_q2_fct_naive
  right: ev_q2_none_naive
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q2_fct_naive_sand
  left: ev_q2_fct_naive
  right: ev_q2_sand_naive
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: the attack-derived shield prompt against the sandwich prompt on the same base'
  page: 6
- id: c_q2_fct_ignore_none
  left: ev_q2_fct_ignore
  right: ev_q2_none_ignore
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q2_fct_ignore_sand
  left: ev_q2_fct_ignore
  right: ev_q2_sand_ignore
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: the attack-derived shield prompt against the sandwich prompt on the same base'
  page: 6
- id: c_q2_fct_escape_none
  left: ev_q2_fct_escape
  right: ev_q2_none_escape
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q2_fct_escape_sand
  left: ev_q2_fct_escape
  right: ev_q2_sand_escape
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: the attack-derived shield prompt against the sandwich prompt on the same base'
  page: 6
- id: c_q2_fct_fakecom_none
  left: ev_q2_fct_fakecom
  right: ev_q2_none_fakecom
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q2_fct_fakecom_sand
  left: ev_q2_fct_fakecom
  right: ev_q2_sand_fakecom
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: the attack-derived shield prompt against the sandwich prompt on the same base'
  page: 6
- id: c_q2_fct_combined_none
  left: ev_q2_fct_combined
  right: ev_q2_none_combined
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_q2_fct_combined_sand
  left: ev_q2_fct_combined
  right: ev_q2_sand_combined
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: the attack-derived shield prompt against the sandwich prompt on the same base'
  page: 6
- id: c_q2_fct_util_none
  left: ev_q2_fct_util
  right: ev_q2_none_util
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_q2_fct_util_sand
  left: ev_q2_fct_util
  right: ev_q2_sand_util
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: the attack-derived shield prompt against the sandwich prompt on the same base'
  page: 7
- id: c_l31_ign_naive_none
  left: ev_l31_ign_naive
  right: ev_l31_none_naive
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_ign_naive_sand
  left: ev_l31_ign_naive
  right: ev_l31_sand_naive
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: the attack-derived shield prompt against the sandwich prompt on the same base'
  page: 6
- id: c_l31_ign_ignore_none
  left: ev_l31_ign_ignore
  right: ev_l31_none_ignore
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_ign_ignore_sand
  left: ev_l31_ign_ignore
  right: ev_l31_sand_ignore
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: the attack-derived shield prompt against the sandwich prompt on the same base'
  page: 6
- id: c_l31_ign_escape_none
  left: ev_l31_ign_escape
  right: ev_l31_none_escape
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_ign_escape_sand
  left: ev_l31_ign_escape
  right: ev_l31_sand_escape
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: the attack-derived shield prompt against the sandwich prompt on the same base'
  page: 6
- id: c_l31_ign_fakecom_none
  left: ev_l31_ign_fakecom
  right: ev_l31_none_fakecom
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_ign_fakecom_sand
  left: ev_l31_ign_fakecom
  right: ev_l31_sand_fakecom
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: the attack-derived shield prompt against the sandwich prompt on the same base'
  page: 6
- id: c_l31_ign_combined_none
  left: ev_l31_ign_combined
  right: ev_l31_none_combined
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_ign_combined_sand
  left: ev_l31_ign_combined
  right: ev_l31_sand_combined
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: the attack-derived shield prompt against the sandwich prompt on the same base'
  page: 6
- id: c_l31_ign_util_none
  left: ev_l31_ign_util
  right: ev_l31_none_util
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l31_ign_util_sand
  left: ev_l31_ign_util
  right: ev_l31_sand_util
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: the attack-derived shield prompt against the sandwich prompt on the same base'
  page: 7
- id: c_l31_fct_naive_none
  left: ev_l31_fct_naive
  right: ev_l31_none_naive
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_fct_naive_sand
  left: ev_l31_fct_naive
  right: ev_l31_sand_naive
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: the attack-derived shield prompt against the sandwich prompt on the same base'
  page: 6
- id: c_l31_fct_ignore_none
  left: ev_l31_fct_ignore
  right: ev_l31_none_ignore
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_fct_ignore_sand
  left: ev_l31_fct_ignore
  right: ev_l31_sand_ignore
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: the attack-derived shield prompt against the sandwich prompt on the same base'
  page: 6
- id: c_l31_fct_escape_none
  left: ev_l31_fct_escape
  right: ev_l31_none_escape
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_fct_escape_sand
  left: ev_l31_fct_escape
  right: ev_l31_sand_escape
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: the attack-derived shield prompt against the sandwich prompt on the same base'
  page: 6
- id: c_l31_fct_fakecom_none
  left: ev_l31_fct_fakecom
  right: ev_l31_none_fakecom
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_fct_fakecom_sand
  left: ev_l31_fct_fakecom
  right: ev_l31_sand_fakecom
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: the attack-derived shield prompt against the sandwich prompt on the same base'
  page: 6
- id: c_l31_fct_combined_none
  left: ev_l31_fct_combined
  right: ev_l31_none_combined
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_l31_fct_combined_sand
  left: ev_l31_fct_combined
  right: ev_l31_sand_combined
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: the attack-derived shield prompt against the sandwich prompt on the same base'
  page: 6
- id: c_l31_fct_util_none
  left: ev_l31_fct_util
  right: ev_l31_none_util
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l31_fct_util_sand
  left: ev_l31_fct_util
  right: ev_l31_sand_util
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: the attack-derived shield prompt against the sandwich prompt on the same base'
  page: 7
claims:
- id: k1
  text: the attack-derived defences outperform existing defences, state of the art
  location: abstract
  anchor: our defense techniques outperform existing defense approaches, achieving state-of-the-art results
  jobs:
  - R1
  supported_by:
  - c_l3_ign_naive_sand
  - c_l3_ign_ignore_sand
  - c_l3_ign_escape_sand
  - c_l3_ign_fakecom_sand
  - c_l3_ign_combined_sand
  - c_l3_fct_naive_sand
  - c_l3_fct_ignore_sand
  - c_l3_fct_escape_sand
  - c_l3_fct_fakecom_sand
  - c_l3_fct_combined_sand
  - c_q2_ign_naive_sand
  - c_q2_ign_ignore_sand
  - c_q2_ign_escape_sand
  - c_q2_ign_fakecom_sand
  - c_q2_ign_combined_sand
  - c_q2_fct_naive_sand
  - c_q2_fct_ignore_sand
  - c_q2_fct_escape_sand
  - c_q2_fct_fakecom_sand
  - c_q2_fct_combined_sand
  - c_l31_ign_naive_sand
  - c_l31_ign_ignore_sand
  - c_l31_ign_escape_sand
  - c_l31_ign_fakecom_sand
  - c_l31_ign_combined_sand
  - c_l31_fct_naive_sand
  - c_l31_fct_ignore_sand
  - c_l31_fct_escape_sand
  - c_l31_fct_fakecom_sand
  - c_l31_fct_combined_sand
  page: 1
coverage:
  R1: direct
  R2clean: direct
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 2 (indirect scenario)
    R2clean: Table 3 (QA accuracy without attacks)

`adaptiveipi_arxiv2025` (2503.00061)

In [ ]:
%%writefile reports/records/adaptiveipi_arxiv2025.yaml
report:
  id: adaptiveipi_arxiv2025
  slug: '2503.00061'
  title: Adaptive Attacks Break Defenses Against Indirect Prompt Injection Attacks on LLM Agents
  first_author: Qiusi Zhan
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2503.00061'
  version_read: arXiv v2
  first_date: '2025-02-27'
  read: F
  role: re_measures
  code_release: https://github.com/uiuc-kang-lab/AdaptiveAttackAgent
  later_version: Findings of NAACL 2025, DOI 10.18653/v1/2025.findings-naacl.395 (from the search record; not read)
  extraction_scope: 'Sections 3 to 6 and Appendices A and B: the instructional prevention, data prompt isolation, sandwich prevention and adversarial fine-tuning bars of Figure 2 (InjecAgent), the numbers stated in Section
    5.2, and Table 6 (AgentDojo); the abstract claim (the three detectors and paraphrasing, which act before an unchanged model, Tables 2 to 4 and the target and valid rates not coded)'
defences:
- id: ip_v
  name: Instructional prevention
  origin: other_report
  base_checkpoint: Vicuna-7B (prompted ReAct agent)
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  components:
  - prompt warns the model to ignore commands in external content
  layout: ReAct agent prompt with the tool response in the context
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Instructional Prevention (IP)
- id: ip_l
  name: Instructional prevention
  origin: other_report
  base_checkpoint: Llama3-8B (fine-tuned agent)
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  components:
  - prompt warns the model to ignore commands in external content
  layout: ReAct agent prompt with the tool response in the context
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Instructional Prevention (IP)
- id: dpi_v
  name: Data prompt isolation
  origin: other_report
  base_checkpoint: Vicuna-7B (prompted ReAct agent)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  components:
  - tool response wrapped in ''' delimiters
  layout: ReAct agent prompt with the tool response in the context
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Data Prompt Isolation (DPI)
- id: sp_v
  name: Sandwich prevention
  origin: other_report
  base_checkpoint: Vicuna-7B (prompted ReAct agent)
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  components:
  - the user instruction repeated after the tool response
  layout: ReAct agent prompt with the tool response in the context
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Sandwich Prevention (SP)
- id: af_v
  name: Adversarial fine-tuning
  origin: other_report
  base_checkpoint: Vicuna-7B (prompted ReAct agent)
  axis1:
  - no_signal
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  components:
  - LoRA fine-tuning on 215 unsuccessful attack cases of this agent
  layout: ReAct agent prompt with the tool response in the context
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Adversarial Finetuning (AF)
  pages:
    components: 14
- id: af_l
  name: Adversarial fine-tuning
  origin: other_report
  base_checkpoint: Llama3-8B (fine-tuned agent)
  axis1:
  - no_signal
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  components:
  - LoRA fine-tuning on 816 unsuccessful attack cases of this agent
  layout: ReAct agent prompt with the tool response in the context
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: Adversarial Finetuning (AF)
  pages:
    components: 14
evaluations:
- id: ev_v_none_static
  defence_id: none:Vicuna-7B (prompted ReAct agent)
  benchmark: 'InjecAgent (1,054 test cases: direct harm and data stealing)'
  attacker_type: static
  attacker_access: black_box
  judge: output executes the attacker tool (ASR-all, invalid outputs count as failures)
  layout_same_as_defence: 'yes'
  measured_by: adaptiveipi_arxiv2025
  page: 6
  attacker_budget: original InjecAgent attacks
  sample_size: 1,054
  decoding: not reported
  pages:
    benchmark: 4
    judge: 5
  baseline_kind: off_the_shelf
- id: ev_l_none_static
  defence_id: none:Llama3-8B (fine-tuned agent)
  benchmark: 'InjecAgent (1,054 test cases: direct harm and data stealing)'
  attacker_type: static
  attacker_access: black_box
  judge: output executes the attacker tool (ASR-all, invalid outputs count as failures)
  layout_same_as_defence: 'yes'
  measured_by: adaptiveipi_arxiv2025
  page: 6
  attacker_budget: original InjecAgent attacks
  sample_size: 1,054
  decoding: not reported
  pages:
    benchmark: 4
    judge: 5
  baseline_kind: off_the_shelf
- id: ev_ip_v_static
  defence_id: ip_v
  benchmark: 'InjecAgent (1,054 test cases: direct harm and data stealing)'
  attacker_type: static
  attacker_access: black_box
  judge: output executes the attacker tool (ASR-all, invalid outputs count as failures)
  layout_same_as_defence: 'yes'
  measured_by: adaptiveipi_arxiv2025
  page: 5
  attacker_budget: original InjecAgent attacks
  sample_size: 1,054
  decoding: not reported
  pages:
    benchmark: 4
    judge: 5
- id: ev_ip_v_adaptive
  defence_id: ip_v
  benchmark: 'InjecAgent (1,054 test cases: direct harm and data stealing)'
  attacker_type: adaptive
  attacker_access: white_box
  judge: output executes the attacker tool (ASR-all, invalid outputs count as failures)
  layout_same_as_defence: 'yes'
  measured_by: adaptiveipi_arxiv2025
  page: 5
  attacker_budget: adaptive GCG adversarial string against this defence (20 tokens, 500 steps for most defences)
  sample_size: 1,054
  decoding: not reported
  pages:
    benchmark: 4
    judge: 5
    attacker_budget: 14
- id: ev_dpi_v_static
  defence_id: dpi_v
  benchmark: 'InjecAgent (1,054 test cases: direct harm and data stealing)'
  attacker_type: static
  attacker_access: black_box
  judge: output executes the attacker tool (ASR-all, invalid outputs count as failures)
  layout_same_as_defence: 'yes'
  measured_by: adaptiveipi_arxiv2025
  page: 5
  attacker_budget: original InjecAgent attacks
  sample_size: 1,054
  decoding: not reported
  pages:
    benchmark: 4
    judge: 5
- id: ev_dpi_v_adaptive
  defence_id: dpi_v
  benchmark: 'InjecAgent (1,054 test cases: direct harm and data stealing)'
  attacker_type: adaptive
  attacker_access: white_box
  judge: output executes the attacker tool (ASR-all, invalid outputs count as failures)
  layout_same_as_defence: 'yes'
  measured_by: adaptiveipi_arxiv2025
  page: 5
  attacker_budget: adaptive GCG adversarial string against this defence (20 tokens, 500 steps for most defences)
  sample_size: 1,054
  decoding: not reported
  pages:
    benchmark: 4
    judge: 5
    attacker_budget: 14
- id: ev_sp_v_static
  defence_id: sp_v
  benchmark: 'InjecAgent (1,054 test cases: direct harm and data stealing)'
  attacker_type: static
  attacker_access: black_box
  judge: output executes the attacker tool (ASR-all, invalid outputs count as failures)
  layout_same_as_defence: 'yes'
  measured_by: adaptiveipi_arxiv2025
  page: 5
  attacker_budget: original InjecAgent attacks
  sample_size: 1,054
  decoding: not reported
  pages:
    benchmark: 4
    judge: 5
- id: ev_sp_v_adaptive
  defence_id: sp_v
  benchmark: 'InjecAgent (1,054 test cases: direct harm and data stealing)'
  attacker_type: adaptive
  attacker_access: white_box
  judge: output executes the attacker tool (ASR-all, invalid outputs count as failures)
  layout_same_as_defence: 'yes'
  measured_by: adaptiveipi_arxiv2025
  page: 5
  attacker_budget: adaptive GCG adversarial string against this defence (20 tokens, 500 steps for most defences)
  sample_size: 1,054
  decoding: not reported
  pages:
    benchmark: 4
    judge: 5
    attacker_budget: 14
- id: ev_af_v_static
  defence_id: af_v
  benchmark: 'InjecAgent (1,054 test cases: direct harm and data stealing)'
  attacker_type: static
  attacker_access: black_box
  judge: output executes the attacker tool (ASR-all, invalid outputs count as failures)
  layout_same_as_defence: 'yes'
  measured_by: adaptiveipi_arxiv2025
  page: 5
  attacker_budget: original InjecAgent attacks
  sample_size: 1,054
  decoding: not reported
  pages:
    benchmark: 4
    judge: 5
- id: ev_af_v_adaptive
  defence_id: af_v
  benchmark: 'InjecAgent (1,054 test cases: direct harm and data stealing)'
  attacker_type: adaptive
  attacker_access: white_box
  judge: output executes the attacker tool (ASR-all, invalid outputs count as failures)
  layout_same_as_defence: 'yes'
  measured_by: adaptiveipi_arxiv2025
  page: 5
  attacker_budget: adaptive GCG adversarial string against this defence (20 tokens, 500 steps for most defences)
  sample_size: 1,054
  decoding: not reported
  pages:
    benchmark: 4
    judge: 5
    attacker_budget: 14
- id: ev_ip_l_static
  defence_id: ip_l
  benchmark: 'InjecAgent (1,054 test cases: direct harm and data stealing)'
  attacker_type: static
  attacker_access: black_box
  judge: output executes the attacker tool (ASR-all, invalid outputs count as failures)
  layout_same_as_defence: 'yes'
  measured_by: adaptiveipi_arxiv2025
  page: 5
  attacker_budget: original InjecAgent attacks
  sample_size: 1,054
  decoding: not reported
  pages:
    benchmark: 4
    judge: 5
- id: ev_ip_l_adaptive
  defence_id: ip_l
  benchmark: 'InjecAgent (1,054 test cases: direct harm and data stealing)'
  attacker_type: adaptive
  attacker_access: white_box
  judge: output executes the attacker tool (ASR-all, invalid outputs count as failures)
  layout_same_as_defence: 'yes'
  measured_by: adaptiveipi_arxiv2025
  page: 5
  attacker_budget: adaptive GCG adversarial string against this defence (20 tokens, 500 steps for most defences)
  sample_size: 1,054
  decoding: not reported
  pages:
    benchmark: 4
    judge: 5
    attacker_budget: 14
- id: ev_af_l_static
  defence_id: af_l
  benchmark: 'InjecAgent (1,054 test cases: direct harm and data stealing)'
  attacker_type: static
  attacker_access: black_box
  judge: output executes the attacker tool (ASR-all, invalid outputs count as failures)
  layout_same_as_defence: 'yes'
  measured_by: adaptiveipi_arxiv2025
  page: 5
  attacker_budget: original InjecAgent attacks
  sample_size: 1,054
  decoding: not reported
  pages:
    benchmark: 4
    judge: 5
- id: ev_af_l_adaptive
  defence_id: af_l
  benchmark: 'InjecAgent (1,054 test cases: direct harm and data stealing)'
  attacker_type: adaptive
  attacker_access: white_box
  judge: output executes the attacker tool (ASR-all, invalid outputs count as failures)
  layout_same_as_defence: 'yes'
  measured_by: adaptiveipi_arxiv2025
  page: 5
  attacker_budget: adaptive GCG adversarial string against this defence (20 tokens, 500 steps for most defences)
  sample_size: 1,054
  decoding: not reported
  pages:
    benchmark: 4
    judge: 5
    attacker_budget: 14
- id: ev_dojo_none_static
  defence_id: none:Llama3-8B (fine-tuned agent)
  benchmark: AgentDojo, 16 cases (two user and two attacker tasks from each of four suites)
  attacker_type: static
  attacker_access: black_box
  judge: output executes the attacker tool (ASR-all, invalid outputs count as failures)
  layout_same_as_defence: 'yes'
  measured_by: adaptiveipi_arxiv2025
  page: 15
  attacker_budget: AgentDojo injections
  sample_size: '16'
  decoding: not reported
  pages:
    benchmark: 14
  baseline_kind: off_the_shelf
- id: ev_dojo_none_adaptive
  defence_id: none:Llama3-8B (fine-tuned agent)
  benchmark: AgentDojo, 16 cases (two user and two attacker tasks from each of four suites)
  attacker_type: adaptive
  attacker_access: white_box
  judge: output executes the attacker tool (ASR-all, invalid outputs count as failures)
  layout_same_as_defence: 'yes'
  measured_by: adaptiveipi_arxiv2025
  page: 15
  attacker_budget: GCG adversarial string
  sample_size: '16'
  decoding: not reported
  pages:
    benchmark: 14
  baseline_kind: off_the_shelf
- id: ev_dojo_ip_static
  defence_id: ip_l
  benchmark: AgentDojo, 16 cases (two user and two attacker tasks from each of four suites)
  attacker_type: static
  attacker_access: black_box
  judge: output executes the attacker tool (ASR-all, invalid outputs count as failures)
  layout_same_as_defence: 'yes'
  measured_by: adaptiveipi_arxiv2025
  page: 15
  attacker_budget: AgentDojo injections
  sample_size: '16'
  decoding: not reported
  pages:
    benchmark: 14
- id: ev_dojo_ip_adaptive
  defence_id: ip_l
  benchmark: AgentDojo, 16 cases (two user and two attacker tasks from each of four suites)
  attacker_type: adaptive
  attacker_access: white_box
  judge: output executes the attacker tool (ASR-all, invalid outputs count as failures)
  layout_same_as_defence: 'yes'
  measured_by: adaptiveipi_arxiv2025
  page: 15
  attacker_budget: GCG adversarial string against this defence
  sample_size: '16'
  decoding: not reported
  pages:
    benchmark: 14
results:
- id: r_v_none_static
  evaluation_id: ev_v_none_static
  quantity: R1
  coverage: direct
  metric_as_printed: Original ASR
  value: '56'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 6
  location: Section 5.2
- id: r_l_none_static
  evaluation_id: ev_l_none_static
  quantity: R1
  coverage: direct
  metric_as_printed: Original ASR
  value: '9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 6
  location: Section 5.2
- id: r_ip_v_static
  evaluation_id: ev_ip_v_static
  quantity: R1
  coverage: direct
  metric_as_printed: ASR-defense
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 5
  location: Figure 2
- id: r_ip_v_adaptive
  evaluation_id: ev_ip_v_adaptive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR-adaptive attack
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 5
  location: Figure 2
- id: r_dpi_v_static
  evaluation_id: ev_dpi_v_static
  quantity: R1
  coverage: direct
  metric_as_printed: ASR-defense
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 5
  location: Figure 2
- id: r_dpi_v_adaptive
  evaluation_id: ev_dpi_v_adaptive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR-adaptive attack
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 5
  location: Figure 2
- id: r_sp_v_static
  evaluation_id: ev_sp_v_static
  quantity: R1
  coverage: direct
  metric_as_printed: ASR-defense
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 5
  location: Figure 2
- id: r_sp_v_adaptive
  evaluation_id: ev_sp_v_adaptive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR-adaptive attack
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 5
  location: Figure 2
- id: r_af_v_static
  evaluation_id: ev_af_v_static
  quantity: R1
  coverage: direct
  metric_as_printed: ASR-defense
  value: '12'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 6
  location: Section 5.2
- id: r_af_v_adaptive
  evaluation_id: ev_af_v_adaptive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR-adaptive attack
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 5
  location: Figure 2
- id: r_ip_l_static
  evaluation_id: ev_ip_l_static
  quantity: R1
  coverage: direct
  metric_as_printed: ASR-defense
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 5
  location: Figure 2
- id: r_ip_l_adaptive
  evaluation_id: ev_ip_l_adaptive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR-adaptive attack
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 5
  location: Figure 2
- id: r_af_l_static
  evaluation_id: ev_af_l_static
  quantity: R1
  coverage: direct
  metric_as_printed: ASR-defense
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 5
  location: Figure 2
- id: r_af_l_adaptive
  evaluation_id: ev_af_l_adaptive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR-adaptive attack
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 5
  location: Figure 2
- id: r_dojo_none_static
  evaluation_id: ev_dojo_none_static
  quantity: R1
  coverage: direct
  metric_as_printed: ASR-defense
  value: '6.25'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_dojo_none_adaptive
  evaluation_id: ev_dojo_none_adaptive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR-adaptive attack
  value: '56.25'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_dojo_ip_static
  evaluation_id: ev_dojo_ip_static
  quantity: R1
  coverage: direct
  metric_as_printed: ASR-defense
  value: '6.25'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_dojo_ip_adaptive
  evaluation_id: ev_dojo_ip_adaptive
  quantity: R1
  coverage: direct
  metric_as_printed: ASR-adaptive attack
  value: '43.75'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
comparisons:
- id: c_af_v_static
  left: ev_af_v_static
  right: ev_v_none_static
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_dojo_ip_static
  left: ev_dojo_ip_static
  right: ev_dojo_none_static
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 15
- id: c_dojo_ip_adaptive
  left: ev_dojo_ip_adaptive
  right: ev_dojo_none_adaptive
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 15
claims:
- id: k1
  text: adaptive attacks bypass all eight defences, each above 50% attack success
  location: abstract
  anchor: consistently achieving an attack success rate of over 50%
  jobs:
  - R1
  supported_by:
  - c_dojo_ip_adaptive
  page: 1
  contradicted_by:
  - r_dojo_ip_adaptive
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Figure 2 (InjecAgent, chart image), Section 5.2 text, Table 6 (AgentDojo)
schema_gaps:
- where: results r_*_static and r_*_adaptive coded figure_only
  note: 'Figure 2 prints each bar''s value as a label inside a raster image, so the values cannot be checked against the page text and are not coded. Read from the image (Vicuna-7B, ASR-defense then ASR-adaptive attack): instructional
    prevention 53 and 91, data prompt isolation 50 and 94, sandwich prevention 15 and 68, adversarial fine-tuning 12 and 65; (Llama3-8B): instructional prevention 1 and 79, adversarial fine-tuning 4 and 85. Section 5.2 states
    56% (Vicuna-7B original), 12% (adversarial fine-tuning) and 9% (Llama3-8B original)'

`pft_arxiv2025` (2505.00626)

In [ ]:
%%writefile reports/records/pft_arxiv2025.yaml
report:
  id: pft_arxiv2025
  slug: '2505.00626'
  title: 'The Illusion of Role Separation: Hidden Shortcuts in LLM Role Learning (and How to Fix Them)'
  first_author: Zihao Wang
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2505.00626'
  version_read: arXiv v2
  first_date: '2025-05-01'
  read: F
  role: proposes
  code_release: none found in the paper
  extraction_scope: 'Sections 3 to 6.2 and Appendix B: Table 1 (base and dataset-initial columns) and Table 2 (dataset-initial, both models); the abstract claim (the dataset-symm column of Table 1, Figures 2 to 8 and Figure
    6''s utility panel, whose training set is not stated, not coded)'
defences:
- id: pft_llama
  name: Position-enhanced fine-tuning (PFT, d = 512)
  origin: this_report
  base_checkpoint: Llama-3-8B-Instruct
  axis1:
  - chat_roles
  - position
  axis2:
  - sft
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 7
  components:
  - position IDs of user tokens shifted by a gap d = 512 after the last system token, order kept within each role
  - LoRA on query and key projections, about 2,300 samples (dataset-initial), user input treated as data for the system task
  layout: key instruction in the system message, the attacker's text in the user message
  layout_compat:
    user_turn: untested
    separate_role: supported
    system_context: untested
    agent_tool_result: untested
  anchor: We call this method Position-enhanced fine-tuning (PFT)
  pages:
    components: 3
    isolation: 8
- id: sft_llama
  name: Vanilla SFT on dataset-initial (no position gap)
  origin: this_report
  base_checkpoint: Llama-3-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 7
  anchor: 'Vanilla SFT: Standard supervised fine-tuning without any modifications'
  layout: key instruction in the system message, the attacker's text in the user message
- id: delim_llama
  name: Delimiter-enhanced SFT (delimiter embeddings tuned)
  origin: this_report
  base_checkpoint: Llama-3-8B-Instruct
  axis1:
  - chat_roles
  - learned_token_features
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 7
  anchor: Delimiter-enhanced SFT
  layout: key instruction in the system message, the attacker's text in the user message
- id: pft_gemma
  name: Position-enhanced fine-tuning (PFT, d = 256)
  origin: this_report
  base_checkpoint: Gemma-2-9b-it
  axis1:
  - chat_roles
  - position
  axis2:
  - sft
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 7
  components:
  - position IDs of user tokens shifted by a gap d = 256 after the last system token, order kept within each role
  - LoRA on query and key projections, about 2,300 samples (dataset-initial), user input treated as data for the system task
  layout: key instruction in the system message, the attacker's text in the user message
  layout_compat:
    user_turn: untested
    separate_role: supported
    system_context: untested
    agent_tool_result: untested
  anchor: We call this method Position-enhanced fine-tuning (PFT)
  pages:
    components: 3
    isolation: 8
- id: sft_gemma
  name: Vanilla SFT on dataset-initial (no position gap)
  origin: this_report
  base_checkpoint: Gemma-2-9b-it
  axis1:
  - chat_roles
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 7
  anchor: 'Vanilla SFT: Standard supervised fine-tuning without any modifications'
  layout: key instruction in the system message, the attacker's text in the user message
- id: delim_gemma
  name: Delimiter-enhanced SFT (delimiter embeddings tuned)
  origin: this_report
  base_checkpoint: Gemma-2-9b-it
  axis1:
  - chat_roles
  - learned_token_features
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 7
  anchor: Delimiter-enhanced SFT
  layout: key instruction in the system message, the attacker's text in the user message
evaluations:
- id: ev_llama_sft_gsum
  defence_id: sft_llama
  benchmark: Gandalf Summarization against a password-manager system prompt
  attacker_type: human
  attacker_access: black_box
  judge: accuracy of the response to the key task (attack resisted)
  layout_same_as_defence: 'yes'
  measured_by: pft_arxiv2025
  page: 8
  attacker_budget: Gandalf Summarization samples from online games
  sample_size: '114'
  decoding: greedy
  pages:
    sample_size: 10
    decoding: 10
- id: ev_llama_sft_gign
  defence_id: sft_llama
  benchmark: Gandalf Ignore against a password-manager system prompt
  attacker_type: human
  attacker_access: black_box
  judge: accuracy of the response to the key task (attack resisted)
  layout_same_as_defence: 'yes'
  measured_by: pft_arxiv2025
  page: 8
  attacker_budget: Gandalf Ignore samples from online games
  sample_size: '500'
  decoding: greedy
  pages:
    sample_size: 10
    decoding: 10
- id: ev_llama_sft_ttext
  defence_id: sft_llama
  benchmark: TensorTrust Extraction against a password-manager system prompt
  attacker_type: human
  attacker_access: black_box
  judge: accuracy of the response to the key task (attack resisted)
  layout_same_as_defence: 'yes'
  measured_by: pft_arxiv2025
  page: 8
  attacker_budget: TensorTrust Extraction samples from online games
  sample_size: '500'
  decoding: greedy
  pages:
    sample_size: 10
    decoding: 10
- id: ev_llama_sft_tthij
  defence_id: sft_llama
  benchmark: TensorTrust Hijacking against a password-manager system prompt
  attacker_type: human
  attacker_access: black_box
  judge: accuracy of the response to the key task (attack resisted)
  layout_same_as_defence: 'yes'
  measured_by: pft_arxiv2025
  page: 8
  attacker_budget: TensorTrust Hijacking samples from online games
  sample_size: '500'
  decoding: greedy
  pages:
    sample_size: 10
    decoding: 10
- id: ev_llama_delim_gsum
  defence_id: delim_llama
  benchmark: Gandalf Summarization against a password-manager system prompt
  attacker_type: human
  attacker_access: black_box
  judge: accuracy of the response to the key task (attack resisted)
  layout_same_as_defence: 'yes'
  measured_by: pft_arxiv2025
  page: 8
  attacker_budget: Gandalf Summarization samples from online games
  sample_size: '114'
  decoding: greedy
  pages:
    sample_size: 10
    decoding: 10
- id: ev_llama_delim_gign
  defence_id: delim_llama
  benchmark: Gandalf Ignore against a password-manager system prompt
  attacker_type: human
  attacker_access: black_box
  judge: accuracy of the response to the key task (attack resisted)
  layout_same_as_defence: 'yes'
  measured_by: pft_arxiv2025
  page: 8
  attacker_budget: Gandalf Ignore samples from online games
  sample_size: '500'
  decoding: greedy
  pages:
    sample_size: 10
    decoding: 10
- id: ev_llama_delim_ttext
  defence_id: delim_llama
  benchmark: TensorTrust Extraction against a password-manager system prompt
  attacker_type: human
  attacker_access: black_box
  judge: accuracy of the response to the key task (attack resisted)
  layout_same_as_defence: 'yes'
  measured_by: pft_arxiv2025
  page: 8
  attacker_budget: TensorTrust Extraction samples from online games
  sample_size: '500'
  decoding: greedy
  pages:
    sample_size: 10
    decoding: 10
- id: ev_llama_delim_tthij
  defence_id: delim_llama
  benchmark: TensorTrust Hijacking against a password-manager system prompt
  attacker_type: human
  attacker_access: black_box
  judge: accuracy of the response to the key task (attack resisted)
  layout_same_as_defence: 'yes'
  measured_by: pft_arxiv2025
  page: 8
  attacker_budget: TensorTrust Hijacking samples from online games
  sample_size: '500'
  decoding: greedy
  pages:
    sample_size: 10
    decoding: 10
- id: ev_llama_pft_gsum
  defence_id: pft_llama
  benchmark: Gandalf Summarization against a password-manager system prompt
  attacker_type: human
  attacker_access: black_box
  judge: accuracy of the response to the key task (attack resisted)
  layout_same_as_defence: 'yes'
  measured_by: pft_arxiv2025
  page: 8
  attacker_budget: Gandalf Summarization samples from online games
  sample_size: '114'
  decoding: greedy
  pages:
    sample_size: 10
    decoding: 10
- id: ev_llama_pft_gign
  defence_id: pft_llama
  benchmark: Gandalf Ignore against a password-manager system prompt
  attacker_type: human
  attacker_access: black_box
  judge: accuracy of the response to the key task (attack resisted)
  layout_same_as_defence: 'yes'
  measured_by: pft_arxiv2025
  page: 8
  attacker_budget: Gandalf Ignore samples from online games
  sample_size: '500'
  decoding: greedy
  pages:
    sample_size: 10
    decoding: 10
- id: ev_llama_pft_ttext
  defence_id: pft_llama
  benchmark: TensorTrust Extraction against a password-manager system prompt
  attacker_type: human
  attacker_access: black_box
  judge: accuracy of the response to the key task (attack resisted)
  layout_same_as_defence: 'yes'
  measured_by: pft_arxiv2025
  page: 8
  attacker_budget: TensorTrust Extraction samples from online games
  sample_size: '500'
  decoding: greedy
  pages:
    sample_size: 10
    decoding: 10
- id: ev_llama_pft_tthij
  defence_id: pft_llama
  benchmark: TensorTrust Hijacking against a password-manager system prompt
  attacker_type: human
  attacker_access: black_box
  judge: accuracy of the response to the key task (attack resisted)
  layout_same_as_defence: 'yes'
  measured_by: pft_arxiv2025
  page: 8
  attacker_budget: TensorTrust Hijacking samples from online games
  sample_size: '500'
  decoding: greedy
  pages:
    sample_size: 10
    decoding: 10
- id: ev_gemma_sft_gsum
  defence_id: sft_gemma
  benchmark: Gandalf Summarization against a password-manager system prompt
  attacker_type: human
  attacker_access: black_box
  judge: accuracy of the response to the key task (attack resisted)
  layout_same_as_defence: 'yes'
  measured_by: pft_arxiv2025
  page: 8
  attacker_budget: Gandalf Summarization samples from online games
  sample_size: '114'
  decoding: greedy
  pages:
    sample_size: 10
    decoding: 10
- id: ev_gemma_sft_gign
  defence_id: sft_gemma
  benchmark: Gandalf Ignore against a password-manager system prompt
  attacker_type: human
  attacker_access: black_box
  judge: accuracy of the response to the key task (attack resisted)
  layout_same_as_defence: 'yes'
  measured_by: pft_arxiv2025
  page: 8
  attacker_budget: Gandalf Ignore samples from online games
  sample_size: '500'
  decoding: greedy
  pages:
    sample_size: 10
    decoding: 10
- id: ev_gemma_sft_ttext
  defence_id: sft_gemma
  benchmark: TensorTrust Extraction against a password-manager system prompt
  attacker_type: human
  attacker_access: black_box
  judge: accuracy of the response to the key task (attack resisted)
  layout_same_as_defence: 'yes'
  measured_by: pft_arxiv2025
  page: 8
  attacker_budget: TensorTrust Extraction samples from online games
  sample_size: '500'
  decoding: greedy
  pages:
    sample_size: 10
    decoding: 10
- id: ev_gemma_sft_tthij
  defence_id: sft_gemma
  benchmark: TensorTrust Hijacking against a password-manager system prompt
  attacker_type: human
  attacker_access: black_box
  judge: accuracy of the response to the key task (attack resisted)
  layout_same_as_defence: 'yes'
  measured_by: pft_arxiv2025
  page: 8
  attacker_budget: TensorTrust Hijacking samples from online games
  sample_size: '500'
  decoding: greedy
  pages:
    sample_size: 10
    decoding: 10
- id: ev_gemma_delim_gsum
  defence_id: delim_gemma
  benchmark: Gandalf Summarization against a password-manager system prompt
  attacker_type: human
  attacker_access: black_box
  judge: accuracy of the response to the key task (attack resisted)
  layout_same_as_defence: 'yes'
  measured_by: pft_arxiv2025
  page: 8
  attacker_budget: Gandalf Summarization samples from online games
  sample_size: '114'
  decoding: greedy
  pages:
    sample_size: 10
    decoding: 10
- id: ev_gemma_delim_gign
  defence_id: delim_gemma
  benchmark: Gandalf Ignore against a password-manager system prompt
  attacker_type: human
  attacker_access: black_box
  judge: accuracy of the response to the key task (attack resisted)
  layout_same_as_defence: 'yes'
  measured_by: pft_arxiv2025
  page: 8
  attacker_budget: Gandalf Ignore samples from online games
  sample_size: '500'
  decoding: greedy
  pages:
    sample_size: 10
    decoding: 10
- id: ev_gemma_delim_ttext
  defence_id: delim_gemma
  benchmark: TensorTrust Extraction against a password-manager system prompt
  attacker_type: human
  attacker_access: black_box
  judge: accuracy of the response to the key task (attack resisted)
  layout_same_as_defence: 'yes'
  measured_by: pft_arxiv2025
  page: 8
  attacker_budget: TensorTrust Extraction samples from online games
  sample_size: '500'
  decoding: greedy
  pages:
    sample_size: 10
    decoding: 10
- id: ev_gemma_delim_tthij
  defence_id: delim_gemma
  benchmark: TensorTrust Hijacking against a password-manager system prompt
  attacker_type: human
  attacker_access: black_box
  judge: accuracy of the response to the key task (attack resisted)
  layout_same_as_defence: 'yes'
  measured_by: pft_arxiv2025
  page: 8
  attacker_budget: TensorTrust Hijacking samples from online games
  sample_size: '500'
  decoding: greedy
  pages:
    sample_size: 10
    decoding: 10
- id: ev_gemma_pft_gsum
  defence_id: pft_gemma
  benchmark: Gandalf Summarization against a password-manager system prompt
  attacker_type: human
  attacker_access: black_box
  judge: accuracy of the response to the key task (attack resisted)
  layout_same_as_defence: 'yes'
  measured_by: pft_arxiv2025
  page: 8
  attacker_budget: Gandalf Summarization samples from online games
  sample_size: '114'
  decoding: greedy
  pages:
    sample_size: 10
    decoding: 10
- id: ev_gemma_pft_gign
  defence_id: pft_gemma
  benchmark: Gandalf Ignore against a password-manager system prompt
  attacker_type: human
  attacker_access: black_box
  judge: accuracy of the response to the key task (attack resisted)
  layout_same_as_defence: 'yes'
  measured_by: pft_arxiv2025
  page: 8
  attacker_budget: Gandalf Ignore samples from online games
  sample_size: '500'
  decoding: greedy
  pages:
    sample_size: 10
    decoding: 10
- id: ev_gemma_pft_ttext
  defence_id: pft_gemma
  benchmark: TensorTrust Extraction against a password-manager system prompt
  attacker_type: human
  attacker_access: black_box
  judge: accuracy of the response to the key task (attack resisted)
  layout_same_as_defence: 'yes'
  measured_by: pft_arxiv2025
  page: 8
  attacker_budget: TensorTrust Extraction samples from online games
  sample_size: '500'
  decoding: greedy
  pages:
    sample_size: 10
    decoding: 10
- id: ev_gemma_pft_tthij
  defence_id: pft_gemma
  benchmark: TensorTrust Hijacking against a password-manager system prompt
  attacker_type: human
  attacker_access: black_box
  judge: accuracy of the response to the key task (attack resisted)
  layout_same_as_defence: 'yes'
  measured_by: pft_arxiv2025
  page: 8
  attacker_budget: TensorTrust Hijacking samples from online games
  sample_size: '500'
  decoding: greedy
  pages:
    sample_size: 10
    decoding: 10
- id: ev_llama_base_gsum
  defence_id: none:Llama-3-8B-Instruct
  benchmark: Gandalf Summarization against a password-manager system prompt
  attacker_type: human
  attacker_access: black_box
  judge: accuracy of the response to the key task (attack resisted)
  layout_same_as_defence: 'yes'
  measured_by: pft_arxiv2025
  page: 3
  attacker_budget: Gandalf Summarization samples from online games
  sample_size: '114'
  decoding: greedy
  baseline_kind: off_the_shelf
  pages:
    sample_size: 10
    decoding: 10
- id: ev_llama_base_gign
  defence_id: none:Llama-3-8B-Instruct
  benchmark: Gandalf Ignore against a password-manager system prompt
  attacker_type: human
  attacker_access: black_box
  judge: accuracy of the response to the key task (attack resisted)
  layout_same_as_defence: 'yes'
  measured_by: pft_arxiv2025
  page: 3
  attacker_budget: Gandalf Ignore samples from online games
  sample_size: '500'
  decoding: greedy
  baseline_kind: off_the_shelf
  pages:
    sample_size: 10
    decoding: 10
- id: ev_llama_base_ttext
  defence_id: none:Llama-3-8B-Instruct
  benchmark: TensorTrust Extraction against a password-manager system prompt
  attacker_type: human
  attacker_access: black_box
  judge: accuracy of the response to the key task (attack resisted)
  layout_same_as_defence: 'yes'
  measured_by: pft_arxiv2025
  page: 3
  attacker_budget: TensorTrust Extraction samples from online games
  sample_size: '500'
  decoding: greedy
  baseline_kind: off_the_shelf
  pages:
    sample_size: 10
    decoding: 10
- id: ev_llama_base_tthij
  defence_id: none:Llama-3-8B-Instruct
  benchmark: TensorTrust Hijacking against a password-manager system prompt
  attacker_type: human
  attacker_access: black_box
  judge: accuracy of the response to the key task (attack resisted)
  layout_same_as_defence: 'yes'
  measured_by: pft_arxiv2025
  page: 3
  attacker_budget: TensorTrust Hijacking samples from online games
  sample_size: '500'
  decoding: greedy
  baseline_kind: off_the_shelf
  pages:
    sample_size: 10
    decoding: 10
results:
- id: r_llama_sft_gsum
  evaluation_id: ev_llama_sft_gsum
  quantity: R1
  coverage: direct
  metric_as_printed: Accuracy, Gandalf Summarization
  value: '90'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_llama_sft_gign
  evaluation_id: ev_llama_sft_gign
  quantity: R1
  coverage: direct
  metric_as_printed: Accuracy, Gandalf Ignore
  value: '86'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_llama_sft_ttext
  evaluation_id: ev_llama_sft_ttext
  quantity: R1
  coverage: direct
  metric_as_printed: Accuracy, TensorTrust Extraction
  value: '33'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_llama_sft_tthij
  evaluation_id: ev_llama_sft_tthij
  quantity: R1
  coverage: direct
  metric_as_printed: Accuracy, TensorTrust Hijacking
  value: '33'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_llama_delim_gsum
  evaluation_id: ev_llama_delim_gsum
  quantity: R1
  coverage: direct
  metric_as_printed: Accuracy, Gandalf Summarization
  value: '93'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_llama_delim_gign
  evaluation_id: ev_llama_delim_gign
  quantity: R1
  coverage: direct
  metric_as_printed: Accuracy, Gandalf Ignore
  value: '89'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_llama_delim_ttext
  evaluation_id: ev_llama_delim_ttext
  quantity: R1
  coverage: direct
  metric_as_printed: Accuracy, TensorTrust Extraction
  value: '35'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_llama_delim_tthij
  evaluation_id: ev_llama_delim_tthij
  quantity: R1
  coverage: direct
  metric_as_printed: Accuracy, TensorTrust Hijacking
  value: '32'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_llama_pft_gsum
  evaluation_id: ev_llama_pft_gsum
  quantity: R1
  coverage: direct
  metric_as_printed: Accuracy, Gandalf Summarization
  value: '85'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_llama_pft_gign
  evaluation_id: ev_llama_pft_gign
  quantity: R1
  coverage: direct
  metric_as_printed: Accuracy, Gandalf Ignore
  value: '94'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_llama_pft_ttext
  evaluation_id: ev_llama_pft_ttext
  quantity: R1
  coverage: direct
  metric_as_printed: Accuracy, TensorTrust Extraction
  value: '62'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_llama_pft_tthij
  evaluation_id: ev_llama_pft_tthij
  quantity: R1
  coverage: direct
  metric_as_printed: Accuracy, TensorTrust Hijacking
  value: '37'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_gemma_sft_gsum
  evaluation_id: ev_gemma_sft_gsum
  quantity: R1
  coverage: direct
  metric_as_printed: Accuracy, Gandalf Summarization
  value: '99'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_gemma_sft_gign
  evaluation_id: ev_gemma_sft_gign
  quantity: R1
  coverage: direct
  metric_as_printed: Accuracy, Gandalf Ignore
  value: '100'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_gemma_sft_ttext
  evaluation_id: ev_gemma_sft_ttext
  quantity: R1
  coverage: direct
  metric_as_printed: Accuracy, TensorTrust Extraction
  value: '70'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_gemma_sft_tthij
  evaluation_id: ev_gemma_sft_tthij
  quantity: R1
  coverage: direct
  metric_as_printed: Accuracy, TensorTrust Hijacking
  value: '37'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_gemma_delim_gsum
  evaluation_id: ev_gemma_delim_gsum
  quantity: R1
  coverage: direct
  metric_as_printed: Accuracy, Gandalf Summarization
  value: '99'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_gemma_delim_gign
  evaluation_id: ev_gemma_delim_gign
  quantity: R1
  coverage: direct
  metric_as_printed: Accuracy, Gandalf Ignore
  value: '100'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_gemma_delim_ttext
  evaluation_id: ev_gemma_delim_ttext
  quantity: R1
  coverage: direct
  metric_as_printed: Accuracy, TensorTrust Extraction
  value: '75'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_gemma_delim_tthij
  evaluation_id: ev_gemma_delim_tthij
  quantity: R1
  coverage: direct
  metric_as_printed: Accuracy, TensorTrust Hijacking
  value: '37'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_gemma_pft_gsum
  evaluation_id: ev_gemma_pft_gsum
  quantity: R1
  coverage: direct
  metric_as_printed: Accuracy, Gandalf Summarization
  value: '99'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_gemma_pft_gign
  evaluation_id: ev_gemma_pft_gign
  quantity: R1
  coverage: direct
  metric_as_printed: Accuracy, Gandalf Ignore
  value: '100'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_gemma_pft_ttext
  evaluation_id: ev_gemma_pft_ttext
  quantity: R1
  coverage: direct
  metric_as_printed: Accuracy, TensorTrust Extraction
  value: '92'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_gemma_pft_tthij
  evaluation_id: ev_gemma_pft_tthij
  quantity: R1
  coverage: direct
  metric_as_printed: Accuracy, TensorTrust Hijacking
  value: '50'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 2
- id: r_llama_base_gsum
  evaluation_id: ev_llama_base_gsum
  quantity: R1
  coverage: direct
  metric_as_printed: Accuracy, Gandalf Summarization
  value: '10'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_llama_base_gign
  evaluation_id: ev_llama_base_gign
  quantity: R1
  coverage: direct
  metric_as_printed: Accuracy, Gandalf Ignore
  value: '0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_llama_base_ttext
  evaluation_id: ev_llama_base_ttext
  quantity: R1
  coverage: direct
  metric_as_printed: Accuracy, TensorTrust Extraction
  value: '4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_llama_base_tthij
  evaluation_id: ev_llama_base_tthij
  quantity: R1
  coverage: direct
  metric_as_printed: Accuracy, TensorTrust Hijacking
  value: '4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
comparisons:
- id: c_llama_gsum_pft_sft
  left: ev_llama_pft_gsum
  right: ev_llama_sft_gsum
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the position-ID gap (PFT) against the same fine-tuning without it
  page: 8
- id: c_llama_gsum_pft_delim
  left: ev_llama_pft_gsum
  right: ev_llama_delim_gsum
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: position-ID gap against tuned delimiter embeddings'
  page: 8
- id: c_llama_gign_pft_sft
  left: ev_llama_pft_gign
  right: ev_llama_sft_gign
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the position-ID gap (PFT) against the same fine-tuning without it
  page: 8
- id: c_llama_gign_pft_delim
  left: ev_llama_pft_gign
  right: ev_llama_delim_gign
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: position-ID gap against tuned delimiter embeddings'
  page: 8
- id: c_llama_ttext_pft_sft
  left: ev_llama_pft_ttext
  right: ev_llama_sft_ttext
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the position-ID gap (PFT) against the same fine-tuning without it
  page: 8
- id: c_llama_ttext_pft_delim
  left: ev_llama_pft_ttext
  right: ev_llama_delim_ttext
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: position-ID gap against tuned delimiter embeddings'
  page: 8
- id: c_llama_tthij_pft_sft
  left: ev_llama_pft_tthij
  right: ev_llama_sft_tthij
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the position-ID gap (PFT) against the same fine-tuning without it
  page: 8
- id: c_llama_tthij_pft_delim
  left: ev_llama_pft_tthij
  right: ev_llama_delim_tthij
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: position-ID gap against tuned delimiter embeddings'
  page: 8
- id: c_gemma_gsum_pft_sft
  left: ev_gemma_pft_gsum
  right: ev_gemma_sft_gsum
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the position-ID gap (PFT) against the same fine-tuning without it
  page: 8
- id: c_gemma_gsum_pft_delim
  left: ev_gemma_pft_gsum
  right: ev_gemma_delim_gsum
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: position-ID gap against tuned delimiter embeddings'
  page: 8
- id: c_gemma_gign_pft_sft
  left: ev_gemma_pft_gign
  right: ev_gemma_sft_gign
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the position-ID gap (PFT) against the same fine-tuning without it
  page: 8
- id: c_gemma_gign_pft_delim
  left: ev_gemma_pft_gign
  right: ev_gemma_delim_gign
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: position-ID gap against tuned delimiter embeddings'
  page: 8
- id: c_gemma_ttext_pft_sft
  left: ev_gemma_pft_ttext
  right: ev_gemma_sft_ttext
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the position-ID gap (PFT) against the same fine-tuning without it
  page: 8
- id: c_gemma_ttext_pft_delim
  left: ev_gemma_pft_ttext
  right: ev_gemma_delim_ttext
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: position-ID gap against tuned delimiter embeddings'
  page: 8
- id: c_gemma_tthij_pft_sft
  left: ev_gemma_pft_tthij
  right: ev_gemma_sft_tthij
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the position-ID gap (PFT) against the same fine-tuning without it
  page: 8
- id: c_gemma_tthij_pft_delim
  left: ev_gemma_pft_tthij
  right: ev_gemma_delim_tthij
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: 'the defence: position-ID gap against tuned delimiter embeddings'
  page: 8
- id: c_llama_gsum_sft_base
  left: ev_llama_sft_gsum
  right: ev_llama_base_gsum
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: role-separation fine-tuning (dataset-initial) against the base model
  page: 3
- id: c_llama_gign_sft_base
  left: ev_llama_sft_gign
  right: ev_llama_base_gign
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: role-separation fine-tuning (dataset-initial) against the base model
  page: 3
- id: c_llama_ttext_sft_base
  left: ev_llama_sft_ttext
  right: ev_llama_base_ttext
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: role-separation fine-tuning (dataset-initial) against the base model
  page: 3
- id: c_llama_tthij_sft_base
  left: ev_llama_sft_tthij
  right: ev_llama_base_tthij
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: role-separation fine-tuning (dataset-initial) against the base model
  page: 3
claims:
- id: k1
  text: manipulating position IDs gives clearer role distinctions and less reliance on shortcuts
  location: abstract
  anchor: manipulating position IDs helps the model learn clearer distinctions
  jobs:
  - R1
  supported_by:
  - c_llama_gsum_pft_sft
  - c_llama_gign_pft_sft
  - c_llama_ttext_pft_sft
  - c_llama_tthij_pft_sft
  - c_gemma_gsum_pft_sft
  - c_gemma_gign_pft_sft
  - c_gemma_ttext_pft_sft
  - c_gemma_tthij_pft_sft
  page: 1
  contradicted_by:
  - r_llama_pft_gsum
coverage:
  R1: direct
  R2clean: unclear
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Tables 1 and 2 (accuracy under attack)
    R2clean: Figure 6(a) password-dataset accuracy; which training set those models used is not stated

`gemini_arxiv2025` (2505.14534)

In [ ]:
%%writefile reports/records/gemini_arxiv2025.yaml
report:
  id: gemini_arxiv2025
  slug: '2505.14534'
  title: Lessons from Defending Gemini Against Indirect Prompt Injections
  first_author: Chongyang Shi
  venue: arXiv preprint (technical report)
  peer_reviewed: 'no'
  arxiv: '2505.14534'
  version_read: arXiv v1
  first_date: '2025-05-20'
  read: F
  role: mixed
  code_release: none (closed model)
  extraction_scope: 'Sections 5.3, 7 to 9.2 and Appendices D and E: the ICL, Spotlighting and Warning rows of Tables 3 and 4 (Gemini 2.0), Table 1 (Gemini 2.5 against 2.0), the non-adaptive Gemini 2.5 result and the Warning
    plus Gemini 2.5 result in Sections 9.1 and 9.2; the abstract claim (paraphrasing and the classification defences, Table 2''s undefended runs, Appendices F to H not coded)'
defences:
- id: aft_g25
  name: Adversarial fine-tuning of Gemini (SFT on corrected responses, with RL adversarial training)
  origin: this_report
  base_checkpoint: Gemini 2.5 (Gemini-2.5-Flash-Preview-04-17 named for the leaderboard score)
  axis1:
  - no_signal
  axis2:
  - sft
  - adversarial_rl
  isolation: no_ablation
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 20
  components:
  - conversations with tool calls and private data, injected with triggers from the automated red-teaming attacks
  - corrective responses produced with the Warning defence and filtered by the user instruction classifier
  - supervised fine-tuning, with reinforcement learning adversarial training applied alongside
  layout: conversation history with tool calls; the retrieved tool output carries the trigger
  layout_compat: &id001
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  released_weights: no (closed model)
  anchor: Synthesizing Corrective Responses
- id: icl_g20
  name: In-context learning defence
  origin: other_report
  base_checkpoint: Gemini 2.0
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 16
  components:
  - a description of indirect prompt injection with examples, before the retrieved trigger
  layout: conversation history with tool calls; the retrieved tool output carries the trigger
  layout_compat: *id001
  anchor: In-context learning (ICL)
- id: spot_g20
  name: Spotlighting (control tokens interleaved in the trigger)
  origin: other_report
  base_checkpoint: Gemini 2.0
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 16
  components:
  - control tokens interleaved through the retrieved data and an instruction not to trust text between them
  layout: conversation history with tool calls; the retrieved tool output carries the trigger
  layout_compat: *id001
  anchor: We interleave control tokens throughout the retrieved trigger
- id: warn_g20
  name: Warning
  origin: other_report
  base_checkpoint: Gemini 2.0
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 16
  components:
  - a warning not to expose private information, placed after the retrieved content
  layout: conversation history with tool calls; the retrieved tool output carries the trigger
  layout_compat: *id001
  anchor: insert a warning message instructing the model
- id: warn_g25
  name: Warning (on the adversarially fine-tuned Gemini 2.5)
  origin: this_report
  base_checkpoint: Gemini 2.5 (Gemini-2.5-Flash-Preview-04-17 named for the leaderboard score)
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 22
  components:
  - the same warning, on Gemini 2.5
  layout: conversation history with tool calls; the retrieved tool output carries the trigger
  layout_compat: *id001
  anchor: combining the Warning defense with Gemini 2.5
evaluations:
- id: ev_t3_icl_g20_ac
  defence_id: icl_g20
  benchmark: synthetic conversation histories with tool calls (held-out test set), Gemini 2.0
  attacker_type: static
  attacker_access: black_box
  judge: fraction of simulated private information exfiltrated to the attacker; best trigger of the attack class on the held-out test set
  layout_same_as_defence: 'yes'
  measured_by: gemini_arxiv2025
  page: 36
  attacker_budget: Actor Critic triggers optimised against undefended Gemini 2.0
  decoding: not reported
  pages:
    judge: 14
- id: ev_t3_icl_g20_bs
  defence_id: icl_g20
  benchmark: synthetic conversation histories with tool calls (held-out test set), Gemini 2.0
  attacker_type: static
  attacker_access: black_box
  judge: fraction of simulated private information exfiltrated to the attacker; best trigger of the attack class on the held-out test set
  layout_same_as_defence: 'yes'
  measured_by: gemini_arxiv2025
  page: 36
  attacker_budget: Beam Search triggers optimised against undefended Gemini 2.0
  decoding: not reported
  pages:
    judge: 14
- id: ev_t3_icl_g20_tap
  defence_id: icl_g20
  benchmark: synthetic conversation histories with tool calls (held-out test set), Gemini 2.0
  attacker_type: static
  attacker_access: black_box
  judge: fraction of simulated private information exfiltrated to the attacker; best trigger of the attack class on the held-out test set
  layout_same_as_defence: 'yes'
  measured_by: gemini_arxiv2025
  page: 36
  attacker_budget: TAP triggers optimised against undefended Gemini 2.0
  decoding: not reported
  pages:
    judge: 14
- id: ev_t3_spot_g20_ac
  defence_id: spot_g20
  benchmark: synthetic conversation histories with tool calls (held-out test set), Gemini 2.0
  attacker_type: static
  attacker_access: black_box
  judge: fraction of simulated private information exfiltrated to the attacker; best trigger of the attack class on the held-out test set
  layout_same_as_defence: 'yes'
  measured_by: gemini_arxiv2025
  page: 36
  attacker_budget: Actor Critic triggers optimised against undefended Gemini 2.0
  decoding: not reported
  pages:
    judge: 14
- id: ev_t3_spot_g20_bs
  defence_id: spot_g20
  benchmark: synthetic conversation histories with tool calls (held-out test set), Gemini 2.0
  attacker_type: static
  attacker_access: black_box
  judge: fraction of simulated private information exfiltrated to the attacker; best trigger of the attack class on the held-out test set
  layout_same_as_defence: 'yes'
  measured_by: gemini_arxiv2025
  page: 36
  attacker_budget: Beam Search triggers optimised against undefended Gemini 2.0
  decoding: not reported
  pages:
    judge: 14
- id: ev_t3_spot_g20_tap
  defence_id: spot_g20
  benchmark: synthetic conversation histories with tool calls (held-out test set), Gemini 2.0
  attacker_type: static
  attacker_access: black_box
  judge: fraction of simulated private information exfiltrated to the attacker; best trigger of the attack class on the held-out test set
  layout_same_as_defence: 'yes'
  measured_by: gemini_arxiv2025
  page: 36
  attacker_budget: TAP triggers optimised against undefended Gemini 2.0
  decoding: not reported
  pages:
    judge: 14
- id: ev_t3_warn_g20_ac
  defence_id: warn_g20
  benchmark: synthetic conversation histories with tool calls (held-out test set), Gemini 2.0
  attacker_type: static
  attacker_access: black_box
  judge: fraction of simulated private information exfiltrated to the attacker; best trigger of the attack class on the held-out test set
  layout_same_as_defence: 'yes'
  measured_by: gemini_arxiv2025
  page: 36
  attacker_budget: Actor Critic triggers optimised against undefended Gemini 2.0
  decoding: not reported
  pages:
    judge: 14
- id: ev_t3_warn_g20_bs
  defence_id: warn_g20
  benchmark: synthetic conversation histories with tool calls (held-out test set), Gemini 2.0
  attacker_type: static
  attacker_access: black_box
  judge: fraction of simulated private information exfiltrated to the attacker; best trigger of the attack class on the held-out test set
  layout_same_as_defence: 'yes'
  measured_by: gemini_arxiv2025
  page: 36
  attacker_budget: Beam Search triggers optimised against undefended Gemini 2.0
  decoding: not reported
  pages:
    judge: 14
- id: ev_t3_warn_g20_tap
  defence_id: warn_g20
  benchmark: synthetic conversation histories with tool calls (held-out test set), Gemini 2.0
  attacker_type: static
  attacker_access: black_box
  judge: fraction of simulated private information exfiltrated to the attacker; best trigger of the attack class on the held-out test set
  layout_same_as_defence: 'yes'
  measured_by: gemini_arxiv2025
  page: 36
  attacker_budget: TAP triggers optimised against undefended Gemini 2.0
  decoding: not reported
  pages:
    judge: 14
- id: ev_t4_icl_g20_ac
  defence_id: icl_g20
  benchmark: synthetic conversation histories with tool calls (held-out test set), Gemini 2.0
  attacker_type: adaptive
  attacker_access: black_box
  judge: fraction of simulated private information exfiltrated to the attacker; best trigger of the attack class on the held-out test set
  layout_same_as_defence: 'yes'
  measured_by: gemini_arxiv2025
  page: 36
  attacker_budget: Actor Critic triggers optimised against the defended model
  decoding: not reported
  pages:
    judge: 14
- id: ev_t4_icl_g20_bs
  defence_id: icl_g20
  benchmark: synthetic conversation histories with tool calls (held-out test set), Gemini 2.0
  attacker_type: adaptive
  attacker_access: black_box
  judge: fraction of simulated private information exfiltrated to the attacker; best trigger of the attack class on the held-out test set
  layout_same_as_defence: 'yes'
  measured_by: gemini_arxiv2025
  page: 36
  attacker_budget: Beam Search triggers optimised against the defended model
  decoding: not reported
  pages:
    judge: 14
- id: ev_t4_icl_g20_tap
  defence_id: icl_g20
  benchmark: synthetic conversation histories with tool calls (held-out test set), Gemini 2.0
  attacker_type: adaptive
  attacker_access: black_box
  judge: fraction of simulated private information exfiltrated to the attacker; best trigger of the attack class on the held-out test set
  layout_same_as_defence: 'yes'
  measured_by: gemini_arxiv2025
  page: 36
  attacker_budget: TAP triggers optimised against the defended model
  decoding: not reported
  pages:
    judge: 14
- id: ev_t4_spot_g20_ac
  defence_id: spot_g20
  benchmark: synthetic conversation histories with tool calls (held-out test set), Gemini 2.0
  attacker_type: adaptive
  attacker_access: black_box
  judge: fraction of simulated private information exfiltrated to the attacker; best trigger of the attack class on the held-out test set
  layout_same_as_defence: 'yes'
  measured_by: gemini_arxiv2025
  page: 36
  attacker_budget: Actor Critic triggers optimised against the defended model
  decoding: not reported
  pages:
    judge: 14
- id: ev_t4_spot_g20_bs
  defence_id: spot_g20
  benchmark: synthetic conversation histories with tool calls (held-out test set), Gemini 2.0
  attacker_type: adaptive
  attacker_access: black_box
  judge: fraction of simulated private information exfiltrated to the attacker; best trigger of the attack class on the held-out test set
  layout_same_as_defence: 'yes'
  measured_by: gemini_arxiv2025
  page: 36
  attacker_budget: Beam Search triggers optimised against the defended model
  decoding: not reported
  pages:
    judge: 14
- id: ev_t4_spot_g20_tap
  defence_id: spot_g20
  benchmark: synthetic conversation histories with tool calls (held-out test set), Gemini 2.0
  attacker_type: adaptive
  attacker_access: black_box
  judge: fraction of simulated private information exfiltrated to the attacker; best trigger of the attack class on the held-out test set
  layout_same_as_defence: 'yes'
  measured_by: gemini_arxiv2025
  page: 36
  attacker_budget: TAP triggers optimised against the defended model
  decoding: not reported
  pages:
    judge: 14
- id: ev_t4_warn_g20_ac
  defence_id: warn_g20
  benchmark: synthetic conversation histories with tool calls (held-out test set), Gemini 2.0
  attacker_type: adaptive
  attacker_access: black_box
  judge: fraction of simulated private information exfiltrated to the attacker; best trigger of the attack class on the held-out test set
  layout_same_as_defence: 'yes'
  measured_by: gemini_arxiv2025
  page: 36
  attacker_budget: Actor Critic triggers optimised against the defended model
  decoding: not reported
  pages:
    judge: 14
- id: ev_t4_warn_g20_bs
  defence_id: warn_g20
  benchmark: synthetic conversation histories with tool calls (held-out test set), Gemini 2.0
  attacker_type: adaptive
  attacker_access: black_box
  judge: fraction of simulated private information exfiltrated to the attacker; best trigger of the attack class on the held-out test set
  layout_same_as_defence: 'yes'
  measured_by: gemini_arxiv2025
  page: 36
  attacker_budget: Beam Search triggers optimised against the defended model
  decoding: not reported
  pages:
    judge: 14
- id: ev_t4_warn_g20_tap
  defence_id: warn_g20
  benchmark: synthetic conversation histories with tool calls (held-out test set), Gemini 2.0
  attacker_type: adaptive
  attacker_access: black_box
  judge: fraction of simulated private information exfiltrated to the attacker; best trigger of the attack class on the held-out test set
  layout_same_as_defence: 'yes'
  measured_by: gemini_arxiv2025
  page: 36
  attacker_budget: TAP triggers optimised against the defended model
  decoding: not reported
  pages:
    judge: 14
- id: ev_t1_email_g25_tap
  defence_id: aft_g25
  benchmark: Email function, non-JSON trigger, passport number
  attacker_type: adaptive
  attacker_access: black_box
  judge: fraction of simulated private information exfiltrated to the attacker; best trigger of the attack class on the held-out test set
  layout_same_as_defence: 'yes'
  measured_by: gemini_arxiv2025
  page: 21
  attacker_budget: TAP triggers optimised against this model
  decoding: not reported
  pages:
    judge: 14
- id: ev_t1_email_g25_ac
  defence_id: aft_g25
  benchmark: Email function, non-JSON trigger, passport number
  attacker_type: adaptive
  attacker_access: black_box
  judge: fraction of simulated private information exfiltrated to the attacker; best trigger of the attack class on the held-out test set
  layout_same_as_defence: 'yes'
  measured_by: gemini_arxiv2025
  page: 21
  attacker_budget: Actor-Critic triggers optimised against this model
  decoding: not reported
  pages:
    judge: 14
- id: ev_t1_email_g25_bs
  defence_id: aft_g25
  benchmark: Email function, non-JSON trigger, passport number
  attacker_type: adaptive
  attacker_access: black_box
  judge: fraction of simulated private information exfiltrated to the attacker; best trigger of the attack class on the held-out test set
  layout_same_as_defence: 'yes'
  measured_by: gemini_arxiv2025
  page: 21
  attacker_budget: Beam Search triggers optimised against this model
  decoding: not reported
  pages:
    judge: 14
- id: ev_t1_email_g20_tap
  defence_id: none:Gemini 2.0
  benchmark: Email function, non-JSON trigger, passport number
  attacker_type: adaptive
  attacker_access: black_box
  judge: fraction of simulated private information exfiltrated to the attacker; best trigger of the attack class on the held-out test set
  layout_same_as_defence: 'yes'
  measured_by: gemini_arxiv2025
  page: 21
  attacker_budget: TAP triggers optimised against this model
  decoding: not reported
  pages:
    judge: 14
  baseline_kind: off_the_shelf
- id: ev_t1_email_g20_ac
  defence_id: none:Gemini 2.0
  benchmark: Email function, non-JSON trigger, passport number
  attacker_type: adaptive
  attacker_access: black_box
  judge: fraction of simulated private information exfiltrated to the attacker; best trigger of the attack class on the held-out test set
  layout_same_as_defence: 'yes'
  measured_by: gemini_arxiv2025
  page: 21
  attacker_budget: Actor-Critic triggers optimised against this model
  decoding: not reported
  pages:
    judge: 14
  baseline_kind: off_the_shelf
- id: ev_t1_email_g20_bs
  defence_id: none:Gemini 2.0
  benchmark: Email function, non-JSON trigger, passport number
  attacker_type: adaptive
  attacker_access: black_box
  judge: fraction of simulated private information exfiltrated to the attacker; best trigger of the attack class on the held-out test set
  layout_same_as_defence: 'yes'
  measured_by: gemini_arxiv2025
  page: 21
  attacker_budget: Beam Search triggers optimised against this model
  decoding: not reported
  pages:
    judge: 14
  baseline_kind: off_the_shelf
- id: ev_t1_cal_g25_tap
  defence_id: aft_g25
  benchmark: Calendar Event function, non-JSON trigger, passport number
  attacker_type: adaptive
  attacker_access: black_box
  judge: fraction of simulated private information exfiltrated to the attacker; best trigger of the attack class on the held-out test set
  layout_same_as_defence: 'yes'
  measured_by: gemini_arxiv2025
  page: 21
  attacker_budget: TAP triggers optimised against this model
  decoding: not reported
  pages:
    judge: 14
- id: ev_t1_cal_g25_ac
  defence_id: aft_g25
  benchmark: Calendar Event function, non-JSON trigger, passport number
  attacker_type: adaptive
  attacker_access: black_box
  judge: fraction of simulated private information exfiltrated to the attacker; best trigger of the attack class on the held-out test set
  layout_same_as_defence: 'yes'
  measured_by: gemini_arxiv2025
  page: 21
  attacker_budget: Actor-Critic triggers optimised against this model
  decoding: not reported
  pages:
    judge: 14
- id: ev_t1_cal_g25_bs
  defence_id: aft_g25
  benchmark: Calendar Event function, non-JSON trigger, passport number
  attacker_type: adaptive
  attacker_access: black_box
  judge: fraction of simulated private information exfiltrated to the attacker; best trigger of the attack class on the held-out test set
  layout_same_as_defence: 'yes'
  measured_by: gemini_arxiv2025
  page: 21
  attacker_budget: Beam Search triggers optimised against this model
  decoding: not reported
  pages:
    judge: 14
- id: ev_t1_cal_g20_tap
  defence_id: none:Gemini 2.0
  benchmark: Calendar Event function, non-JSON trigger, passport number
  attacker_type: adaptive
  attacker_access: black_box
  judge: fraction of simulated private information exfiltrated to the attacker; best trigger of the attack class on the held-out test set
  layout_same_as_defence: 'yes'
  measured_by: gemini_arxiv2025
  page: 21
  attacker_budget: TAP triggers optimised against this model
  decoding: not reported
  pages:
    judge: 14
  baseline_kind: off_the_shelf
- id: ev_t1_cal_g20_ac
  defence_id: none:Gemini 2.0
  benchmark: Calendar Event function, non-JSON trigger, passport number
  attacker_type: adaptive
  attacker_access: black_box
  judge: fraction of simulated private information exfiltrated to the attacker; best trigger of the attack class on the held-out test set
  layout_same_as_defence: 'yes'
  measured_by: gemini_arxiv2025
  page: 21
  attacker_budget: Actor-Critic triggers optimised against this model
  decoding: not reported
  pages:
    judge: 14
  baseline_kind: off_the_shelf
- id: ev_t1_cal_g20_bs
  defence_id: none:Gemini 2.0
  benchmark: Calendar Event function, non-JSON trigger, passport number
  attacker_type: adaptive
  attacker_access: black_box
  judge: fraction of simulated private information exfiltrated to the attacker; best trigger of the attack class on the held-out test set
  layout_same_as_defence: 'yes'
  measured_by: gemini_arxiv2025
  page: 21
  attacker_budget: Beam Search triggers optimised against this model
  decoding: not reported
  pages:
    judge: 14
  baseline_kind: off_the_shelf
- id: ev_na_g20
  defence_id: none:Gemini 2.0
  benchmark: 1,799 prompts with triggers optimised against Gemini 2.0, five tools not used in adversarial training, all three private data types
  attacker_type: static
  attacker_access: black_box
  judge: fraction of simulated private information exfiltrated to the attacker; best trigger of the attack class on the held-out test set
  layout_same_as_defence: 'yes'
  measured_by: gemini_arxiv2025
  page: 21
  attacker_budget: triggers optimised against Gemini 2.0
  sample_size: 1,799
  decoding: not reported
  pages:
    judge: 14
  baseline_kind: off_the_shelf
- id: ev_na_g25
  defence_id: aft_g25
  benchmark: 1,799 prompts with triggers optimised against Gemini 2.0, five tools not used in adversarial training, all three private data types
  attacker_type: static
  attacker_access: black_box
  judge: fraction of simulated private information exfiltrated to the attacker; best trigger of the attack class on the held-out test set
  layout_same_as_defence: 'yes'
  measured_by: gemini_arxiv2025
  page: 21
  attacker_budget: triggers optimised against Gemini 2.0
  sample_size: 1,799
  decoding: not reported
  pages:
    judge: 14
- id: ev_combo_warn_g25
  defence_id: warn_g25
  benchmark: Calendar Event scenario
  attacker_type: adaptive
  attacker_access: black_box
  judge: fraction of simulated private information exfiltrated to the attacker; best trigger of the attack class on the held-out test set
  layout_same_as_defence: 'yes'
  measured_by: gemini_arxiv2025
  page: 22
  attacker_budget: TAP
  decoding: not reported
  pages:
    judge: 14
results:
- id: r_t3_icl_g20_ac
  evaluation_id: ev_t3_icl_g20_ac
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Actor Critic
  value: '0.462'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 36
  location: Table 3
- id: r_t3_icl_g20_bs
  evaluation_id: ev_t3_icl_g20_bs
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Beam Search
  value: '0.344'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 36
  location: Table 3
- id: r_t3_icl_g20_tap
  evaluation_id: ev_t3_icl_g20_tap
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, TAP
  value: '0.676'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 36
  location: Table 3
- id: r_t3_spot_g20_ac
  evaluation_id: ev_t3_spot_g20_ac
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Actor Critic
  value: '0.180'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 36
  location: Table 3
- id: r_t3_spot_g20_bs
  evaluation_id: ev_t3_spot_g20_bs
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Beam Search
  value: '0.130'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 36
  location: Table 3
- id: r_t3_spot_g20_tap
  evaluation_id: ev_t3_spot_g20_tap
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, TAP
  value: '0.752'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 36
  location: Table 3
- id: r_t3_warn_g20_ac
  evaluation_id: ev_t3_warn_g20_ac
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Actor Critic
  value: '0.058'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 36
  location: Table 3
- id: r_t3_warn_g20_bs
  evaluation_id: ev_t3_warn_g20_bs
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Beam Search
  value: '0.002'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 36
  location: Table 3
- id: r_t3_warn_g20_tap
  evaluation_id: ev_t3_warn_g20_tap
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, TAP
  value: '0.012'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 36
  location: Table 3
- id: r_t4_icl_g20_ac
  evaluation_id: ev_t4_icl_g20_ac
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Actor Critic
  value: '0.598'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 36
  location: Table 4
- id: r_t4_icl_g20_bs
  evaluation_id: ev_t4_icl_g20_bs
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Beam Search
  value: '0.666'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 36
  location: Table 4
- id: r_t4_icl_g20_tap
  evaluation_id: ev_t4_icl_g20_tap
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, TAP
  value: '0.984'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 36
  location: Table 4
- id: r_t4_spot_g20_ac
  evaluation_id: ev_t4_spot_g20_ac
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Actor Critic
  value: '0.824'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 36
  location: Table 4
- id: r_t4_spot_g20_bs
  evaluation_id: ev_t4_spot_g20_bs
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Beam Search
  value: '0.648'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 36
  location: Table 4
- id: r_t4_spot_g20_tap
  evaluation_id: ev_t4_spot_g20_tap
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, TAP
  value: '0.822'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 36
  location: Table 4
- id: r_t4_warn_g20_ac
  evaluation_id: ev_t4_warn_g20_ac
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Actor Critic
  value: '0.084'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 36
  location: Table 4
- id: r_t4_warn_g20_bs
  evaluation_id: ev_t4_warn_g20_bs
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Beam Search
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 36
  location: Table 4
- id: r_t4_warn_g20_tap
  evaluation_id: ev_t4_warn_g20_tap
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, TAP
  value: '0.108'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 36
  location: Table 4
- id: r_t1_email_g25_tap
  evaluation_id: ev_t1_email_g25_tap
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), TAP
  value: '53.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 1
- id: r_t1_email_g25_ac
  evaluation_id: ev_t1_email_g25_ac
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), Actor-Critic
  value: '40.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 1
- id: r_t1_email_g25_bs
  evaluation_id: ev_t1_email_g25_bs
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), Beam Search
  value: '4.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 1
- id: r_t1_email_g20_tap
  evaluation_id: ev_t1_email_g20_tap
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), TAP
  value: '99.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 1
- id: r_t1_email_g20_ac
  evaluation_id: ev_t1_email_g20_ac
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), Actor-Critic
  value: '66.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 1
- id: r_t1_email_g20_bs
  evaluation_id: ev_t1_email_g20_bs
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), Beam Search
  value: '74.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 1
- id: r_t1_cal_g25_tap
  evaluation_id: ev_t1_cal_g25_tap
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), TAP
  value: '94.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 1
- id: r_t1_cal_g25_ac
  evaluation_id: ev_t1_cal_g25_ac
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), Actor-Critic
  value: '59.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 1
- id: r_t1_cal_g25_bs
  evaluation_id: ev_t1_cal_g25_bs
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), Beam Search
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 1
- id: r_t1_cal_g20_tap
  evaluation_id: ev_t1_cal_g20_tap
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), TAP
  value: '100'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 1
- id: r_t1_cal_g20_ac
  evaluation_id: ev_t1_cal_g20_ac
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), Actor-Critic
  value: '96.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 1
- id: r_t1_cal_g20_bs
  evaluation_id: ev_t1_cal_g20_bs
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), Beam Search
  value: '98.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 1
- id: r_na_g20
  evaluation_id: ev_na_g20
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '92'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 21
  location: Section 9.1
- id: r_na_g25
  evaluation_id: ev_na_g25
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '18'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 21
  location: Section 9.1
- id: r_combo_warn_g25
  evaluation_id: ev_combo_warn_g25
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '6.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 22
  location: Section 9.2
comparisons:
- id: c_t1_email_tap
  left: ev_t1_email_g25_tap
  right: ev_t1_email_g20_tap
  purpose: W
  c1_base: unmatched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the adversarially trained Gemini 2.5 against Gemini 2.0 (a different and newer model)
  page: 21
- id: c_t1_email_ac
  left: ev_t1_email_g25_ac
  right: ev_t1_email_g20_ac
  purpose: W
  c1_base: unmatched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the adversarially trained Gemini 2.5 against Gemini 2.0 (a different and newer model)
  page: 21
- id: c_t1_email_bs
  left: ev_t1_email_g25_bs
  right: ev_t1_email_g20_bs
  purpose: W
  c1_base: unmatched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the adversarially trained Gemini 2.5 against Gemini 2.0 (a different and newer model)
  page: 21
- id: c_t1_cal_tap
  left: ev_t1_cal_g25_tap
  right: ev_t1_cal_g20_tap
  purpose: W
  c1_base: unmatched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the adversarially trained Gemini 2.5 against Gemini 2.0 (a different and newer model)
  page: 21
- id: c_t1_cal_ac
  left: ev_t1_cal_g25_ac
  right: ev_t1_cal_g20_ac
  purpose: W
  c1_base: unmatched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the adversarially trained Gemini 2.5 against Gemini 2.0 (a different and newer model)
  page: 21
- id: c_t1_cal_bs
  left: ev_t1_cal_g25_bs
  right: ev_t1_cal_g20_bs
  purpose: W
  c1_base: unmatched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the adversarially trained Gemini 2.5 against Gemini 2.0 (a different and newer model)
  page: 21
- id: c_na
  left: ev_na_g25
  right: ev_na_g20
  purpose: W
  c1_base: unmatched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the adversarially trained Gemini 2.5 against Gemini 2.0 on triggers built against Gemini 2.0
  page: 21
- id: c_combo_warn_on_g25
  left: ev_combo_warn_g25
  right: ev_t1_cal_g25_tap
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the Warning defence added to Gemini 2.5 (against Gemini 2.5 alone, Table 1)
  page: 22
- id: c_combo_g25_vs_g20_warn
  left: ev_combo_warn_g25
  right: ev_t4_warn_g20_tap
  purpose: W
  c1_base: unmatched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: Warning on Gemini 2.5 against Warning on Gemini 2.0 (Table 4)
  page: 22
claims:
- id: k1
  text: ongoing adaptive evaluations directly help make Gemini more resilient
  location: abstract
  anchor: these ongoing evaluations directly help make Gemini more resilient against manipulation
  jobs:
  - R1
  supported_by:
  - c_t1_email_tap
  - c_t1_email_ac
  - c_t1_email_bs
  - c_t1_cal_tap
  - c_t1_cal_ac
  - c_t1_cal_bs
  - c_na
  page: 1
  contradicted_by:
  - r_t1_cal_g25_tap
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Tables 1, 3 and 4; Sections 9.1 and 9.2
    general_utility: Section 9.1 (leaderboard score only); Appendix H not coded
schema_gaps:
- where: comparisons c_combo_*
  note: Section 9.2 compares Warning on Gemini 2.5 (6.2%) with Warning on Gemini 2.0 against adaptive TAP in the Calendar Event scenario (10.8%, the TAP cell of Table 4) and with Gemini 2.5 alone (94.6%, Table 1); the scenario's
    data type and trigger format for 6.2% are not stated

`ckptgcg_haips2026` (2505.15738)

In [ ]:
%%writefile reports/records/ckptgcg_haips2026.yaml
report:
  id: ckptgcg_haips2026
  slug: '2505.15738'
  title: 'Auditing Prompt Injection Defenses Before They Reach Users: What Strong Adversaries Reveal'
  first_author: Xiaoxue Yang
  venue: HAIPS workshop at COLM 2026
  peer_reviewed: 'yes'
  arxiv: '2505.15738'
  version_read: arXiv v3 (marked as a workshop paper at HAIPS@COLM 2026)
  first_date: '2025-05-21'
  read: F
  role: re_measures
  code_release: not checked
  extraction_scope: 'Sections 4 to 6.1.1 and Appendix C: Table 4 (all rows and attacks, AlpacaFarm, 50 samples); the abstract claim (universal suffixes in Tables 5 to 7, SecAlign++ transfer, BEAST, Tables 8 to 11 and the jailbreak
    results not coded)'
defences:
- id: struq_l3
  name: StruQ (replicated by the authors)
  origin: other_report
  base_checkpoint: Llama-3-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  layout: as in the original method; not detailed in this report
  anchor: 'We audit two state-of-the-art fine-tuning-based defenses: StruQ'
- id: secalign_l3
  name: SecAlign (replicated by the authors)
  origin: other_report
  base_checkpoint: Llama-3-8B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  layout: as in the original method; not detailed in this report
  anchor: 'We audit two state-of-the-art fine-tuning-based defenses: StruQ'
- id: struq_m7
  name: StruQ (replicated by the authors)
  origin: other_report
  base_checkpoint: Mistral-7B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  layout: as in the original method; not detailed in this report
  anchor: 'We audit two state-of-the-art fine-tuning-based defenses: StruQ'
- id: secalign_m7
  name: SecAlign (replicated by the authors)
  origin: other_report
  base_checkpoint: Mistral-7B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  layout: as in the original method; not detailed in this report
  anchor: 'We audit two state-of-the-art fine-tuning-based defenses: StruQ'
- id: struq_q2
  name: StruQ (replicated by the authors)
  origin: other_report
  base_checkpoint: Qwen2-1.5B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  layout: as in the original method; not detailed in this report
  anchor: 'We audit two state-of-the-art fine-tuning-based defenses: StruQ'
- id: secalign_q2
  name: SecAlign (replicated by the authors)
  origin: other_report
  base_checkpoint: Qwen2-1.5B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  layout: as in the original method; not detailed in this report
  anchor: 'We audit two state-of-the-art fine-tuning-based defenses: StruQ'
evaluations:
- id: ev_l3_none_gcg500
  defence_id: none:Llama-3-8B-Instruct
  benchmark: AlpacaFarm samples with a data part, Print exactly 'Hacked'! plus the suffix at the end of the data
  attacker_type: optimisation
  attacker_access: white_box
  judge: response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: ckptgcg_haips2026
  page: 20
  attacker_budget: Standard GCG on the final model, T = 500 steps
  sample_size: '50'
  injection_position: end
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
  baseline_kind: off_the_shelf
- id: ev_l3_none_gcgbud
  defence_id: none:Llama-3-8B-Instruct
  benchmark: AlpacaFarm samples with a data part, Print exactly 'Hacked'! plus the suffix at the end of the data
  attacker_type: optimisation
  attacker_access: white_box
  judge: response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: ckptgcg_haips2026
  page: 20
  attacker_budget: Standard GCG on the final model, Checkpoint-GCG budget
  sample_size: '50'
  injection_position: end
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
  baseline_kind: off_the_shelf
- id: ev_l3_none_ckpt
  defence_id: none:Llama-3-8B-Instruct
  benchmark: AlpacaFarm samples with a data part, Print exactly 'Hacked'! plus the suffix at the end of the data
  attacker_type: optimisation
  attacker_access: white_box
  judge: response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: ckptgcg_haips2026
  page: 20
  attacker_budget: 'Checkpoint-GCG: GCG run across the defence''s intermediate training checkpoints, each initialised from the previous suffix'
  sample_size: '50'
  injection_position: end
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
  baseline_kind: off_the_shelf
- id: ev_m7_none_gcg500
  defence_id: none:Mistral-7B-Instruct
  benchmark: AlpacaFarm samples with a data part, Print exactly 'Hacked'! plus the suffix at the end of the data
  attacker_type: optimisation
  attacker_access: white_box
  judge: response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: ckptgcg_haips2026
  page: 20
  attacker_budget: Standard GCG on the final model, T = 500 steps
  sample_size: '50'
  injection_position: end
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
  baseline_kind: off_the_shelf
- id: ev_m7_none_gcgbud
  defence_id: none:Mistral-7B-Instruct
  benchmark: AlpacaFarm samples with a data part, Print exactly 'Hacked'! plus the suffix at the end of the data
  attacker_type: optimisation
  attacker_access: white_box
  judge: response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: ckptgcg_haips2026
  page: 20
  attacker_budget: Standard GCG on the final model, Checkpoint-GCG budget
  sample_size: '50'
  injection_position: end
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
  baseline_kind: off_the_shelf
- id: ev_m7_none_ckpt
  defence_id: none:Mistral-7B-Instruct
  benchmark: AlpacaFarm samples with a data part, Print exactly 'Hacked'! plus the suffix at the end of the data
  attacker_type: optimisation
  attacker_access: white_box
  judge: response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: ckptgcg_haips2026
  page: 20
  attacker_budget: 'Checkpoint-GCG: GCG run across the defence''s intermediate training checkpoints, each initialised from the previous suffix'
  sample_size: '50'
  injection_position: end
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
  baseline_kind: off_the_shelf
- id: ev_q2_none_gcg500
  defence_id: none:Qwen2-1.5B-Instruct
  benchmark: AlpacaFarm samples with a data part, Print exactly 'Hacked'! plus the suffix at the end of the data
  attacker_type: optimisation
  attacker_access: white_box
  judge: response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: ckptgcg_haips2026
  page: 20
  attacker_budget: Standard GCG on the final model, T = 500 steps
  sample_size: '50'
  injection_position: end
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
  baseline_kind: off_the_shelf
- id: ev_q2_none_gcgbud
  defence_id: none:Qwen2-1.5B-Instruct
  benchmark: AlpacaFarm samples with a data part, Print exactly 'Hacked'! plus the suffix at the end of the data
  attacker_type: optimisation
  attacker_access: white_box
  judge: response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: ckptgcg_haips2026
  page: 20
  attacker_budget: Standard GCG on the final model, Checkpoint-GCG budget
  sample_size: '50'
  injection_position: end
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
  baseline_kind: off_the_shelf
- id: ev_q2_none_ckpt
  defence_id: none:Qwen2-1.5B-Instruct
  benchmark: AlpacaFarm samples with a data part, Print exactly 'Hacked'! plus the suffix at the end of the data
  attacker_type: optimisation
  attacker_access: white_box
  judge: response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: ckptgcg_haips2026
  page: 20
  attacker_budget: 'Checkpoint-GCG: GCG run across the defence''s intermediate training checkpoints, each initialised from the previous suffix'
  sample_size: '50'
  injection_position: end
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
  baseline_kind: off_the_shelf
- id: ev_l3_struq_gcg500
  defence_id: struq_l3
  benchmark: AlpacaFarm samples with a data part, Print exactly 'Hacked'! plus the suffix at the end of the data
  attacker_type: optimisation
  attacker_access: white_box
  judge: response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: ckptgcg_haips2026
  page: 20
  attacker_budget: Standard GCG on the final model, T = 500 steps
  sample_size: '50'
  injection_position: end
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_l3_struq_gcgbud
  defence_id: struq_l3
  benchmark: AlpacaFarm samples with a data part, Print exactly 'Hacked'! plus the suffix at the end of the data
  attacker_type: optimisation
  attacker_access: white_box
  judge: response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: ckptgcg_haips2026
  page: 20
  attacker_budget: Standard GCG on the final model, Checkpoint-GCG budget
  sample_size: '50'
  injection_position: end
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_l3_struq_ckpt
  defence_id: struq_l3
  benchmark: AlpacaFarm samples with a data part, Print exactly 'Hacked'! plus the suffix at the end of the data
  attacker_type: optimisation
  attacker_access: white_box
  judge: response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: ckptgcg_haips2026
  page: 20
  attacker_budget: 'Checkpoint-GCG: GCG run across the defence''s intermediate training checkpoints, each initialised from the previous suffix'
  sample_size: '50'
  injection_position: end
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_m7_struq_gcg500
  defence_id: struq_m7
  benchmark: AlpacaFarm samples with a data part, Print exactly 'Hacked'! plus the suffix at the end of the data
  attacker_type: optimisation
  attacker_access: white_box
  judge: response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: ckptgcg_haips2026
  page: 20
  attacker_budget: Standard GCG on the final model, T = 500 steps
  sample_size: '50'
  injection_position: end
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_m7_struq_gcgbud
  defence_id: struq_m7
  benchmark: AlpacaFarm samples with a data part, Print exactly 'Hacked'! plus the suffix at the end of the data
  attacker_type: optimisation
  attacker_access: white_box
  judge: response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: ckptgcg_haips2026
  page: 20
  attacker_budget: Standard GCG on the final model, Checkpoint-GCG budget
  sample_size: '50'
  injection_position: end
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_m7_struq_ckpt
  defence_id: struq_m7
  benchmark: AlpacaFarm samples with a data part, Print exactly 'Hacked'! plus the suffix at the end of the data
  attacker_type: optimisation
  attacker_access: white_box
  judge: response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: ckptgcg_haips2026
  page: 20
  attacker_budget: 'Checkpoint-GCG: GCG run across the defence''s intermediate training checkpoints, each initialised from the previous suffix'
  sample_size: '50'
  injection_position: end
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_q2_struq_gcg500
  defence_id: struq_q2
  benchmark: AlpacaFarm samples with a data part, Print exactly 'Hacked'! plus the suffix at the end of the data
  attacker_type: optimisation
  attacker_access: white_box
  judge: response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: ckptgcg_haips2026
  page: 20
  attacker_budget: Standard GCG on the final model, T = 500 steps
  sample_size: '50'
  injection_position: end
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_q2_struq_gcgbud
  defence_id: struq_q2
  benchmark: AlpacaFarm samples with a data part, Print exactly 'Hacked'! plus the suffix at the end of the data
  attacker_type: optimisation
  attacker_access: white_box
  judge: response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: ckptgcg_haips2026
  page: 20
  attacker_budget: Standard GCG on the final model, Checkpoint-GCG budget
  sample_size: '50'
  injection_position: end
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_q2_struq_ckpt
  defence_id: struq_q2
  benchmark: AlpacaFarm samples with a data part, Print exactly 'Hacked'! plus the suffix at the end of the data
  attacker_type: optimisation
  attacker_access: white_box
  judge: response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: ckptgcg_haips2026
  page: 20
  attacker_budget: 'Checkpoint-GCG: GCG run across the defence''s intermediate training checkpoints, each initialised from the previous suffix'
  sample_size: '50'
  injection_position: end
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_l3_secalign_gcg500
  defence_id: secalign_l3
  benchmark: AlpacaFarm samples with a data part, Print exactly 'Hacked'! plus the suffix at the end of the data
  attacker_type: optimisation
  attacker_access: white_box
  judge: response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: ckptgcg_haips2026
  page: 20
  attacker_budget: Standard GCG on the final model, T = 500 steps
  sample_size: '50'
  injection_position: end
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_l3_secalign_gcgbud
  defence_id: secalign_l3
  benchmark: AlpacaFarm samples with a data part, Print exactly 'Hacked'! plus the suffix at the end of the data
  attacker_type: optimisation
  attacker_access: white_box
  judge: response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: ckptgcg_haips2026
  page: 20
  attacker_budget: Standard GCG on the final model, Checkpoint-GCG budget
  sample_size: '50'
  injection_position: end
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_l3_secalign_ckpt
  defence_id: secalign_l3
  benchmark: AlpacaFarm samples with a data part, Print exactly 'Hacked'! plus the suffix at the end of the data
  attacker_type: optimisation
  attacker_access: white_box
  judge: response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: ckptgcg_haips2026
  page: 20
  attacker_budget: 'Checkpoint-GCG: GCG run across the defence''s intermediate training checkpoints, each initialised from the previous suffix'
  sample_size: '50'
  injection_position: end
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_m7_secalign_gcg500
  defence_id: secalign_m7
  benchmark: AlpacaFarm samples with a data part, Print exactly 'Hacked'! plus the suffix at the end of the data
  attacker_type: optimisation
  attacker_access: white_box
  judge: response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: ckptgcg_haips2026
  page: 20
  attacker_budget: Standard GCG on the final model, T = 500 steps
  sample_size: '50'
  injection_position: end
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_m7_secalign_gcgbud
  defence_id: secalign_m7
  benchmark: AlpacaFarm samples with a data part, Print exactly 'Hacked'! plus the suffix at the end of the data
  attacker_type: optimisation
  attacker_access: white_box
  judge: response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: ckptgcg_haips2026
  page: 20
  attacker_budget: Standard GCG on the final model, Checkpoint-GCG budget
  sample_size: '50'
  injection_position: end
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_m7_secalign_ckpt
  defence_id: secalign_m7
  benchmark: AlpacaFarm samples with a data part, Print exactly 'Hacked'! plus the suffix at the end of the data
  attacker_type: optimisation
  attacker_access: white_box
  judge: response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: ckptgcg_haips2026
  page: 20
  attacker_budget: 'Checkpoint-GCG: GCG run across the defence''s intermediate training checkpoints, each initialised from the previous suffix'
  sample_size: '50'
  injection_position: end
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_q2_secalign_gcg500
  defence_id: secalign_q2
  benchmark: AlpacaFarm samples with a data part, Print exactly 'Hacked'! plus the suffix at the end of the data
  attacker_type: optimisation
  attacker_access: white_box
  judge: response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: ckptgcg_haips2026
  page: 20
  attacker_budget: Standard GCG on the final model, T = 500 steps
  sample_size: '50'
  injection_position: end
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_q2_secalign_gcgbud
  defence_id: secalign_q2
  benchmark: AlpacaFarm samples with a data part, Print exactly 'Hacked'! plus the suffix at the end of the data
  attacker_type: optimisation
  attacker_access: white_box
  judge: response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: ckptgcg_haips2026
  page: 20
  attacker_budget: Standard GCG on the final model, Checkpoint-GCG budget
  sample_size: '50'
  injection_position: end
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
- id: ev_q2_secalign_ckpt
  defence_id: secalign_q2
  benchmark: AlpacaFarm samples with a data part, Print exactly 'Hacked'! plus the suffix at the end of the data
  attacker_type: optimisation
  attacker_access: white_box
  judge: response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: ckptgcg_haips2026
  page: 20
  attacker_budget: 'Checkpoint-GCG: GCG run across the defence''s intermediate training checkpoints, each initialised from the previous suffix'
  sample_size: '50'
  injection_position: end
  decoding: not reported
  pages:
    benchmark: 6
    judge: 6
results:
- id: r_l3_none_gcg500
  evaluation_id: ev_l3_none_gcg500
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), Standard GCG on the final model, T = 500 steps
  value: '100'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 4
  spread: 95% Wilson interval drawn in Figure 2
- id: r_l3_none_gcgbud
  evaluation_id: ev_l3_none_gcgbud
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), Standard GCG on the final model, Checkpoint-GCG budget
  value: '100'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 4
  spread: 95% Wilson interval drawn in Figure 2
- id: r_l3_none_ckpt
  evaluation_id: ev_l3_none_ckpt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), Checkpoint-GCG
  value: '100'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 4
  spread: 95% Wilson interval drawn in Figure 2
- id: r_m7_none_gcg500
  evaluation_id: ev_m7_none_gcg500
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), Standard GCG on the final model, T = 500 steps
  value: '100'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 4
  spread: 95% Wilson interval drawn in Figure 2
- id: r_m7_none_gcgbud
  evaluation_id: ev_m7_none_gcgbud
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), Standard GCG on the final model, Checkpoint-GCG budget
  value: '100'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 4
  spread: 95% Wilson interval drawn in Figure 2
- id: r_m7_none_ckpt
  evaluation_id: ev_m7_none_ckpt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), Checkpoint-GCG
  value: '100'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 4
  spread: 95% Wilson interval drawn in Figure 2
- id: r_q2_none_gcg500
  evaluation_id: ev_q2_none_gcg500
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), Standard GCG on the final model, T = 500 steps
  value: '98'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 4
  spread: 95% Wilson interval drawn in Figure 2
- id: r_q2_none_gcgbud
  evaluation_id: ev_q2_none_gcgbud
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), Standard GCG on the final model, Checkpoint-GCG budget
  value: '98'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 4
  spread: 95% Wilson interval drawn in Figure 2
- id: r_q2_none_ckpt
  evaluation_id: ev_q2_none_ckpt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), Checkpoint-GCG
  value: '98'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 4
  spread: 95% Wilson interval drawn in Figure 2
- id: r_l3_struq_gcg500
  evaluation_id: ev_l3_struq_gcg500
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), Standard GCG on the final model, T = 500 steps
  value: '26'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 4
  spread: 95% Wilson interval drawn in Figure 2
- id: r_l3_struq_gcgbud
  evaluation_id: ev_l3_struq_gcgbud
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), Standard GCG on the final model, Checkpoint-GCG budget
  value: '50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 4
  spread: 95% Wilson interval drawn in Figure 2
- id: r_l3_struq_ckpt
  evaluation_id: ev_l3_struq_ckpt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), Checkpoint-GCG
  value: '100'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 4
  spread: 95% Wilson interval drawn in Figure 2
- id: r_m7_struq_gcg500
  evaluation_id: ev_m7_struq_gcg500
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), Standard GCG on the final model, T = 500 steps
  value: '88'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 4
  spread: 95% Wilson interval drawn in Figure 2
- id: r_m7_struq_gcgbud
  evaluation_id: ev_m7_struq_gcgbud
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), Standard GCG on the final model, Checkpoint-GCG budget
  value: '90'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 4
  spread: 95% Wilson interval drawn in Figure 2
- id: r_m7_struq_ckpt
  evaluation_id: ev_m7_struq_ckpt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), Checkpoint-GCG
  value: '100'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 4
  spread: 95% Wilson interval drawn in Figure 2
- id: r_q2_struq_gcg500
  evaluation_id: ev_q2_struq_gcg500
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), Standard GCG on the final model, T = 500 steps
  value: '48'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 4
  spread: 95% Wilson interval drawn in Figure 2
- id: r_q2_struq_gcgbud
  evaluation_id: ev_q2_struq_gcgbud
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), Standard GCG on the final model, Checkpoint-GCG budget
  value: '52'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 4
  spread: 95% Wilson interval drawn in Figure 2
- id: r_q2_struq_ckpt
  evaluation_id: ev_q2_struq_ckpt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), Checkpoint-GCG
  value: '90'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 4
  spread: 95% Wilson interval drawn in Figure 2
- id: r_l3_secalign_gcg500
  evaluation_id: ev_l3_secalign_gcg500
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), Standard GCG on the final model, T = 500 steps
  value: '6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 4
  spread: 95% Wilson interval drawn in Figure 2
- id: r_l3_secalign_gcgbud
  evaluation_id: ev_l3_secalign_gcgbud
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), Standard GCG on the final model, Checkpoint-GCG budget
  value: '12'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 4
  spread: 95% Wilson interval drawn in Figure 2
- id: r_l3_secalign_ckpt
  evaluation_id: ev_l3_secalign_ckpt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), Checkpoint-GCG
  value: '88'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 4
  spread: 95% Wilson interval drawn in Figure 2
- id: r_m7_secalign_gcg500
  evaluation_id: ev_m7_secalign_gcg500
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), Standard GCG on the final model, T = 500 steps
  value: '18'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 4
  spread: 95% Wilson interval drawn in Figure 2
- id: r_m7_secalign_gcgbud
  evaluation_id: ev_m7_secalign_gcgbud
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), Standard GCG on the final model, Checkpoint-GCG budget
  value: '22'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 4
  spread: 95% Wilson interval drawn in Figure 2
- id: r_m7_secalign_ckpt
  evaluation_id: ev_m7_secalign_ckpt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), Checkpoint-GCG
  value: '96'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 4
  spread: 95% Wilson interval drawn in Figure 2
- id: r_q2_secalign_gcg500
  evaluation_id: ev_q2_secalign_gcg500
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), Standard GCG on the final model, T = 500 steps
  value: '32'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 4
  spread: 95% Wilson interval drawn in Figure 2
- id: r_q2_secalign_gcgbud
  evaluation_id: ev_q2_secalign_gcgbud
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), Standard GCG on the final model, Checkpoint-GCG budget
  value: '32'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 4
  spread: 95% Wilson interval drawn in Figure 2
- id: r_q2_secalign_ckpt
  evaluation_id: ev_q2_secalign_ckpt
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%), Checkpoint-GCG
  value: '84'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 4
  spread: 95% Wilson interval drawn in Figure 2
comparisons:
- id: c_l3_struq_gcg500
  left: ev_l3_struq_gcg500
  right: ev_l3_none_gcg500
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 20
- id: c_l3_struq_gcgbud
  left: ev_l3_struq_gcgbud
  right: ev_l3_none_gcgbud
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 20
- id: c_l3_struq_ckpt
  left: ev_l3_struq_ckpt
  right: ev_l3_none_ckpt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 20
- id: c_l3_struq_ckpt_vs_gcg
  left: ev_l3_struq_ckpt
  right: ev_l3_struq_gcg500
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: manipulated
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the attack: Checkpoint-GCG against standard GCG with 500 steps, same defended model'
  page: 20
- id: c_l3_secalign_gcg500
  left: ev_l3_secalign_gcg500
  right: ev_l3_none_gcg500
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 20
- id: c_l3_secalign_gcgbud
  left: ev_l3_secalign_gcgbud
  right: ev_l3_none_gcgbud
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 20
- id: c_l3_secalign_ckpt
  left: ev_l3_secalign_ckpt
  right: ev_l3_none_ckpt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 20
- id: c_l3_secalign_ckpt_vs_gcg
  left: ev_l3_secalign_ckpt
  right: ev_l3_secalign_gcg500
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: manipulated
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the attack: Checkpoint-GCG against standard GCG with 500 steps, same defended model'
  page: 20
- id: c_m7_struq_gcg500
  left: ev_m7_struq_gcg500
  right: ev_m7_none_gcg500
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 20
- id: c_m7_struq_gcgbud
  left: ev_m7_struq_gcgbud
  right: ev_m7_none_gcgbud
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 20
- id: c_m7_struq_ckpt
  left: ev_m7_struq_ckpt
  right: ev_m7_none_ckpt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 20
- id: c_m7_struq_ckpt_vs_gcg
  left: ev_m7_struq_ckpt
  right: ev_m7_struq_gcg500
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: manipulated
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the attack: Checkpoint-GCG against standard GCG with 500 steps, same defended model'
  page: 20
- id: c_m7_secalign_gcg500
  left: ev_m7_secalign_gcg500
  right: ev_m7_none_gcg500
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 20
- id: c_m7_secalign_gcgbud
  left: ev_m7_secalign_gcgbud
  right: ev_m7_none_gcgbud
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 20
- id: c_m7_secalign_ckpt
  left: ev_m7_secalign_ckpt
  right: ev_m7_none_ckpt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 20
- id: c_m7_secalign_ckpt_vs_gcg
  left: ev_m7_secalign_ckpt
  right: ev_m7_secalign_gcg500
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: manipulated
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the attack: Checkpoint-GCG against standard GCG with 500 steps, same defended model'
  page: 20
- id: c_q2_struq_gcg500
  left: ev_q2_struq_gcg500
  right: ev_q2_none_gcg500
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 20
- id: c_q2_struq_gcgbud
  left: ev_q2_struq_gcgbud
  right: ev_q2_none_gcgbud
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 20
- id: c_q2_struq_ckpt
  left: ev_q2_struq_ckpt
  right: ev_q2_none_ckpt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 20
- id: c_q2_struq_ckpt_vs_gcg
  left: ev_q2_struq_ckpt
  right: ev_q2_struq_gcg500
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: manipulated
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the attack: Checkpoint-GCG against standard GCG with 500 steps, same defended model'
  page: 20
- id: c_q2_secalign_gcg500
  left: ev_q2_secalign_gcg500
  right: ev_q2_none_gcg500
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 20
- id: c_q2_secalign_gcgbud
  left: ev_q2_secalign_gcgbud
  right: ev_q2_none_gcgbud
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 20
- id: c_q2_secalign_ckpt
  left: ev_q2_secalign_ckpt
  right: ev_q2_none_ckpt
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 20
- id: c_q2_secalign_ckpt_vs_gcg
  left: ev_q2_secalign_ckpt
  right: ev_q2_secalign_gcg500
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: manipulated
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: 'the attack: Checkpoint-GCG against standard GCG with 500 steps, same defended model'
  page: 20
claims:
- id: k1
  text: checkpoint-based attacks raise attack success against state-of-the-art defences from near zero to as high as 96%
  location: abstract
  anchor: raise attack success rates against state-of-the-art defenses from near zero
  jobs:
  - R1
  supported_by:
  - c_l3_struq_ckpt_vs_gcg
  - c_l3_secalign_ckpt_vs_gcg
  - c_m7_struq_ckpt_vs_gcg
  - c_m7_secalign_ckpt_vs_gcg
  - c_q2_struq_ckpt_vs_gcg
  - c_q2_secalign_ckpt_vs_gcg
  page: 1
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 4 (AlpacaFarm, 50 samples)

`air_arxiv2025` (2505.18907)

In [ ]:
%%writefile reports/records/air_arxiv2025.yaml
report:
  id: air_arxiv2025
  slug: '2505.18907'
  title: Stronger Enforcement of Instruction Hierarchy via Augmented Intermediate Representations
  first_author: Sanjay Kariyappa
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2505.18907'
  version_read: arXiv v2
  first_date: '2025-05-25'
  read: F
  role: proposes
  code_release: none found in the paper
  extraction_scope: 'Sections 4 to 6.1 and Appendix B.1: Table 1, the Completion, Escape Sep. and GCG rows for all three models and all seven columns; the abstract claim (Naive and Ignore rows, which training saw, Figures
    5 to 8 and the SEP results not coded)'
defences:
- id: delim_sft_l32
  name: Delimiters (trainable [INST] and [INPT] tokens), SFT
  origin: other_report
  base_checkpoint: Llama-3.2-3B (pretrained, instruction-tuned on cleaned Alpaca by the authors)
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  components:
  - Delimiters (trainable [INST] and [INPT] tokens)
  - 'adversarial SFT on Alpaca: half the data-bearing samples get a Naive or Ignore injection of another sample''s instruction'
  layout: 'chat template: system and user instruction at privilege P0, data segment P1, response P2'
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: 'Delimiters [24, 3]: We use two trainable special tokens'
  pages:
    components: 12
- id: ise_sft_l32
  name: ISE (trainable segment embeddings at the input layer), SFT
  origin: other_report
  base_checkpoint: Llama-3.2-3B (pretrained, instruction-tuned on cleaned Alpaca by the authors)
  axis1:
  - learned_token_features
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  components:
  - ISE (trainable segment embeddings at the input layer)
  - 'adversarial SFT on Alpaca: half the data-bearing samples get a Naive or Ignore injection of another sample''s instruction'
  layout: 'chat template: system and user instruction at privilege P0, data segment P1, response P2'
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: 'Instructional Segment Embedding (ISE) [25]: This method adds'
  pages:
    components: 12
- id: air_sft_l32
  name: AIR (trainable privilege embeddings added in every decoder block), SFT
  origin: this_report
  base_checkpoint: Llama-3.2-3B (pretrained, instruction-tuned on cleaned Alpaca by the authors)
  axis1:
  - learned_token_features
  - internal_intervention
  axis2:
  - sft
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  components:
  - AIR (trainable privilege embeddings added in every decoder block)
  - 'adversarial SFT on Alpaca: half the data-bearing samples get a Naive or Ignore injection of another sample''s instruction'
  layout: 'chat template: system and user instruction at privilege P0, data segment P1, response P2'
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: AIR directly injects the IH signals
  pages:
    components: 12
    isolation: 8
- id: delim_dpo_l32
  name: Delimiters (trainable [INST] and [INPT] tokens), DPO
  origin: other_report
  base_checkpoint: Llama-3.2-3B (pretrained, instruction-tuned on cleaned Alpaca by the authors)
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  components:
  - Delimiters (trainable [INST] and [INPT] tokens)
  - 'DPO with LoRA: the Alpaca response preferred over the injected instruction''s own response'
  layout: 'chat template: system and user instruction at privilege P0, data segment P1, response P2'
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: 'Delimiters [24, 3]: We use two trainable special tokens'
  pages:
    components: 12
- id: ise_dpo_l32
  name: ISE (trainable segment embeddings at the input layer), DPO
  origin: other_report
  base_checkpoint: Llama-3.2-3B (pretrained, instruction-tuned on cleaned Alpaca by the authors)
  axis1:
  - learned_token_features
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  components:
  - ISE (trainable segment embeddings at the input layer)
  - 'DPO with LoRA: the Alpaca response preferred over the injected instruction''s own response'
  layout: 'chat template: system and user instruction at privilege P0, data segment P1, response P2'
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: 'Instructional Segment Embedding (ISE) [25]: This method adds'
  pages:
    components: 12
- id: air_dpo_l32
  name: AIR (trainable privilege embeddings added in every decoder block), DPO
  origin: this_report
  base_checkpoint: Llama-3.2-3B (pretrained, instruction-tuned on cleaned Alpaca by the authors)
  axis1:
  - learned_token_features
  - internal_intervention
  axis2:
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  components:
  - AIR (trainable privilege embeddings added in every decoder block)
  - 'DPO with LoRA: the Alpaca response preferred over the injected instruction''s own response'
  layout: 'chat template: system and user instruction at privilege P0, data segment P1, response P2'
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: AIR directly injects the IH signals
  pages:
    components: 12
    isolation: 8
- id: delim_sft_q25
  name: Delimiters (trainable [INST] and [INPT] tokens), SFT
  origin: other_report
  base_checkpoint: Qwen2.5-7B (pretrained, instruction-tuned on cleaned Alpaca by the authors)
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  components:
  - Delimiters (trainable [INST] and [INPT] tokens)
  - 'adversarial SFT on Alpaca: half the data-bearing samples get a Naive or Ignore injection of another sample''s instruction'
  layout: 'chat template: system and user instruction at privilege P0, data segment P1, response P2'
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: 'Delimiters [24, 3]: We use two trainable special tokens'
  pages:
    components: 12
- id: ise_sft_q25
  name: ISE (trainable segment embeddings at the input layer), SFT
  origin: other_report
  base_checkpoint: Qwen2.5-7B (pretrained, instruction-tuned on cleaned Alpaca by the authors)
  axis1:
  - learned_token_features
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  components:
  - ISE (trainable segment embeddings at the input layer)
  - 'adversarial SFT on Alpaca: half the data-bearing samples get a Naive or Ignore injection of another sample''s instruction'
  layout: 'chat template: system and user instruction at privilege P0, data segment P1, response P2'
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: 'Instructional Segment Embedding (ISE) [25]: This method adds'
  pages:
    components: 12
- id: air_sft_q25
  name: AIR (trainable privilege embeddings added in every decoder block), SFT
  origin: this_report
  base_checkpoint: Qwen2.5-7B (pretrained, instruction-tuned on cleaned Alpaca by the authors)
  axis1:
  - learned_token_features
  - internal_intervention
  axis2:
  - sft
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  components:
  - AIR (trainable privilege embeddings added in every decoder block)
  - 'adversarial SFT on Alpaca: half the data-bearing samples get a Naive or Ignore injection of another sample''s instruction'
  layout: 'chat template: system and user instruction at privilege P0, data segment P1, response P2'
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: AIR directly injects the IH signals
  pages:
    components: 12
    isolation: 8
- id: delim_dpo_q25
  name: Delimiters (trainable [INST] and [INPT] tokens), DPO
  origin: other_report
  base_checkpoint: Qwen2.5-7B (pretrained, instruction-tuned on cleaned Alpaca by the authors)
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  components:
  - Delimiters (trainable [INST] and [INPT] tokens)
  - 'DPO with LoRA: the Alpaca response preferred over the injected instruction''s own response'
  layout: 'chat template: system and user instruction at privilege P0, data segment P1, response P2'
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: 'Delimiters [24, 3]: We use two trainable special tokens'
  pages:
    components: 12
- id: ise_dpo_q25
  name: ISE (trainable segment embeddings at the input layer), DPO
  origin: other_report
  base_checkpoint: Qwen2.5-7B (pretrained, instruction-tuned on cleaned Alpaca by the authors)
  axis1:
  - learned_token_features
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  components:
  - ISE (trainable segment embeddings at the input layer)
  - 'DPO with LoRA: the Alpaca response preferred over the injected instruction''s own response'
  layout: 'chat template: system and user instruction at privilege P0, data segment P1, response P2'
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: 'Instructional Segment Embedding (ISE) [25]: This method adds'
  pages:
    components: 12
- id: air_dpo_q25
  name: AIR (trainable privilege embeddings added in every decoder block), DPO
  origin: this_report
  base_checkpoint: Qwen2.5-7B (pretrained, instruction-tuned on cleaned Alpaca by the authors)
  axis1:
  - learned_token_features
  - internal_intervention
  axis2:
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  components:
  - AIR (trainable privilege embeddings added in every decoder block)
  - 'DPO with LoRA: the Alpaca response preferred over the injected instruction''s own response'
  layout: 'chat template: system and user instruction at privilege P0, data segment P1, response P2'
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: AIR directly injects the IH signals
  pages:
    components: 12
    isolation: 8
- id: delim_sft_l31
  name: Delimiters (trainable [INST] and [INPT] tokens), SFT
  origin: other_report
  base_checkpoint: Llama-3.1-8B (pretrained, instruction-tuned on cleaned Alpaca by the authors)
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  components:
  - Delimiters (trainable [INST] and [INPT] tokens)
  - 'adversarial SFT on Alpaca: half the data-bearing samples get a Naive or Ignore injection of another sample''s instruction'
  layout: 'chat template: system and user instruction at privilege P0, data segment P1, response P2'
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: 'Delimiters [24, 3]: We use two trainable special tokens'
  pages:
    components: 12
- id: ise_sft_l31
  name: ISE (trainable segment embeddings at the input layer), SFT
  origin: other_report
  base_checkpoint: Llama-3.1-8B (pretrained, instruction-tuned on cleaned Alpaca by the authors)
  axis1:
  - learned_token_features
  axis2:
  - sft
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  components:
  - ISE (trainable segment embeddings at the input layer)
  - 'adversarial SFT on Alpaca: half the data-bearing samples get a Naive or Ignore injection of another sample''s instruction'
  layout: 'chat template: system and user instruction at privilege P0, data segment P1, response P2'
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: 'Instructional Segment Embedding (ISE) [25]: This method adds'
  pages:
    components: 12
- id: air_sft_l31
  name: AIR (trainable privilege embeddings added in every decoder block), SFT
  origin: this_report
  base_checkpoint: Llama-3.1-8B (pretrained, instruction-tuned on cleaned Alpaca by the authors)
  axis1:
  - learned_token_features
  - internal_intervention
  axis2:
  - sft
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  components:
  - AIR (trainable privilege embeddings added in every decoder block)
  - 'adversarial SFT on Alpaca: half the data-bearing samples get a Naive or Ignore injection of another sample''s instruction'
  layout: 'chat template: system and user instruction at privilege P0, data segment P1, response P2'
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: AIR directly injects the IH signals
  pages:
    components: 12
    isolation: 8
- id: delim_dpo_l31
  name: Delimiters (trainable [INST] and [INPT] tokens), DPO
  origin: other_report
  base_checkpoint: Llama-3.1-8B (pretrained, instruction-tuned on cleaned Alpaca by the authors)
  axis1:
  - delimiters_markers
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  components:
  - Delimiters (trainable [INST] and [INPT] tokens)
  - 'DPO with LoRA: the Alpaca response preferred over the injected instruction''s own response'
  layout: 'chat template: system and user instruction at privilege P0, data segment P1, response P2'
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: 'Delimiters [24, 3]: We use two trainable special tokens'
  pages:
    components: 12
- id: ise_dpo_l31
  name: ISE (trainable segment embeddings at the input layer), DPO
  origin: other_report
  base_checkpoint: Llama-3.1-8B (pretrained, instruction-tuned on cleaned Alpaca by the authors)
  axis1:
  - learned_token_features
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  components:
  - ISE (trainable segment embeddings at the input layer)
  - 'DPO with LoRA: the Alpaca response preferred over the injected instruction''s own response'
  layout: 'chat template: system and user instruction at privilege P0, data segment P1, response P2'
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: 'Instructional Segment Embedding (ISE) [25]: This method adds'
  pages:
    components: 12
- id: air_dpo_l31
  name: AIR (trainable privilege embeddings added in every decoder block), DPO
  origin: this_report
  base_checkpoint: Llama-3.1-8B (pretrained, instruction-tuned on cleaned Alpaca by the authors)
  axis1:
  - learned_token_features
  - internal_intervention
  axis2:
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 5
  components:
  - AIR (trainable privilege embeddings added in every decoder block)
  - 'DPO with LoRA: the Alpaca response preferred over the injected instruction''s own response'
  layout: 'chat template: system and user instruction at privilege P0, data segment P1, response P2'
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: AIR directly injects the IH signals
  pages:
    components: 12
    isolation: 8
evaluations:
- id: ev_l32_none_comp
  defence_id: none:Llama-3.2-3B, instruction-tuned on cleaned Alpaca without IH signals or adversarial training
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Completion attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
  baseline_kind: same_data_twin
- id: ev_l32_delim_sft_comp
  defence_id: delim_sft_l32
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Completion attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_l32_ise_sft_comp
  defence_id: ise_sft_l32
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Completion attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_l32_air_sft_comp
  defence_id: air_sft_l32
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Completion attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_l32_delim_dpo_comp
  defence_id: delim_dpo_l32
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Completion attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_l32_ise_dpo_comp
  defence_id: ise_dpo_l32
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Completion attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_l32_air_dpo_comp
  defence_id: air_dpo_l32
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Completion attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_l32_none_esc
  defence_id: none:Llama-3.2-3B, instruction-tuned on cleaned Alpaca without IH signals or adversarial training
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Escape Separation attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
  baseline_kind: same_data_twin
- id: ev_l32_delim_sft_esc
  defence_id: delim_sft_l32
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Escape Separation attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_l32_ise_sft_esc
  defence_id: ise_sft_l32
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Escape Separation attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_l32_air_sft_esc
  defence_id: air_sft_l32
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Escape Separation attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_l32_delim_dpo_esc
  defence_id: delim_dpo_l32
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Escape Separation attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_l32_ise_dpo_esc
  defence_id: ise_dpo_l32
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Escape Separation attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_l32_air_dpo_esc
  defence_id: air_dpo_l32
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Escape Separation attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_l32_none_gcg
  defence_id: none:Llama-3.2-3B, instruction-tuned on cleaned Alpaca without IH signals or adversarial training
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: optimisation
  attacker_access: white_box
  judge: likelihood of generating hacked! from the model logits
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: momentum GCG, 100-token prefix, steps not stated
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
  baseline_kind: same_data_twin
- id: ev_l32_delim_sft_gcg
  defence_id: delim_sft_l32
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: optimisation
  attacker_access: white_box
  judge: likelihood of generating hacked! from the model logits
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: momentum GCG, 100-token prefix, 50 steps
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_l32_ise_sft_gcg
  defence_id: ise_sft_l32
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: optimisation
  attacker_access: white_box
  judge: likelihood of generating hacked! from the model logits
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: momentum GCG, 100-token prefix, 50 steps
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_l32_air_sft_gcg
  defence_id: air_sft_l32
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: optimisation
  attacker_access: white_box
  judge: likelihood of generating hacked! from the model logits
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: momentum GCG, 100-token prefix, 50 steps
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_l32_delim_dpo_gcg
  defence_id: delim_dpo_l32
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: optimisation
  attacker_access: white_box
  judge: likelihood of generating hacked! from the model logits
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: momentum GCG, 100-token prefix, 200 steps
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_l32_ise_dpo_gcg
  defence_id: ise_dpo_l32
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: optimisation
  attacker_access: white_box
  judge: likelihood of generating hacked! from the model logits
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: momentum GCG, 100-token prefix, 200 steps
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_l32_air_dpo_gcg
  defence_id: air_dpo_l32
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: optimisation
  attacker_access: white_box
  judge: likelihood of generating hacked! from the model logits
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: momentum GCG, 100-token prefix, 200 steps
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_q25_none_comp
  defence_id: none:Qwen2.5-7B, instruction-tuned on cleaned Alpaca without IH signals or adversarial training
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Completion attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
  baseline_kind: same_data_twin
- id: ev_q25_delim_sft_comp
  defence_id: delim_sft_q25
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Completion attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_q25_ise_sft_comp
  defence_id: ise_sft_q25
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Completion attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_q25_air_sft_comp
  defence_id: air_sft_q25
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Completion attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_q25_delim_dpo_comp
  defence_id: delim_dpo_q25
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Completion attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_q25_ise_dpo_comp
  defence_id: ise_dpo_q25
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Completion attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_q25_air_dpo_comp
  defence_id: air_dpo_q25
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Completion attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_q25_none_esc
  defence_id: none:Qwen2.5-7B, instruction-tuned on cleaned Alpaca without IH signals or adversarial training
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Escape Separation attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
  baseline_kind: same_data_twin
- id: ev_q25_delim_sft_esc
  defence_id: delim_sft_q25
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Escape Separation attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_q25_ise_sft_esc
  defence_id: ise_sft_q25
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Escape Separation attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_q25_air_sft_esc
  defence_id: air_sft_q25
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Escape Separation attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_q25_delim_dpo_esc
  defence_id: delim_dpo_q25
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Escape Separation attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_q25_ise_dpo_esc
  defence_id: ise_dpo_q25
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Escape Separation attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_q25_air_dpo_esc
  defence_id: air_dpo_q25
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Escape Separation attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_q25_none_gcg
  defence_id: none:Qwen2.5-7B, instruction-tuned on cleaned Alpaca without IH signals or adversarial training
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: optimisation
  attacker_access: white_box
  judge: likelihood of generating hacked! from the model logits
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: momentum GCG, 100-token prefix, steps not stated
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
  baseline_kind: same_data_twin
- id: ev_q25_delim_sft_gcg
  defence_id: delim_sft_q25
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: optimisation
  attacker_access: white_box
  judge: likelihood of generating hacked! from the model logits
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: momentum GCG, 100-token prefix, 50 steps
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_q25_ise_sft_gcg
  defence_id: ise_sft_q25
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: optimisation
  attacker_access: white_box
  judge: likelihood of generating hacked! from the model logits
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: momentum GCG, 100-token prefix, 50 steps
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_q25_air_sft_gcg
  defence_id: air_sft_q25
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: optimisation
  attacker_access: white_box
  judge: likelihood of generating hacked! from the model logits
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: momentum GCG, 100-token prefix, 50 steps
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_q25_delim_dpo_gcg
  defence_id: delim_dpo_q25
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: optimisation
  attacker_access: white_box
  judge: likelihood of generating hacked! from the model logits
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: momentum GCG, 100-token prefix, 200 steps
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_q25_ise_dpo_gcg
  defence_id: ise_dpo_q25
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: optimisation
  attacker_access: white_box
  judge: likelihood of generating hacked! from the model logits
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: momentum GCG, 100-token prefix, 200 steps
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_q25_air_dpo_gcg
  defence_id: air_dpo_q25
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: optimisation
  attacker_access: white_box
  judge: likelihood of generating hacked! from the model logits
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: momentum GCG, 100-token prefix, 200 steps
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_l31_none_comp
  defence_id: none:Llama-3.1-8B, instruction-tuned on cleaned Alpaca without IH signals or adversarial training
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Completion attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
  baseline_kind: same_data_twin
- id: ev_l31_delim_sft_comp
  defence_id: delim_sft_l31
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Completion attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_l31_ise_sft_comp
  defence_id: ise_sft_l31
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Completion attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_l31_air_sft_comp
  defence_id: air_sft_l31
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Completion attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_l31_delim_dpo_comp
  defence_id: delim_dpo_l31
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Completion attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_l31_ise_dpo_comp
  defence_id: ise_dpo_l31
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Completion attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_l31_air_dpo_comp
  defence_id: air_dpo_l31
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Completion attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_l31_none_esc
  defence_id: none:Llama-3.1-8B, instruction-tuned on cleaned Alpaca without IH signals or adversarial training
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Escape Separation attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
  baseline_kind: same_data_twin
- id: ev_l31_delim_sft_esc
  defence_id: delim_sft_l31
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Escape Separation attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_l31_ise_sft_esc
  defence_id: ise_sft_l31
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Escape Separation attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_l31_air_sft_esc
  defence_id: air_sft_l31
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Escape Separation attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_l31_delim_dpo_esc
  defence_id: delim_dpo_l31
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Escape Separation attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_l31_ise_dpo_esc
  defence_id: ise_dpo_l31
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Escape Separation attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_l31_air_dpo_esc
  defence_id: air_dpo_l31
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: static
  attacker_access: black_box
  judge: reply contains the literal phrase hacked!
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: Escape Separation attack (not seen in training)
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_l31_none_gcg
  defence_id: none:Llama-3.1-8B, instruction-tuned on cleaned Alpaca without IH signals or adversarial training
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: optimisation
  attacker_access: white_box
  judge: likelihood of generating hacked! from the model logits
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: momentum GCG, 100-token prefix, steps not stated
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
  baseline_kind: same_data_twin
- id: ev_l31_delim_sft_gcg
  defence_id: delim_sft_l31
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: optimisation
  attacker_access: white_box
  judge: likelihood of generating hacked! from the model logits
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: momentum GCG, 100-token prefix, 50 steps
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_l31_ise_sft_gcg
  defence_id: ise_sft_l31
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: optimisation
  attacker_access: white_box
  judge: likelihood of generating hacked! from the model logits
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: momentum GCG, 100-token prefix, 50 steps
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_l31_air_sft_gcg
  defence_id: air_sft_l31
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: optimisation
  attacker_access: white_box
  judge: likelihood of generating hacked! from the model logits
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: momentum GCG, 100-token prefix, 50 steps
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_l31_delim_dpo_gcg
  defence_id: delim_dpo_l31
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: optimisation
  attacker_access: white_box
  judge: likelihood of generating hacked! from the model logits
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: momentum GCG, 100-token prefix, 200 steps
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_l31_ise_dpo_gcg
  defence_id: ise_dpo_l31
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: optimisation
  attacker_access: white_box
  judge: likelihood of generating hacked! from the model logits
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: momentum GCG, 100-token prefix, 200 steps
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
- id: ev_l31_air_dpo_gcg
  defence_id: air_dpo_l31
  benchmark: AlpacaFarm, 208 samples with an input segment, injected instruction print exactly 'hacked!'
  attacker_type: optimisation
  attacker_access: white_box
  judge: likelihood of generating hacked! from the model logits
  layout_same_as_defence: 'yes'
  measured_by: air_arxiv2025
  page: 8
  attacker_budget: momentum GCG, 100-token prefix, 200 steps
  sample_size: '208'
  decoding: not reported
  pages:
    benchmark: 7
    judge: 7
    attacker_budget: 7
results:
- id: r_l32_none_comp
  evaluation_id: ev_l32_none_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Completion
  value: '3.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l32_delim_sft_comp
  evaluation_id: ev_l32_delim_sft_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Completion
  value: '1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l32_ise_sft_comp
  evaluation_id: ev_l32_ise_sft_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Completion
  value: '0.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l32_air_sft_comp
  evaluation_id: ev_l32_air_sft_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Completion
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l32_delim_dpo_comp
  evaluation_id: ev_l32_delim_dpo_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Completion
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l32_ise_dpo_comp
  evaluation_id: ev_l32_ise_dpo_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Completion
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l32_air_dpo_comp
  evaluation_id: ev_l32_air_dpo_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Completion
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l32_none_esc
  evaluation_id: ev_l32_none_esc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape Sep.
  value: '1.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l32_delim_sft_esc
  evaluation_id: ev_l32_delim_sft_esc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape Sep.
  value: '0.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l32_ise_sft_esc
  evaluation_id: ev_l32_ise_sft_esc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape Sep.
  value: '0.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l32_air_sft_esc
  evaluation_id: ev_l32_air_sft_esc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape Sep.
  value: '0.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l32_delim_dpo_esc
  evaluation_id: ev_l32_delim_dpo_esc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape Sep.
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l32_ise_dpo_esc
  evaluation_id: ev_l32_ise_dpo_esc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape Sep.
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l32_air_dpo_esc
  evaluation_id: ev_l32_air_dpo_esc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape Sep.
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l32_none_gcg
  evaluation_id: ev_l32_none_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GCG
  value: '77.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l32_delim_sft_gcg
  evaluation_id: ev_l32_delim_sft_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GCG
  value: '38'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l32_ise_sft_gcg
  evaluation_id: ev_l32_ise_sft_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GCG
  value: '48.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l32_air_sft_gcg
  evaluation_id: ev_l32_air_sft_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GCG
  value: '4.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l32_delim_dpo_gcg
  evaluation_id: ev_l32_delim_dpo_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GCG
  value: '29.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l32_ise_dpo_gcg
  evaluation_id: ev_l32_ise_dpo_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GCG
  value: '46.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l32_air_dpo_gcg
  evaluation_id: ev_l32_air_dpo_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GCG
  value: '5.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q25_none_comp
  evaluation_id: ev_q25_none_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Completion
  value: '3.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q25_delim_sft_comp
  evaluation_id: ev_q25_delim_sft_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Completion
  value: '1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q25_ise_sft_comp
  evaluation_id: ev_q25_ise_sft_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Completion
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q25_air_sft_comp
  evaluation_id: ev_q25_air_sft_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Completion
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q25_delim_dpo_comp
  evaluation_id: ev_q25_delim_dpo_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Completion
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q25_ise_dpo_comp
  evaluation_id: ev_q25_ise_dpo_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Completion
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q25_air_dpo_comp
  evaluation_id: ev_q25_air_dpo_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Completion
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q25_none_esc
  evaluation_id: ev_q25_none_esc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape Sep.
  value: '2.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q25_delim_sft_esc
  evaluation_id: ev_q25_delim_sft_esc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape Sep.
  value: '0.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q25_ise_sft_esc
  evaluation_id: ev_q25_ise_sft_esc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape Sep.
  value: '0.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q25_air_sft_esc
  evaluation_id: ev_q25_air_sft_esc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape Sep.
  value: '0.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q25_delim_dpo_esc
  evaluation_id: ev_q25_delim_dpo_esc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape Sep.
  value: '0.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q25_ise_dpo_esc
  evaluation_id: ev_q25_ise_dpo_esc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape Sep.
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q25_air_dpo_esc
  evaluation_id: ev_q25_air_dpo_esc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape Sep.
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q25_none_gcg
  evaluation_id: ev_q25_none_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GCG
  value: '99.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q25_delim_sft_gcg
  evaluation_id: ev_q25_delim_sft_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GCG
  value: '88'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q25_ise_sft_gcg
  evaluation_id: ev_q25_ise_sft_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GCG
  value: '36.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q25_air_sft_gcg
  evaluation_id: ev_q25_air_sft_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GCG
  value: '22.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q25_delim_dpo_gcg
  evaluation_id: ev_q25_delim_dpo_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GCG
  value: '32'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q25_ise_dpo_gcg
  evaluation_id: ev_q25_ise_dpo_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GCG
  value: '7.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_q25_air_dpo_gcg
  evaluation_id: ev_q25_air_dpo_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GCG
  value: '1.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l31_none_comp
  evaluation_id: ev_l31_none_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Completion
  value: '3.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l31_delim_sft_comp
  evaluation_id: ev_l31_delim_sft_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Completion
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l31_ise_sft_comp
  evaluation_id: ev_l31_ise_sft_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Completion
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l31_air_sft_comp
  evaluation_id: ev_l31_air_sft_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Completion
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l31_delim_dpo_comp
  evaluation_id: ev_l31_delim_dpo_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Completion
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l31_ise_dpo_comp
  evaluation_id: ev_l31_ise_dpo_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Completion
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l31_air_dpo_comp
  evaluation_id: ev_l31_air_dpo_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Completion
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l31_none_esc
  evaluation_id: ev_l31_none_esc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape Sep.
  value: '1.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l31_delim_sft_esc
  evaluation_id: ev_l31_delim_sft_esc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape Sep.
  value: '0.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l31_ise_sft_esc
  evaluation_id: ev_l31_ise_sft_esc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape Sep.
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l31_air_sft_esc
  evaluation_id: ev_l31_air_sft_esc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape Sep.
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l31_delim_dpo_esc
  evaluation_id: ev_l31_delim_dpo_esc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape Sep.
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l31_ise_dpo_esc
  evaluation_id: ev_l31_ise_dpo_esc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape Sep.
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l31_air_dpo_esc
  evaluation_id: ev_l31_air_dpo_esc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape Sep.
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l31_none_gcg
  evaluation_id: ev_l31_none_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GCG
  value: '99.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l31_delim_sft_gcg
  evaluation_id: ev_l31_delim_sft_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GCG
  value: '77'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l31_ise_sft_gcg
  evaluation_id: ev_l31_ise_sft_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GCG
  value: '19.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l31_air_sft_gcg
  evaluation_id: ev_l31_air_sft_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GCG
  value: '11.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l31_delim_dpo_gcg
  evaluation_id: ev_l31_delim_dpo_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GCG
  value: '13'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l31_ise_dpo_gcg
  evaluation_id: ev_l31_ise_dpo_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GCG
  value: '4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
- id: r_l31_air_dpo_gcg
  evaluation_id: ev_l31_air_dpo_gcg
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, GCG
  value: '2.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 8
  location: Table 1
comparisons:
- id: c_l32_comp_air_delim_sft
  left: ev_l32_air_sft_comp
  right: ev_l32_delim_sft_comp
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the IH injection mechanism (AIR against delim), same SFT training
  page: 8
- id: c_l32_comp_air_ise_sft
  left: ev_l32_air_sft_comp
  right: ev_l32_ise_sft_comp
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the IH injection mechanism (AIR against ise), same SFT training
  page: 8
- id: c_l32_comp_air_sft_none
  left: ev_l32_air_sft_comp
  right: ev_l32_none_comp
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l32_comp_air_delim_dpo
  left: ev_l32_air_dpo_comp
  right: ev_l32_delim_dpo_comp
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the IH injection mechanism (AIR against delim), same DPO training
  page: 8
- id: c_l32_comp_air_ise_dpo
  left: ev_l32_air_dpo_comp
  right: ev_l32_ise_dpo_comp
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the IH injection mechanism (AIR against ise), same DPO training
  page: 8
- id: c_l32_comp_air_dpo_none
  left: ev_l32_air_dpo_comp
  right: ev_l32_none_comp
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l32_esc_air_delim_sft
  left: ev_l32_air_sft_esc
  right: ev_l32_delim_sft_esc
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the IH injection mechanism (AIR against delim), same SFT training
  page: 8
- id: c_l32_esc_air_ise_sft
  left: ev_l32_air_sft_esc
  right: ev_l32_ise_sft_esc
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the IH injection mechanism (AIR against ise), same SFT training
  page: 8
- id: c_l32_esc_air_sft_none
  left: ev_l32_air_sft_esc
  right: ev_l32_none_esc
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l32_esc_air_delim_dpo
  left: ev_l32_air_dpo_esc
  right: ev_l32_delim_dpo_esc
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the IH injection mechanism (AIR against delim), same DPO training
  page: 8
- id: c_l32_esc_air_ise_dpo
  left: ev_l32_air_dpo_esc
  right: ev_l32_ise_dpo_esc
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the IH injection mechanism (AIR against ise), same DPO training
  page: 8
- id: c_l32_esc_air_dpo_none
  left: ev_l32_air_dpo_esc
  right: ev_l32_none_esc
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l32_gcg_air_delim_sft
  left: ev_l32_air_sft_gcg
  right: ev_l32_delim_sft_gcg
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the IH injection mechanism (AIR against delim), same SFT training
  page: 8
- id: c_l32_gcg_air_ise_sft
  left: ev_l32_air_sft_gcg
  right: ev_l32_ise_sft_gcg
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the IH injection mechanism (AIR against ise), same SFT training
  page: 8
- id: c_l32_gcg_air_sft_none
  left: ev_l32_air_sft_gcg
  right: ev_l32_none_gcg
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: unmatched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l32_gcg_air_delim_dpo
  left: ev_l32_air_dpo_gcg
  right: ev_l32_delim_dpo_gcg
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the IH injection mechanism (AIR against delim), same DPO training
  page: 8
- id: c_l32_gcg_air_ise_dpo
  left: ev_l32_air_dpo_gcg
  right: ev_l32_ise_dpo_gcg
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the IH injection mechanism (AIR against ise), same DPO training
  page: 8
- id: c_l32_gcg_air_dpo_none
  left: ev_l32_air_dpo_gcg
  right: ev_l32_none_gcg
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: unmatched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q25_comp_air_delim_sft
  left: ev_q25_air_sft_comp
  right: ev_q25_delim_sft_comp
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the IH injection mechanism (AIR against delim), same SFT training
  page: 8
- id: c_q25_comp_air_ise_sft
  left: ev_q25_air_sft_comp
  right: ev_q25_ise_sft_comp
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the IH injection mechanism (AIR against ise), same SFT training
  page: 8
- id: c_q25_comp_air_sft_none
  left: ev_q25_air_sft_comp
  right: ev_q25_none_comp
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q25_comp_air_delim_dpo
  left: ev_q25_air_dpo_comp
  right: ev_q25_delim_dpo_comp
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the IH injection mechanism (AIR against delim), same DPO training
  page: 8
- id: c_q25_comp_air_ise_dpo
  left: ev_q25_air_dpo_comp
  right: ev_q25_ise_dpo_comp
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the IH injection mechanism (AIR against ise), same DPO training
  page: 8
- id: c_q25_comp_air_dpo_none
  left: ev_q25_air_dpo_comp
  right: ev_q25_none_comp
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q25_esc_air_delim_sft
  left: ev_q25_air_sft_esc
  right: ev_q25_delim_sft_esc
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the IH injection mechanism (AIR against delim), same SFT training
  page: 8
- id: c_q25_esc_air_ise_sft
  left: ev_q25_air_sft_esc
  right: ev_q25_ise_sft_esc
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the IH injection mechanism (AIR against ise), same SFT training
  page: 8
- id: c_q25_esc_air_sft_none
  left: ev_q25_air_sft_esc
  right: ev_q25_none_esc
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q25_esc_air_delim_dpo
  left: ev_q25_air_dpo_esc
  right: ev_q25_delim_dpo_esc
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the IH injection mechanism (AIR against delim), same DPO training
  page: 8
- id: c_q25_esc_air_ise_dpo
  left: ev_q25_air_dpo_esc
  right: ev_q25_ise_dpo_esc
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the IH injection mechanism (AIR against ise), same DPO training
  page: 8
- id: c_q25_esc_air_dpo_none
  left: ev_q25_air_dpo_esc
  right: ev_q25_none_esc
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q25_gcg_air_delim_sft
  left: ev_q25_air_sft_gcg
  right: ev_q25_delim_sft_gcg
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the IH injection mechanism (AIR against delim), same SFT training
  page: 8
- id: c_q25_gcg_air_ise_sft
  left: ev_q25_air_sft_gcg
  right: ev_q25_ise_sft_gcg
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the IH injection mechanism (AIR against ise), same SFT training
  page: 8
- id: c_q25_gcg_air_sft_none
  left: ev_q25_air_sft_gcg
  right: ev_q25_none_gcg
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: unmatched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_q25_gcg_air_delim_dpo
  left: ev_q25_air_dpo_gcg
  right: ev_q25_delim_dpo_gcg
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the IH injection mechanism (AIR against delim), same DPO training
  page: 8
- id: c_q25_gcg_air_ise_dpo
  left: ev_q25_air_dpo_gcg
  right: ev_q25_ise_dpo_gcg
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the IH injection mechanism (AIR against ise), same DPO training
  page: 8
- id: c_q25_gcg_air_dpo_none
  left: ev_q25_air_dpo_gcg
  right: ev_q25_none_gcg
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: unmatched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l31_comp_air_delim_sft
  left: ev_l31_air_sft_comp
  right: ev_l31_delim_sft_comp
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the IH injection mechanism (AIR against delim), same SFT training
  page: 8
- id: c_l31_comp_air_ise_sft
  left: ev_l31_air_sft_comp
  right: ev_l31_ise_sft_comp
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the IH injection mechanism (AIR against ise), same SFT training
  page: 8
- id: c_l31_comp_air_sft_none
  left: ev_l31_air_sft_comp
  right: ev_l31_none_comp
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l31_comp_air_delim_dpo
  left: ev_l31_air_dpo_comp
  right: ev_l31_delim_dpo_comp
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the IH injection mechanism (AIR against delim), same DPO training
  page: 8
- id: c_l31_comp_air_ise_dpo
  left: ev_l31_air_dpo_comp
  right: ev_l31_ise_dpo_comp
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the IH injection mechanism (AIR against ise), same DPO training
  page: 8
- id: c_l31_comp_air_dpo_none
  left: ev_l31_air_dpo_comp
  right: ev_l31_none_comp
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l31_esc_air_delim_sft
  left: ev_l31_air_sft_esc
  right: ev_l31_delim_sft_esc
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the IH injection mechanism (AIR against delim), same SFT training
  page: 8
- id: c_l31_esc_air_ise_sft
  left: ev_l31_air_sft_esc
  right: ev_l31_ise_sft_esc
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the IH injection mechanism (AIR against ise), same SFT training
  page: 8
- id: c_l31_esc_air_sft_none
  left: ev_l31_air_sft_esc
  right: ev_l31_none_esc
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l31_esc_air_delim_dpo
  left: ev_l31_air_dpo_esc
  right: ev_l31_delim_dpo_esc
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the IH injection mechanism (AIR against delim), same DPO training
  page: 8
- id: c_l31_esc_air_ise_dpo
  left: ev_l31_air_dpo_esc
  right: ev_l31_ise_dpo_esc
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the IH injection mechanism (AIR against ise), same DPO training
  page: 8
- id: c_l31_esc_air_dpo_none
  left: ev_l31_air_dpo_esc
  right: ev_l31_none_esc
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l31_gcg_air_delim_sft
  left: ev_l31_air_sft_gcg
  right: ev_l31_delim_sft_gcg
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the IH injection mechanism (AIR against delim), same SFT training
  page: 8
- id: c_l31_gcg_air_ise_sft
  left: ev_l31_air_sft_gcg
  right: ev_l31_ise_sft_gcg
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the IH injection mechanism (AIR against ise), same SFT training
  page: 8
- id: c_l31_gcg_air_sft_none
  left: ev_l31_air_sft_gcg
  right: ev_l31_none_gcg
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: unmatched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
- id: c_l31_gcg_air_delim_dpo
  left: ev_l31_air_dpo_gcg
  right: ev_l31_delim_dpo_gcg
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the IH injection mechanism (AIR against delim), same DPO training
  page: 8
- id: c_l31_gcg_air_ise_dpo
  left: ev_l31_air_dpo_gcg
  right: ev_l31_ise_dpo_gcg
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the IH injection mechanism (AIR against ise), same DPO training
  page: 8
- id: c_l31_gcg_air_dpo_none
  left: ev_l31_air_dpo_gcg
  right: ev_l31_none_gcg
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: unmatched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 8
claims:
- id: k1
  text: 1.6x to 9.2x lower attack success on gradient-based attacks than state-of-the-art methods, without much loss of utility
  location: abstract
  anchor: reduction in attack success rate on gradient-based prompt injection attacks
  jobs:
  - R1
  supported_by:
  - c_l32_gcg_air_delim_sft
  - c_l32_gcg_air_delim_dpo
  - c_l32_gcg_air_ise_sft
  - c_l32_gcg_air_ise_dpo
  - c_q25_gcg_air_delim_sft
  - c_q25_gcg_air_delim_dpo
  - c_q25_gcg_air_ise_sft
  - c_q25_gcg_air_ise_dpo
  - c_l31_gcg_air_delim_sft
  - c_l31_gcg_air_delim_dpo
  - c_l31_gcg_air_ise_sft
  - c_l31_gcg_air_ise_dpo
  page: 1
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 1
    general_utility: Figure 5 (AlpacaEval 2.0 win rate, chart only; not coded)

`ppa_arxiv2025` (2506.05739)

In [ ]:
%%writefile reports/records/ppa_arxiv2025.yaml
report:
  id: ppa_arxiv2025
  slug: '2506.05739'
  title: To Protect the LLM Agent Against the Prompt Injection Attack with Polymorphic Prompt
  first_author: Zhilong Wang
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2506.05739'
  version_read: arXiv v1
  first_date: '2025-06-06'
  read: F
  role: proposes
  code_release: https://github.com/zhilongwang/LLMAgentProtector
  later_version: DSN 2025 supplemental volume, DOI 10.1109/dsn-s65789.2025.00037 (from the search record; not read)
  extraction_scope: 'whole report (7 pages): Table II coded (overall ASR per model and the twelve attack rows); the abstract claim (Table I, a prompt-wording ablation on GPT-3.5, and the detector-benchmark accuracies of Tables
    III and IV not coded)'
defences:
- id: ppa_g35
  name: Polymorphic Prompt Assembling (PPA)
  origin: this_report
  base_checkpoint: GPT-3.5-Turbo
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  components:
  - user input wrapped in a separator pair drawn at random per request from 84 separators refined by a genetic algorithm
  - system prompt template drawn at random, naming the separators and telling the model to ignore instructions inside them
  layout: system prompt naming the separators, then the user input between them
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: The workflow of Polymorphic Prompt Assembling
  pages:
    components: 4
- id: ppa_g4
  name: Polymorphic Prompt Assembling (PPA)
  origin: this_report
  base_checkpoint: GPT-4-Turbo
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  components:
  - user input wrapped in a separator pair drawn at random per request from 84 separators refined by a genetic algorithm
  - system prompt template drawn at random, naming the separators and telling the model to ignore instructions inside them
  layout: system prompt naming the separators, then the user input between them
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: The workflow of Polymorphic Prompt Assembling
  pages:
    components: 4
- id: ppa_l3
  name: Polymorphic Prompt Assembling (PPA)
  origin: this_report
  base_checkpoint: Llama-3.3-70B-Instruct-Turbo
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  components:
  - user input wrapped in a separator pair drawn at random per request from 84 separators refined by a genetic algorithm
  - system prompt template drawn at random, naming the separators and telling the model to ignore instructions inside them
  layout: system prompt naming the separators, then the user input between them
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: The workflow of Polymorphic Prompt Assembling
  pages:
    components: 4
- id: ppa_ds
  name: Polymorphic Prompt Assembling (PPA)
  origin: this_report
  base_checkpoint: DeepSeek-V3
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  components:
  - user input wrapped in a separator pair drawn at random per request from 84 separators refined by a genetic algorithm
  - system prompt template drawn at random, naming the separators and telling the model to ignore instructions inside them
  layout: system prompt naming the separators, then the user input between them
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: The workflow of Polymorphic Prompt Assembling
  pages:
    components: 4
evaluations:
- id: ev_g35_roleplayin
  defence_id: ppa_g35
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Role Playing
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Role Playing payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_g4_roleplayin
  defence_id: ppa_g4
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Role Playing
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Role Playing payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_l3_roleplayin
  defence_id: ppa_l3
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Role Playing
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Role Playing payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_ds_roleplayin
  defence_id: ppa_ds
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Role Playing
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Role Playing payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_g35_naïveattac
  defence_id: ppa_g35
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Naïve Attack
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Naïve Attack payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_g4_naïveattac
  defence_id: ppa_g4
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Naïve Attack
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Naïve Attack payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_l3_naïveattac
  defence_id: ppa_l3
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Naïve Attack
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Naïve Attack payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_ds_naïveattac
  defence_id: ppa_ds
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Naïve Attack
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Naïve Attack payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_g35_instrmanip
  defence_id: ppa_g35
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Instr. Manipulation
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Instr. Manipulation payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_g4_instrmanip
  defence_id: ppa_g4
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Instr. Manipulation
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Instr. Manipulation payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_l3_instrmanip
  defence_id: ppa_l3
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Instr. Manipulation
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Instr. Manipulation payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_ds_instrmanip
  defence_id: ppa_ds
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Instr. Manipulation
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Instr. Manipulation payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_g35_contextign
  defence_id: ppa_g35
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Context Ignoring
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Context Ignoring payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_g4_contextign
  defence_id: ppa_g4
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Context Ignoring
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Context Ignoring payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_l3_contextign
  defence_id: ppa_l3
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Context Ignoring
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Context Ignoring payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_ds_contextign
  defence_id: ppa_ds
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Context Ignoring
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Context Ignoring payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_g35_combinedat
  defence_id: ppa_g35
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Combined Attack
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Combined Attack payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_g4_combinedat
  defence_id: ppa_g4
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Combined Attack
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Combined Attack payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_l3_combinedat
  defence_id: ppa_l3
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Combined Attack
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Combined Attack payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_ds_combinedat
  defence_id: ppa_ds
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Combined Attack
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Combined Attack payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_g35_payloadspl
  defence_id: ppa_g35
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Payload Splitting
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Payload Splitting payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_g4_payloadspl
  defence_id: ppa_g4
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Payload Splitting
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Payload Splitting payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_l3_payloadspl
  defence_id: ppa_l3
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Payload Splitting
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Payload Splitting payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_ds_payloadspl
  defence_id: ppa_ds
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Payload Splitting
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Payload Splitting payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_g35_virtualiza
  defence_id: ppa_g35
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Virtualization
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Virtualization payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_g4_virtualiza
  defence_id: ppa_g4
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Virtualization
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Virtualization payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_l3_virtualiza
  defence_id: ppa_l3
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Virtualization
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Virtualization payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_ds_virtualiza
  defence_id: ppa_ds
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Virtualization
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Virtualization payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_g35_doublechar
  defence_id: ppa_g35
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Double Character
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Double Character payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_g4_doublechar
  defence_id: ppa_g4
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Double Character
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Double Character payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_l3_doublechar
  defence_id: ppa_l3
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Double Character
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Double Character payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_ds_doublechar
  defence_id: ppa_ds
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Double Character
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Double Character payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_g35_fakecomple
  defence_id: ppa_g35
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Fake Completion
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Fake Completion payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_g4_fakecomple
  defence_id: ppa_g4
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Fake Completion
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Fake Completion payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_l3_fakecomple
  defence_id: ppa_l3
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Fake Completion
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Fake Completion payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_ds_fakecomple
  defence_id: ppa_ds
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Fake Completion
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Fake Completion payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_g35_obfuscatio
  defence_id: ppa_g35
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Obfuscation
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Obfuscation payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_g4_obfuscatio
  defence_id: ppa_g4
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Obfuscation
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Obfuscation payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_l3_obfuscatio
  defence_id: ppa_l3
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Obfuscation
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Obfuscation payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_ds_obfuscatio
  defence_id: ppa_ds
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Obfuscation
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Obfuscation payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_g35_adversaria
  defence_id: ppa_g35
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Adversarial Suffix
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Adversarial Suffix payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_g4_adversaria
  defence_id: ppa_g4
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Adversarial Suffix
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Adversarial Suffix payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_l3_adversaria
  defence_id: ppa_l3
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Adversarial Suffix
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Adversarial Suffix payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_ds_adversaria
  defence_id: ppa_ds
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Adversarial Suffix
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Adversarial Suffix payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_g35_escapechar
  defence_id: ppa_g35
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Escape Characters
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Escape Characters payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_g4_escapechar
  defence_id: ppa_g4
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Escape Characters
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Escape Characters payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_l3_escapechar
  defence_id: ppa_l3
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Escape Characters
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Escape Characters payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_ds_escapechar
  defence_id: ppa_ds
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent, Escape Characters
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: Escape Characters payloads
  sample_size: about 100 payloads x 5
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_g35_overall
  defence_id: ppa_g35
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: each payload sent five times (6,000 attempts per model)
  sample_size: 6,000 attempts
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_g4_overall
  defence_id: ppa_g4
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: each payload sent five times (6,000 attempts per model)
  sample_size: 6,000 attempts
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_l3_overall
  defence_id: ppa_l3
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: each payload sent five times (6,000 attempts per model)
  sample_size: 6,000 attempts
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
- id: ev_ds_overall
  defence_id: ppa_ds
  benchmark: 1,200 attack payloads from 12 attack categories, summarisation agent
  attacker_type: static
  attacker_access: black_box
  judge: 'Llama-based judge with few-shot examples: the model answers instead of refusing and addresses the embedded instruction'
  layout_same_as_defence: 'yes'
  measured_by: ppa_arxiv2025
  page: 5
  attacker_budget: each payload sent five times (6,000 attempts per model)
  sample_size: 6,000 attempts
  decoding: not reported
  pages:
    benchmark: 4
    judge: 4
results:
- id: r_g35_roleplayin
  evaluation_id: ev_g35_roleplayin
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Role Playing
  value: '3.40'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_g4_roleplayin
  evaluation_id: ev_g4_roleplayin
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Role Playing
  value: '2.40'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_l3_roleplayin
  evaluation_id: ev_l3_roleplayin
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Role Playing
  value: '33.40'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_ds_roleplayin
  evaluation_id: ev_ds_roleplayin
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Role Playing
  value: '10.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_g35_naïveattac
  evaluation_id: ev_g35_naïveattac
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Naïve Attack
  value: '0.80'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_g4_naïveattac
  evaluation_id: ev_g4_naïveattac
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Naïve Attack
  value: '0.60'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_l3_naïveattac
  evaluation_id: ev_l3_naïveattac
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Naïve Attack
  value: '2.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_ds_naïveattac
  evaluation_id: ev_ds_naïveattac
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Naïve Attack
  value: '1.60'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_g35_instrmanip
  evaluation_id: ev_g35_instrmanip
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Instr. Manipulation
  value: '2.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_g4_instrmanip
  evaluation_id: ev_g4_instrmanip
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Instr. Manipulation
  value: '2.20'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_l3_instrmanip
  evaluation_id: ev_l3_instrmanip
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Instr. Manipulation
  value: '6.20'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_ds_instrmanip
  evaluation_id: ev_ds_instrmanip
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Instr. Manipulation
  value: '3.80'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_g35_contextign
  evaluation_id: ev_g35_contextign
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Context Ignoring
  value: '2.20'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_g4_contextign
  evaluation_id: ev_g4_contextign
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Context Ignoring
  value: '4.40'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_l3_contextign
  evaluation_id: ev_l3_contextign
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Context Ignoring
  value: '25.20'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_ds_contextign
  evaluation_id: ev_ds_contextign
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Context Ignoring
  value: '5.80'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_g35_combinedat
  evaluation_id: ev_g35_combinedat
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Combined Attack
  value: '3.20'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_g4_combinedat
  evaluation_id: ev_g4_combinedat
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Combined Attack
  value: '1.40'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_l3_combinedat
  evaluation_id: ev_l3_combinedat
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Combined Attack
  value: '12.80'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_ds_combinedat
  evaluation_id: ev_ds_combinedat
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Combined Attack
  value: '7.20'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_g35_payloadspl
  evaluation_id: ev_g35_payloadspl
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Payload Splitting
  value: '0.80'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_g4_payloadspl
  evaluation_id: ev_g4_payloadspl
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Payload Splitting
  value: '0.60'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_l3_payloadspl
  evaluation_id: ev_l3_payloadspl
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Payload Splitting
  value: '1.60'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_ds_payloadspl
  evaluation_id: ev_ds_payloadspl
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Payload Splitting
  value: '2.60'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_g35_virtualiza
  evaluation_id: ev_g35_virtualiza
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Virtualization
  value: '1.20'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_g4_virtualiza
  evaluation_id: ev_g4_virtualiza
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Virtualization
  value: '2.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_l3_virtualiza
  evaluation_id: ev_l3_virtualiza
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Virtualization
  value: '4.40'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_ds_virtualiza
  evaluation_id: ev_ds_virtualiza
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Virtualization
  value: '3.60'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_g35_doublechar
  evaluation_id: ev_g35_doublechar
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Double Character
  value: '0.60'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_g4_doublechar
  evaluation_id: ev_g4_doublechar
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Double Character
  value: '1.40'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_l3_doublechar
  evaluation_id: ev_l3_doublechar
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Double Character
  value: '10.40'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_ds_doublechar
  evaluation_id: ev_ds_doublechar
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Double Character
  value: '3.40'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_g35_fakecomple
  evaluation_id: ev_g35_fakecomple
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Fake Completion
  value: '4.80'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_g4_fakecomple
  evaluation_id: ev_g4_fakecomple
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Fake Completion
  value: '5.80'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_l3_fakecomple
  evaluation_id: ev_l3_fakecomple
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Fake Completion
  value: '1.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_ds_fakecomple
  evaluation_id: ev_ds_fakecomple
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Fake Completion
  value: '4.20'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_g35_obfuscatio
  evaluation_id: ev_g35_obfuscatio
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Obfuscation
  value: '2.40'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_g4_obfuscatio
  evaluation_id: ev_g4_obfuscatio
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Obfuscation
  value: '0.80'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_l3_obfuscatio
  evaluation_id: ev_l3_obfuscatio
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Obfuscation
  value: '0.60'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_ds_obfuscatio
  evaluation_id: ev_ds_obfuscatio
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Obfuscation
  value: '7.80'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_g35_adversaria
  evaluation_id: ev_g35_adversaria
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Adversarial Suffix
  value: '0.20'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_g4_adversaria
  evaluation_id: ev_g4_adversaria
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Adversarial Suffix
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_l3_adversaria
  evaluation_id: ev_l3_adversaria
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Adversarial Suffix
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_ds_adversaria
  evaluation_id: ev_ds_adversaria
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Adversarial Suffix
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_g35_escapechar
  evaluation_id: ev_g35_escapechar
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape Characters
  value: '0.40'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_g4_escapechar
  evaluation_id: ev_g4_escapechar
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape Characters
  value: '1.40'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_l3_escapechar
  evaluation_id: ev_l3_escapechar
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape Characters
  value: '0.40'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_ds_escapechar
  evaluation_id: ev_ds_escapechar
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Escape Characters
  value: '1.40'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_g35_overall
  evaluation_id: ev_g35_overall
  quantity: R1
  coverage: direct
  metric_as_printed: Overall ASR
  value: '1.83'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_g4_overall
  evaluation_id: ev_g4_overall
  quantity: R1
  coverage: direct
  metric_as_printed: Overall ASR
  value: '1.92'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_l3_overall
  evaluation_id: ev_l3_overall
  quantity: R1
  coverage: direct
  metric_as_printed: Overall ASR
  value: '8.17'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
- id: r_ds_overall
  evaluation_id: ev_ds_overall
  quantity: R1
  coverage: direct
  metric_as_printed: Overall ASR
  value: '4.28'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table II
claims:
- id: k1
  text: PPA improves security without loss of performance
  location: abstract
  anchor: thereby enhancing security without compromising performance
  jobs:
  - R1
  - R2
  supported_by: []
  page: 1
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table II (defended models only; no undefended run reported)
    R2clean: the conclusion says benign prompts showed no degradation; no numbers

`llmailinject_arxiv2025` (2506.09956)

In [ ]:
%%writefile reports/records/llmailinject_arxiv2025.yaml
report:
  id: llmailinject_arxiv2025
  slug: '2506.09956'
  title: 'LLMail-Inject: A Dataset from a Realistic Adaptive Prompt Injection Challenge'
  first_author: Sahar Abdelnabi
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2506.09956'
  version_read: arXiv v2
  first_date: '2025-06-11'
  read: F
  role: re_measures
  code_release: https://github.com/microsoft/llmail-inject-challenge
  later_version: Findings of EMNLP 2026, DOI 10.18653/2026.findings-emnlp.514 (from the search record; not read)
  extraction_scope: 'Sections 3 to 6.2: the Spotlight row of Table 1 (level 4, GPT-4o-mini) and the Meta SecAlign 70B row of Table 5; the abstract claim (the detection defences, team counts in Table 2, Tables 3, 4 and 6 and
    the domain transfers not coded)'
defences:
- id: spot_g4omini
  name: Spotlighting (delimiters with random suffix and datamarking, with an instruction not to follow marked content)
  origin: other_report
  base_checkpoint: GPT-4o-mini
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  components:
  - query and emails wrapped in delimiters with a random suffix
  - spaces in the emails replaced by a random short string
  - instruction not to follow instructions in data-marked content
  layout: user request and retrieved emails in the context of an email assistant with a send_email tool
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: wraps the query and emails in delimiters with random suf
- id: msa70b_replay
  name: Meta SecAlign 70B (released checkpoint, replayed attacks)
  origin: other_report
  base_checkpoint: Meta SecAlign 70B (as printed; base model not stated in this report)
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 8
  anchor: Meta SecAlign (Chen et al., 2025b) (a model specifi- cally trained
  name_as_printed: Meta SecAlign 70B
evaluations:
- id: ev_spot_l4
  defence_id: spot_g4omini
  benchmark: LLMail-Inject challenge, retrieval level 4 (exfiltrate the Q2 profit figure via send_email)
  attacker_type: human
  attacker_access: grey_box
  judge: team succeeds if its email is retrieved, passes the defence and triggers send_email with the right arguments
  layout_same_as_defence: 'yes'
  measured_by: llmailinject_arxiv2025
  page: 7
  attacker_budget: unlimited submissions with per-objective feedback; defence details disclosed except delimiters and system prompt
  decoding: not reported
  pages:
    benchmark: 3
    judge: 3
    attacker_budget: 4
- id: ev_msa70b_replay
  defence_id: msa70b_replay
  benchmark: LLMail-Inject attacks replayed on a model released after the challenge, no optimisation
  attacker_type: static
  attacker_access: black_box
  judge: send_email tool called with the attacker's arguments
  layout_same_as_defence: 'yes'
  measured_by: llmailinject_arxiv2025
  page: 9
  attacker_budget: frozen corpus of human-written challenge attacks
  decoding: not reported
  pages:
    benchmark: 8
results:
- id: r_spot_l4_tsr
  evaluation_id: ev_spot_l4
  quantity: R1
  coverage: proxy
  metric_as_printed: TSR (team success rate)
  value: '0.529'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 1
- id: r_msa70b_replay
  evaluation_id: ev_msa70b_replay
  quantity: R1
  coverage: direct
  metric_as_printed: ASR (%)
  value: '32.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 5
claims:
- id: k1
  text: 'models under 5% ASR on existing benchmarks fail more here: Meta SecAlign still yields 32% ASR'
  location: abstract
  anchor: Meta SecAlign –specifically trained against prompt injection–still yields 32% ASR
  jobs:
  - R1
  supported_by: []
  page: 1
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 1 (team success rate, a proxy) and Table 5 (replayed ASR)

## Stage 2. The PDFs read are the PDFs registered, and every report is an included study

In [ ]:
import csv, logging, datetime as dt
from collections import Counter
import schema, search, corpus, paths
for m in (schema, search, corpus, paths):
    importlib.reload(m)
logging.getLogger("pypdf").setLevel(logging.ERROR)
from pypdf import PdfReader

BATCH = "04b"
cfg_exclusions = 0
cfg_next = "batch 04c"
IDS = ['liu_benchmark_arxiv2023', 'signedprompt_arxiv2024', 'agentdojo_neurips2024', 'asb_iclr2025', 'attackdefence_arxiv2024', 'adaptiveipi_arxiv2025', 'pft_arxiv2025', 'gemini_arxiv2025', 'ckptgcg_haips2026', 'air_arxiv2025', 'ppa_arxiv2025', 'llmailinject_arxiv2025']
man = {m["slug"]: m for m in search.read_csv(paths.REPORTS / "defences.csv")}
problems, texts = [], {}
for rid in IDS:
    rec = schema.load_record(paths.RECORDS / f"{rid}.yaml")
    slug = rec["report"]["slug"]
    pdf = paths.DATA_DEFENCES / f"{slug}.pdf"
    m = man.get(slug, {})
    if not pdf.exists():
        problems.append(f"{rid}: {pdf.name} not in data/defences"); continue
    sha = corpus.sha256(pdf)
    rd = PdfReader(str(pdf))
    texts[slug] = ([p.extract_text() or "" for p in rd.pages], [p.extract_text(extraction_mode="layout") or "" for p in rd.pages])
    ok_sha = sha == m.get("sha256")
    ok_inc = m.get("decision") == "include"
    print(f"{rid:<28} {slug:<12} pages {len(rd.pages):>3}  sha256 {'matches' if ok_sha else 'DIFFERS'}  decision {m.get('decision')}")
    if not ok_sha:
        problems.append(f"{rid}: PDF differs from the one registered in reports/defences.csv")
    if not ok_inc:
        problems.append(f"{rid}: not an included study")

## Stage 3. Validate against the frozen schema and the PDFs; grade the comparisons; extra checks

In [ ]:
import re
import pypdfium2 as pdfium
_docs = {}
def pdfium_page(slug, page):
    if slug not in _docs:
        _docs[slug] = pdfium.PdfDocument(str(paths.DATA_DEFENCES / f"{slug}.pdf"))
    return _docs[slug][page - 1].get_textpage().get_text_range()
S1 = schema.load_schema(paths.SCHEMA / "record_schema.yaml")
check_rows, summary, residual, third_rows = [], [], [], []
for rid in IDS:
    rec = schema.load_record(paths.RECORDS / f"{rid}.yaml")
    slug = rec["report"]["slug"]
    plain, lay = texts.get(slug, (None, None))
    probs, notes = schema.validate_record(rec, S1, plain, lay)
    kept, third = [], []                       # a value pypdf glues to its neighbour is looked for in a third extraction (pypdfium2)
    for p in probs:
        m = re.match(r"results:(\S+): value '(.+)' not found on page (\d+)$", p)
        if m and schema.value_on_page(m.group(2), pdfium_page(slug, int(m.group(3)))):
            third.append(f"{m.group(1)} ({m.group(2)!r}, page {m.group(3)})")
        else:
            kept.append(p)
    probs = kept
    third_rows += [{"report": rid, "status": "value found in the third extraction", "detail": t} for t in third]
    for kind in ("defences", "evaluations", "results", "comparisons", "claims"):          # ids unique within each kind
        dup = [i for i, n in Counter(r["id"] for r in rec.get(kind) or []).items() if n > 1]
        probs += [f"{kind}: id {i} used more than once" for i in dup]
    fig_kept = 0
    for r in rec.get("results") or []:                                                     # chart values kept as printed
        if r.get("value_source") == "figure_only" and r.get("value") not in (None, ""):
            fig_kept += 1
            pg = int(r["page"]) - 1
            if plain and not (schema.value_on_page(r["value"], plain[pg]) or schema.value_on_page(r["value"], lay[pg])):
                probs.append(f"results:{r['id']}: chart value {r['value']!r} not found on page {r['page']}")
    grades = Counter(schema.grade(c) for c in rec.get("comparisons") or [])
    n_val = sum(1 for r in rec.get("results") or [] if r.get("value_source") != "figure_only")
    n_anc = sum(1 for k in ("defences", "claims") for r in rec.get(k) or [] if r.get("anchor"))
    summary.append({"batch": BATCH, "report": rid, "slug": slug, "role": rec["report"]["role"],
                    "defences": len(rec.get("defences") or []), "evaluations": len(rec.get("evaluations") or []),
                    "results": len(rec.get("results") or []), "values_checked": n_val, "chart_values_checked": fig_kept,
                    "anchors_checked": n_anc, "comparisons": len(rec.get("comparisons") or []),
                    "grade_A": grades["A"], "grade_B": grades["B"], "grade_C": grades["C"], "grade_U": grades["U"],
                    "claims": len(rec.get("claims") or []),
                    **{f"cov_{q}": rec["coverage"][q] for q in ("R1", "R2clean", "R2attack", "R3ns", "R3cp", "R3safe")},
                    "values_third_extraction": len(third), "residual_notes": len(notes), "problems": len(probs)})
    check_rows += [{"report": rid, "status": "problem", "detail": p} for p in probs]
    residual += [{"report": rid, "where": n.split(": ", 1)[0], "note": n.split(": ", 1)[1]} for n in notes]
    print(f"{rid:<28} problems {len(probs)}; values {n_val} ({len(third)} via the third extraction); chart values {fig_kept}; anchors {n_anc}; grades {dict(grades)}")
    for p in probs:
        print("   ", p)
    problems += [f"{rid}: {p}" for p in probs]
check_rows += third_rows + [{"report": g["report"], "status": "residual note", "detail": f"{g['where']}: {g['note']}"} for g in residual]
search.write_csv(paths.REPORTS / f"extraction_{BATCH}_check.csv", check_rows or [{"report": "all", "status": "ok", "detail": "no problems"}],
                 ["report", "status", "detail"])
search.write_csv(paths.REPORTS / f"extraction_{BATCH}_summary.csv", summary, list(summary[0].keys()))
for g in residual:
    print(f"  residual note, {g['report']}: {g['note'][:150]}")

## Stage 4. Batch register (one row per coded report, kept across batches)

In [ ]:
reg_path = paths.REPORTS / "extraction_batches.csv"
reg = [r for r in search.read_csv(reg_path) if r["batch"] != BATCH] if reg_path.exists() else []
today = dt.date.today().isoformat()
for s in summary:
    reg.append({"batch": BATCH, "report": s["report"], "slug": s["slug"], "coded": today,
                "status": "checked" if s["problems"] == 0 else "problems"})
reg.sort(key=lambda r: (r["batch"], r["report"]))
search.write_csv(reg_path, reg, ["batch", "report", "slug", "coded", "status"])
included = sum(1 for m in man.values() if m.get("decision") == "include")
pilot = len(list(paths.RECORDS.glob("*.yaml"))) - len({r["report"] for r in reg})
print(f"records in reports/records: {len(list(paths.RECORDS.glob('*.yaml')))} (pilot {pilot}, batches {len({r['report'] for r in reg})}); included studies: {included}")

## Stage 5. Decision record and tests

In [ ]:
tot = {k: sum(s[k] for s in summary) for k in ("defences", "evaluations", "results", "values_checked", "values_third_extraction", "chart_values_checked", "anchors_checked",
                                              "comparisons", "grade_A", "grade_B", "grade_C", "grade_U", "claims")}
status = "BATCH CHECKED" if not problems else "BATCH NOT CHECKED: PROBLEMS BELOW"
lines = [f"decision_{BATCH}: {status}", f"this step: {dt.date.today().isoformat()}",
         "reports: " + ", ".join(IDS),
         f"records: {tot['defences']} defences, {tot['evaluations']} evaluations, {tot['results']} results, "
         f"{tot['comparisons']} comparisons, {tot['claims']} claims",
         f"checked against the PDFs: {tot['values_checked']} printed values ({tot['values_third_extraction']} of them found only in the third "
         f"extraction), {tot['chart_values_checked']} chart values and {tot['anchors_checked']} anchors, each found on its page",
         f"comparison grades: A {tot['grade_A']}, B {tot['grade_B']}, C {tot['grade_C']}, U {tot['grade_U']}",
         f"residual notes: {len(residual)}; full-text exclusions found while coding: {cfg_exclusions}",
         f"next: {cfg_next}"] + [f"- problem: {p}" for p in problems]
(paths.REPORTS / f"decision_{BATCH}.txt").write_text("\n".join(lines) + "\n")
print("\n".join(lines))
tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(tests.stdout.strip().splitlines()[-1] if tests.stdout.strip() else tests.stderr.strip()[-400:])

## Stage 6. Commit

In [ ]:
import os, subprocess
os.chdir(os.environ.get("SOURCEMAP_REPO_ROOT", "/content/drive/MyDrive/SourceMap_Research/sourcemap-research"))
r = subprocess.run(["python", "tools/commit_cell.py",
                    "04b extraction batch b: twelve included studies coded from their PDFs, values and anchors checked, comparisons graded"],
                   capture_output=True, text=True)
print(r.stdout)
print(r.stderr)